# Quant Research and Execution Lab — Run All
Select TPU in Colab. Install dependencies, restart once after core library upgrades, then run all cells. All historical evaluation windows have been observed; see the report.

# 第1课 研究基础 数据与趋势策略

**Quant Research and Execution Lab · 标准课件 01_foundations_data_and_trend.ipynb · 修订 2026-10-13**

目标：从一笔能手算的交易开始，用同一套现金与持仓账本研究 SPY 双均线、时间序列动量和买入持有。完成后应能解释每笔收益的时间来源、费用和现金，并说明趋势模型什么时候失败。按下方01至09顺序运行，所有代码直接展示在本笔记本中。

**金融直觉与术语**

ETF是可在交易所买卖的一篮子资产基金；SPY是本课预设研究对象。做多是在价格上涨时获益；做空是先卖借来的资产；杠杆是超过自有本金的敞口。本课只做多或持有现金，不借券、不融资。基准是同起点的买入持有，帮助判断规则是否优于简单持有。回撤是账户权益相对此前峰值的跌幅，不等于某一天的跌幅。

趋势假设：价格变化可能延续。双均线用短期平均价格高于长期平均价格作为持有条件；时间序列动量用本资产过去126个交易日收益为正作为持有条件。它们不是上涨保证，震荡时可能反复买卖。20/100、50/200、126是预先登记的教学窗口，不是已经找到的最优参数。

**符号与公式**

$C_t$为调整后的收盘价，$O_t$为同口径开盘价，$s_t$是收盘信号，$w_t$为开盘目标权重，$q_t$为模型持仓单位，$B_t$为现金，$E_t$为账户权益，$c$为单边成本率。

$$SMA_{n,t}=\frac{1}{n}\sum_{j=0}^{n-1}C_{t-j},\qquad s_t^{MA}=1(SMA_{20,t}>SMA_{100,t})$$
$$MOM_{L,t}=C_t/C_{t-L}-1,\qquad s_t^{MOM}=1(MOM_{126,t}>0),\qquad w_t=s_{t-1}$$

周一收盘形成信号 → 周二开盘成交并付费 → 周三开盘估值。策略只能获得买入之后的收益。信号用过去和当日数据；下一开盘只能用于事后评估，不能进入当日决策。252个交易日统一预热，所有模型同一起点，预热时不交易。

$$E_{pre}=B+qO_t,\quad F=|\Delta q|O_t c,\quad B'=B-\Delta q O_t-F$$
$$E_{end}=B'+(q+\Delta q)O_{t+1},\quad r_t=E_{end}/E_{pre}-1$$
$$V_t=\prod_{i\leq t}(1+r_i),\quad DD_t=V_t/\max(1,V_1,\ldots,V_t)-1$$

本课用现金约束精确求解扣费后的目标持仓，不能先把全部现金买完，再另扣费用导致透支。年化收益为$V_T^{252/N}-1$；年化波动为日收益标准差乘$\sqrt{252}$；零无风险率夏普为日均收益除以日标准差乘$\sqrt{252}$。252是日频股票的近似，不能套到逐笔数据。

**手算示例与参考答案**

本金1000元，100元买5份，成交额500元。5bp即0.05%，费用0.25元，剩余现金499.75元。下一开盘102元时，权益为499.75+5×102=1009.75元，账户收益0.975%；若98元则权益989.75元，收益−1.025%。标的涨2%不代表整个账户涨2%，因为部分本金仍是现金。

**数据口径与边界**

默认只下载2008—2022年开发数据，end='2023-01-01'是排他的结束日期。价格索引来自真实交易日，节假日不补虚构价格；缺失值先停止核对，不使用未来值回填。保存vendor Open/Close、Adj Close、调整因子和分红拆股事件。Yahoo的非股息复权字段也不能直接视为交易所原始逐笔价格。

账户算法使用调整后的价格和**模型单位**，便于统一总回报研究。它们不是历史真实报价和可下单股数；复权已包含公司行动的调整，账本不再次发分红或增加拆股股数。该简化不能验证真实支付日、税、整数股、成交容量和冲击；进入实盘账本前必须另建原始价格及公司行动处理。调整序列可能被供应商修订，本次哈希用于标识版本，不保证未来重新下载字节完全相同。

2023—2026年9月原规划为历史留出；此前笔记本已显示后期表现，故不能再宣称完全未见的测试。此课不下载、不据其调参。后续冻结规则后须如实记录已接触数据，使用真正未用的数据或前瞻模拟检验。SPY固定资产研究不等于已处理全市场幸存者偏差。

**代码阅读路线**

|单元格|运行产出与逐段解释|
|---|---|
|01 环境配置|固定依赖、种子、research_config；字典记录研究设置，CPU足够。|
|02 手算|核对现金和权益；rolling窗口形成后才有均线；shift延迟一行即下一交易日。|
|03 数据|下载或显式CSV接口，检查字段与重复，保留事件、覆盖、版本、SHA256与失败日志。|
|04 信号|金融规则转成0/1目标；instrument、model_id、available_at、horizon统一供后续模块使用。|
|05 账本|simulate是共用函数；成交额、费用、现金、持仓、权益、换手与暴露逐日记录。|
|06 验证与基线|假设数据验证记账；改未来价格不改变过去信号；三个模型同起点同费用。|
|07 指标|看收益、夏普、最大回撤、订单数、换手和暴露；分段只是开发稳定性观察。|
|08 三张图|价格与均线、净/毛权益和基准、回撤与暴露；净值点画在收益结束时点。|
|09 对照实验|5/10/20bp成本、50/200慢均线、额外延迟一日；每次只改变一个因素，保留负结果。|

**调优问题与练习参考**

1. 基础实验：顺序运行全部单元格，手算核对02和06，解释signal、target_weight和执行时点。参考：周一收盘信息在周一开盘不可得，shift(1)将指令推至周二。
2. 成本实验：比较5/10/20bp。参考：它们是教学总成本情景，不是实际券商费率；相同0/1指令成本提高，终值应不升，具体差额读取09的实际输出。
3. 个人对照：从慢均线或额外延迟中选一项，先写假设，再报告实际收益、回撤、订单数和市场暴露。参考：慢均线可能减少反应，也可能错过趋势；延迟可好可坏，不预先写结论。
4. 机制讨论：策略赚的是趋势择时还是长期市场暴露？参考：结合买入持有、持仓比例和分段结果，单凭高夏普不能证明独立超额收益。
5. 进阶：预先登记一个过去窗口最高价突破信号，复用make_interface和simulate，在相同预热和开发区间比较。不要在最终留出期选择窗口，不必以复杂模型作为通过标准。

**失败条件与验收**

Yahoo失败、字段缺失、重复日期、价格非正、样本不足立即停止；不静默替换成模拟真实行情。频繁震荡、趋势逆转、跳空、滑点上升、市场暴露降低和供应商修订都可能使结论失效。数据哈希、版本和试验记录保留在输出；当前课不向本地写CSV、图片或额外课件文件。

验收：在新Colab会话自上而下运行；费用方向、现金约束、记账守恒、因果性与共同窗口检查通过；图表有结果；能解释复权局限并独立完成一项对照。Notebook保存不等于运行内存永久保存，依赖输出和数据记录需随课件保留。个人作业尚需学习者完成，不能由代码跑通替代。

**官方资料**

- [Colab FAQ：会话、保存与资源](https://research.google.com/colaboratory/faq.html)
- [yfinance download：日期、调整与事件参数](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
- [yfinance项目与数据使用说明](https://github.com/ranaroussi/yfinance)
- [pandas rolling](https://pandas.pydata.org/docs/reference/api/pandas.Series.rolling.html) · [shift](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html)
- [Backtesting.py：成交、佣金与时点参考](https://kernc.github.io/backtesting.py/doc/backtesting/backtesting.html)

---
下面所有单元格保持展开。先全部运行标准版，再逐段学习；从本课开始复用相同信号接口和账本。


In [ ]:
# 01 环境与研究配置：CPU 足够；依赖安装只发生在 Colab
%pip -q install numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.3 yfinance==0.2.66
import sys, json, hashlib, platform
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
research_config = {
    "lesson": "01_foundations_data_and_trend",
    "instrument": "SPY", "start": "2008-01-01", "end_exclusive": "2023-01-01",
    "fast": 20, "slow": 100, "momentum_lookback": 126,
    "common_warmup": 252, "initial_cash": 10000.0,
    "one_way_cost_bps": 5.0, "execution_delay_sessions": 1,
    "cash_rate": 0.0, "risk_free_rate": 0.0, "annual_sessions": 252,
    "accounting": "adjusted_price_synthetic_units", "seed": SEED,
    "development": "2008-2022", "historical_holdout": "2023-2026-09",
    "holdout_status": "previous notebook displayed later results; not pristine unseen"
}
# 默认不下载、不查看历史留出区间，不以它选择参数。
run_log = {"started_utc": datetime.now(timezone.utc).isoformat(),
           "python": sys.version, "platform": platform.platform(),
           "versions": {p: version(p) for p in ["numpy", "pandas", "matplotlib", "yfinance"]},
           "seed": SEED, "status": "started"}
experiment_registry = []
print(json.dumps(research_config, ensure_ascii=False, indent=2))
print("实际依赖版本：", run_log["versions"])
# 字典存研究设置；函数复用同一算法；DataFrame 按交易日索引存价格和账本。


In [ ]:
# 02 手算账本：假设数据，与 Yahoo 行情明确分开
initial_cash = 1000.0
quantity, entry_price, next_price = 5.0, 100.0, 102.0
cost_rate = 5 / 10000  # 1 bp = 0.01%；5 bp = 0.05%
notional = quantity * entry_price
fee = notional * cost_rate
cash = initial_cash - notional - fee
equity = cash + quantity * next_price
print(f"成交额={notional:.2f}，费用={fee:.2f}，剩余现金={cash:.2f}")
print(f"下一开盘权益={equity:.2f}，账户收益={equity/initial_cash-1:.3%}")
assert np.isclose(equity, 1009.75)
# 参考答案：跌至98元，权益989.75元，收益-1.025%。
print("跌至98元：", cash + quantity*98, (cash+quantity*98)/initial_cash-1)
# 手算 rolling 和 shift：3日均线在第3个交易日才形成。
toy = pd.DataFrame({"close": [100., 102., 104., 99., 98.]},
                   index=pd.date_range("2020-01-06", periods=5, freq="B"))
toy["sma3"] = toy.close.rolling(3, min_periods=3).mean()
toy["signal_at_close"] = (toy.close > toy.sma3).astype(int)
toy["target_at_next_open"] = toy.signal_at_close.shift(1).fillna(0)
display(toy)
# 第3天均线=(100+102+104)/3=102；信号1在第4天开盘才能执行。
assert toy.iloc[2].sma3 == 102 and toy.iloc[3].target_at_next_open == 1


In [ ]:
# 03 数据接口、原始/调整口径、事件与数据清单
# Yahoo字段的“未做股息调整”不等于交易所原始逐笔价；历史价格仍可能已按拆股调整。
SOURCE = "yahoo"  # 下载失败可改"csv"，指定Colab中已有CSV；不自动伪造真实数据。
CSV_PATH = None  # CSV需Date, Open, Close, Adj Close；事件列可选。
def load_prices(source="yahoo", csv_path=None):
    if source == "yahoo":
        data = yf.download(research_config["instrument"],
            start=research_config["start"], end=research_config["end_exclusive"],
            auto_adjust=False, actions=True, multi_level_index=False,
            progress=False, threads=False)
    elif source == "csv" and csv_path:
        data = pd.read_csv(csv_path, index_col="Date", parse_dates=True)
        data = data.loc[(data.index >= research_config["start"]) &
                        (data.index < research_config["end_exclusive"])]
    else:
        raise ValueError("请使用yahoo或提供Colab中CSV路径")
    if data is None or data.empty:
        raise RuntimeError("行情下载失败：检查Yahoo网络/限流。停止研究，不填入合成行情。")
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data = data.sort_index().copy()
    data.index = pd.DatetimeIndex(data.index).tz_localize(None).normalize()
    if data.index.has_duplicates:
        raise ValueError("发现重复交易日，需核对来源")
    needed = ["Open", "Close", "Adj Close"]
    if not set(needed).issubset(data.columns):
        raise ValueError("缺少Open/Close/Adj Close，不能静默混用价格口径")
    data[needed] = data[needed].apply(pd.to_numeric, errors="coerce")
    if data[needed].isna().any().any() or (data[needed] <= 0).any().any():
        raise ValueError("价格缺失或非正数：先核对，不用未来值回填")
    if len(data) < 500:
        raise ValueError("样本不足，先核对覆盖范围")
    return data
try:
    raw = load_prices(SOURCE, CSV_PATH)
except Exception as exc:
    run_log.update(status="data_failed", failure=str(exc))
    print(json.dumps(run_log, ensure_ascii=False, indent=2))
    raise
factor = raw["Adj Close"] / raw["Close"]
prices = pd.DataFrame({"vendor_open": raw.Open, "vendor_close": raw.Close,
    "adjustment_factor": factor, "open": raw.Open*factor, "close": raw["Adj Close"]})
for name in ["Dividends", "Stock Splits"]:
    prices[name] = raw[name].fillna(0) if name in raw else 0.0
prices.index.name = "session"
prices["close_available_at"] = (prices.index + pd.Timedelta(hours=16)).tz_localize("America/New_York")
prices["open_execution_at"] = (prices.index + pd.Timedelta(hours=9, minutes=30)).tz_localize("America/New_York")
# CSV字节仅在内存生成，用哈希识别本次数据；不在本地落盘。
data_manifest = {"source": SOURCE, "instrument": research_config["instrument"],
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "rows": len(raw), "first_session": str(raw.index.min().date()),
    "last_session": str(raw.index.max().date()),
    "sha256": hashlib.sha256(raw.to_csv(float_format="%.10g").encode()).hexdigest(),
    "price_basis": "Yahoo adjusted OHLC synthetic units; vendor fields retained",
    "events_present": [x for x in ["Dividends", "Stock Splits"] if x in raw],
    "license": "教学个人研究；发布前另核验Yahoo条款，不默认再分发行情"}
print(json.dumps(data_manifest, ensure_ascii=False, indent=2))
display(prices.head(), prices.tail())
display(prices.loc[(prices["Dividends"] != 0) | (prices["Stock Splits"] != 0)].tail())
# 交易日只取数据索引：节假日不造价格；星期一的上一行通常是星期五。
# 复权已反映公司行动，本课模型账本不再次加分红或乘拆股，避免重复记账。


In [ ]:
# 04 两类趋势信号与统一时间接口
# SMA_n(t)=过去n个收盘价均值；MOM_L(t)=C_t/C_(t-L)-1。
# 两种规则都是做多/现金：不卖空、不加杠杆。20/100和126是预设教学窗口。
def make_signal(close, model="ma", fast=20, slow=100, lookback=126):
    if model == "ma":
        if not 1 < fast < slow: raise ValueError("需要1<fast<slow")
        a = close.rolling(fast, min_periods=fast).mean()
        b = close.rolling(slow, min_periods=slow).mean()
        return (a > b).where(b.notna()).astype(float)
    if model == "mom":
        m = close/close.shift(lookback)-1
        return (m > 0).where(m.notna()).astype(float)
    if model == "buy_hold": return pd.Series(1.0, index=close.index)
    raise ValueError("未知模型")

def make_interface(model="ma", fast=20, slow=100, lookback=126, delay=1):
    if delay < 1: raise ValueError("至少延迟1个交易日，禁止收盘信号当天开盘执行")
    s = make_signal(prices.close, model, fast, slow, lookback)
    out = pd.DataFrame({"instrument": research_config["instrument"],
        "model_id": f"{model}_{fast}_{slow}_{lookback}_delay{delay}",
        "signal": s, "available_at": prices.close_available_at,
        "target_weight": s.shift(delay),
        "decision_available_at": prices.close_available_at.shift(delay),
        "execution_at": prices.open_execution_at, "horizon": "next_open"})
    # 252个预热交易日供所有参数共用；预热不交易；最后一行只做下一开盘估值。
    out = out.iloc[research_config["common_warmup"]:-1].copy()
    if out.target_weight.isna().any(): raise ValueError("预热不足")
    assert (out.decision_available_at < out.execution_at).all()
    return out

signal_table = make_interface()
features = prices[["close"]].copy()
features["sma20"] = prices.close.rolling(20).mean()
features["sma100"] = prices.close.rolling(100).mean()
features["momentum126"] = prices.close/prices.close.shift(126)-1
display(signal_table.head(), features.tail())
# rolling不指定center=True；shift(+1)延后指令；shift(-1)仅用于事后评估下一开盘。
# available_at使用纽约时间16:00保守记录；早收盘日仍晚于真实收盘但早于下一开盘。


In [ ]:
# 05 共用交易账本：按成交额计费，买入费用由现金支付，不能透支
# 本课使用复权价格和“模型单位”，不是实际券商股数/成交报价。
# E_pre=cash+q*O；F=|delta_q|*O*c；cash'=cash-delta_q*O-F。
# E_end=cash'+q'*O_next；目标权重按扣费后权益求解，避免先满仓再产生负现金。
def simulate(open_prices, target_weight, cost_bps=5.0, initial=10000.0):
    if not 0 <= cost_bps < 10000 or initial <= 0: raise ValueError("成本或本金无效")
    p = open_prices.reindex(target_weight.index)
    next_p = open_prices.shift(-1).reindex(target_weight.index)
    next_date = pd.Series(open_prices.index, index=open_prices.index).shift(-1).reindex(target_weight.index)
    if p.isna().any() or next_p.isna().any(): raise ValueError("缺少执行或下一开盘价格")
    if (p <= 0).any() or (next_p <= 0).any(): raise ValueError("价格必须为正")
    if not target_weight.between(0,1).all(): raise ValueError("只允许0到1权重")
    cash, units, c, rows = float(initial), 0.0, cost_bps/10000, []
    for date, weight in target_weight.items():
        price, next_price = float(p.loc[date]), float(next_p.loc[date])
        before = cash + units*price
        # 求解d = w*(E_pre-c*abs(d))-旧持仓市值；d为有方向成交额。
        gap = float(weight)*before - units*price
        signed_notional = gap/(1+float(weight)*c*np.sign(gap))
        if abs(signed_notional) < 1e-8: signed_notional = 0.0
        delta_units = signed_notional/price
        fee = abs(signed_notional)*c
        cash = cash-signed_notional-fee
        units = units+delta_units
        after = cash+units*price
        end = cash+units*next_price
        assert np.isclose(after, before-fee, rtol=1e-10, atol=1e-7)
        assert cash >= -1e-7 and units >= -1e-7
        rows.append({"session": date, "end_session": next_date.loc[date],
            "target_weight": weight, "model_open": price, "next_model_open": next_price,
            "equity_before": before, "delta_model_units": delta_units,
            "trade_notional": abs(signed_notional), "fee": fee, "cash": cash,
            "model_units": units, "equity_after_trade": after, "equity_end": end,
            "net_return": end/before-1, "turnover": abs(signed_notional)/before,
            "exposure": units*price/after if after > 0 else np.nan})
    ledger = pd.DataFrame(rows).set_index("session")
    ledger["wealth"] = ledger.equity_end/initial
    ledger["drawdown"] = ledger.wealth/ledger.wealth.cummax().clip(lower=1)-1
    return ledger
# 后续课程使用同一simulate接口，不为每个信号重写收益算法。
# end_session表示收益终点；最后一个执行日之后仍有一个开盘估值日。
# 不强制末日卖出；现金利息、税、整数股约束和实际冲击本课暂未建模。


In [ ]:
# 06 关键验证与标准实验：先验证算法，再解释真实数据结果
# 用标注的假设价格验证费用方向、账本守恒和未来信息边界。
test_open = pd.Series([100., 102., 101.], index=pd.date_range("2020-01-06", periods=3, freq="B"))
test_weight = pd.Series([1., 0.], index=test_open.index[:2])
test_ledger = simulate(test_open, test_weight, 5, 1000)
expected_units = 1000/(100*1.0005)
expected_final = expected_units*102*(1-.0005)
assert np.isclose(test_ledger.iloc[-1].equity_end, expected_final)
assert np.isclose(test_ledger.fee.sum(), expected_units*(100+102)*.0005)
free = simulate(test_open, test_weight, 0, 1000)
assert test_ledger.iloc[-1].equity_end <= free.iloc[-1].equity_end
all_cash = simulate(test_open, test_weight*0, 20, 1000)
assert np.allclose(all_cash.equity_end,1000) and all_cash.fee.sum()==0
assert np.allclose(test_ledger.equity_end, test_ledger.cash+test_ledger.model_units*test_ledger.next_model_open)
assert np.allclose((1+test_ledger.net_return).cumprod(),test_ledger.wealth)
# 改写后半段的未来收盘价，过去信号必须完全不变。
cut = len(prices)//2
changed = prices.close.copy()
changed.iloc[cut:] *= 7
for model in ["ma", "mom"]:
    pd.testing.assert_series_equal(make_signal(prices.close,model).iloc[:cut],
                                   make_signal(changed,model).iloc[:cut])
print("PASS：买卖费用、无透支、现金策略、记账守恒、复利及信号因果性")

interfaces, ledgers = {}, {}
for name, model in [("SMA20/100","ma"),("Momentum126","mom"),("BuyHold","buy_hold")]:
    interfaces[name] = make_interface(model)
    ledgers[name] = simulate(prices.open, interfaces[name].target_weight,
                            research_config["one_way_cost_bps"],research_config["initial_cash"])
ledgers["SMA gross"] = simulate(prices.open,interfaces["SMA20/100"].target_weight,0,
                                research_config["initial_cash"])
common_index = ledgers["SMA20/100"].index
assert all(x.index.equals(common_index) for x in ledgers.values())
experiment_registry.append({"id":"baseline", "data_sha256":data_manifest["sha256"],
    "config":research_config.copy(), "models":list(ledgers), "scope":"development only"})
ledger = ledgers["SMA20/100"].join(interfaces["SMA20/100"][["decision_available_at","execution_at"]])
ledger["instrument"] = research_config["instrument"]
ledger["model_id"] = "SMA20/100"
display(ledger.head())
display(ledger.loc[ledger.trade_notional>1e-8].tail(10))
print("所有模型同起点、同终点、同本金、同成本口径；BuyHold初始买入也扣费。")


In [ ]:
# 07 收益、风险与暴露：都从共同账本计算
# r_t=E_end/E_pre-1；净值=prod(1+r)；回撤=净值/历史峰值-1。
# 年化收益用几何复利；夏普用日均收益/日标准差*sqrt(252)，本课Rf=0。
def performance(frame):
    r = frame.net_return.dropna()
    if len(r) < 2: return {}
    wealth = (1+r).cumprod()
    std = r.std(ddof=1)
    return {"total_return": wealth.iloc[-1]-1,
        "annual_return": wealth.iloc[-1]**(252/len(r))-1,
        "annual_vol": std*np.sqrt(252),
        "sharpe_rf0": r.mean()/std*np.sqrt(252) if std>1e-12 else np.nan,
        "max_drawdown": (wealth/wealth.cummax().clip(lower=1)-1).min(),
        "order_count": int((frame.trade_notional>1e-8).sum()),
        "turnover_sum": frame.turnover.sum(), "mean_exposure": frame.exposure.mean(),
        "fee_cash_sum": frame.fee.sum()}
summary = pd.DataFrame({name:performance(x) for name,x in ledgers.items()}).T
percentage_cols = ["total_return","annual_return","annual_vol","max_drawdown","mean_exposure"]
formats = {x:"{:.2%}" for x in percentage_cols}
formats.update({"sharpe_rf0":"{:.2f}","turnover_sum":"{:.2f}",
                "order_count":"{:.0f}","fee_cash_sum":"{:.2f}"})
print("开发区间执行起点：",common_index[0].date(),"最后估值：",ledger.end_session.iloc[-1].date())
display(summary.style.format(formats))
# 分段仅作开发区间稳定性观察，每段按该段收益重新归一，不声称独立测试。
period_results=[]
for label, start, end in [("2009-2012","2009-01-01","2013-01-01"),
                          ("2013-2017","2013-01-01","2018-01-01"),
                          ("2018-2022","2018-01-01","2023-01-01")]:
    for name,x in ledgers.items():
        sub=x.loc[(x.index>=start)&(x.index<end)]
        period_results.append({"period":label,"model":name,**performance(sub)})
display(pd.DataFrame(period_results).set_index(["period","model"]).style.format(formats))
# 费用金额合计不能直接从累计收益减去；费用改变以后可投资本金和复利路径。
# 低波动可能只是现金持有更多；结合mean_exposure判断，不能只看夏普。


In [ ]:
# 08 三张图：价格信号、同窗口净值、风险与市场暴露
fig,ax=plt.subplots(figsize=(13,4))
for col,label in [("close","Adjusted close"),("sma20","SMA20"),("sma100","SMA100")]:
    ax.plot(features.loc[common_index,col],label=label,lw=1)
ax.set(title="1. SPY adjusted-price trend signals (development)",ylabel="Adjusted model price")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

fig,ax=plt.subplots(figsize=(13,4))
for name,x in ledgers.items():
    # 净值点放在收益终点，并显式补初始权益1；不把下一开盘结果画到当天开盘。
    dates=pd.DatetimeIndex([x.index[0],*x.end_session.tolist()])
    ax.plot(dates,np.r_[1.,x.wealth.to_numpy()],label=name,lw=1.5)
ax.set(title="2. Wealth: identical starting capital, dates and accounting",ylabel="Initial wealth = 1")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
for name in ["SMA20/100","Momentum126","BuyHold"]:
    x=ledgers[name]
    axes[0].plot(x.end_session,100*x.drawdown,label=name)
    axes[1].plot(x.index,x.exposure,label=name,alpha=.7)
axes[0].set(title="3. Drawdown and exposure",ylabel="Drawdown (%)")
axes[1].set(ylabel="Equity in market",xlabel="Trading session",ylim=(-.05,1.05))
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
# 读图：净值较高是否伴随更深回撤？策略持有现金时是否错过上涨？
# 第一张为研究复权价；第二张需同时看净/毛；第三张把风险和现金暴露联系起来。


In [ ]:
# 09 个人对照实验、参考解答和运行记录
# 一次只改变一个因素，不搜索“最佳”参数，不读取历史留出期。
scenarios=[("baseline",20,100,5,1),("cost10",20,100,10,1),
           ("cost20",20,100,20,1),("slower50/200",50,200,5,1),
           ("delay2",20,100,5,2)]
scenario_ledgers, experiment_rows = {}, []
for name, fast, slow, bps, delay in scenarios:
    interface=make_interface("ma",fast=fast,slow=slow,delay=delay)
    result=simulate(prices.open,interface.target_weight,bps,research_config["initial_cash"])
    assert result.index.equals(common_index)
    scenario_ledgers[name]=result
    experiment_rows.append({"experiment":name,"fast":fast,"slow":slow,"cost_bps":bps,
        "delay_sessions":delay,**performance(result)})
    experiment_registry.append({"id":name,"model":"ma","fast":fast,"slow":slow,
        "cost_bps":bps,"delay":delay,"data_sha256":data_manifest["sha256"],
        "scope":"development only; predeclared comparison"})
experiment_table=pd.DataFrame(experiment_rows).set_index("experiment")
display(experiment_table.style.format(formats))
assert (scenario_ledgers["baseline"].equity_end.iloc[-1] >=
        scenario_ledgers["cost10"].equity_end.iloc[-1] >=
        scenario_ledgers["cost20"].equity_end.iloc[-1])
fig,ax=plt.subplots(figsize=(13,4))
for name,x in scenario_ledgers.items(): ax.plot(x.end_session,x.wealth,label=name)
ax.set(title="One-factor-at-a-time experiments: development only",ylabel="Wealth")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

base=experiment_table.loc["baseline"]
print("自动参考解答（数值由本次数据计算）：")
print(f"基线年化收益{base.annual_return:.2%}，夏普{base.sharpe_rf0:.2f}，最大回撤{base.max_drawdown:.2%}")
for name in ["cost10","cost20","slower50/200","delay2"]:
    x=experiment_table.loc[name]
    print(f"{name}: 累计收益相对基线变化{x.total_return-base.total_return:+.2%}，"
          f"订单数{x.order_count:.0f}，夏普{x.sharpe_rf0:.2f}，回撤{x.max_drawdown:.2%}")
print("费用提高时，相同0/1信号的终值应不升；慢均线/额外延迟的表现不能预先断言。")
print("换仓次数是买入与卖出订单的总数，不等于完整往返交易次数。")
print("个人作业：选一个对照，先写假设，再解释表中实际结果、金融机制和失败证据。")
run_log.update(status="completed_in_current_session", finished_utc=datetime.now(timezone.utc).isoformat(),
    checks="accounting, fees, no_negative_cash, causality, timing, common_window, cost_direction passed",
    data_sha256=data_manifest["sha256"], experiment_count=len(experiment_registry),
    clean_session_note="重启运行时后全部运行；以实际运行结果核验，不据代码存在声称通过")
print("数据清单、配置、版本、实验记录均保留在本笔记本输出和内存；不自动写本地文件。")
print(json.dumps(run_log,ensure_ascii=False,indent=2))
display(pd.DataFrame(experiment_registry))
# 研究结论只限本数据版本与开发区间；负结果同样保留，不用复杂模型掩盖失败。


# 第2课 均值回归与统计套利

**Quant Research and Execution Lab · 02_mean_reversion_and_stat_arb · 完整课件 · 2026-10-13**

本课作为第二模块紧接第一课，代码与产物保留在当前合并IPYNB。02.1—02.10可按顺序独立运行；CPU足够。依赖刚更新但内存仍是旧版本时，使用“代码执行程序 → 重启会话并运行所有单元”。本课只读取2008—2022开发数据；不发布、不连接券商、不导出本地文件。

## 本课目标与研究假设

由单资产z-score进入两资产相对价值策略，判断价差偏离是否会回归，并把双腿交易费、借券、现金与融资写进同一个账本。预先固定IWM/SPY候选：它们具有共同的美国股票风险敞口，但小盘与更广泛股票市场的风格差异可能使关系长期改变。这个经济联系只提供候选理由，不保证协整或盈利。

均值回归认为偏离某个稳定水平后，未来变化倾向于回到该水平。统计套利是用统计关系提出交易假设，仍承担估计误差、结构变化、流动性和尾部风险。本课仅为固定ETF配对研究，不等同于全市场统计套利。

## 术语与符号

|术语|含义与本课用法|
|---|---|
|相关性|两组变动同步程度；共同市场上涨可产生高相关，不保证残差回归。|
|平稳性|分布的统计性质随时间较稳定；有限样本检验不能证明永远平稳。|
|I(1)与协整|价格水平非平稳、差分更接近平稳；某个线性组合平稳时可能有协整关系。|
|OLS|只用过去窗口估计截距alpha与系数beta，最小化残差平方和。|
|价差与z-score|本课价差是对数回归残差u；z是相对于过去残差均值和标准差的偏离。|
|做多／做空|正持仓受益于上涨，负持仓受益于下跌；空头需借券、可能被召回。|
|净／总敞口|净敞口为有符号金额之和；总敞口为绝对金额之和，净额小不代表风险小。|
|半衰期|局部AR(1)估计下偏离缩小一半的交易日数；不是未来盈利时间承诺。|

A=IWM，B=SPY；$P_{A,t},P_{B,t}$为一致口径收盘价；$\alpha_t,\beta_t$为过去样本估计参数；$u_t$为残差，$\mu_t,\sigma_t$为估计窗口残差均值与标准差；$s_t$为收盘状态，$w_{i,t}$为开盘目标权重，$q_i$为模型单位，$B$为现金，$E$为账户权益。

$$u_t=\log P_{A,t}-\alpha_t-\beta_t\log P_{B,t},\qquad z_t=(u_t-\mu_t)/\sigma_t$$
$$u_{j+1}=a+\phi u_j+\varepsilon_{j+1},\qquad h=-\log(2)/\log(\phi),\quad 0<\phi<1$$

先用过去252个交易日估计，每21日重估。当前收盘不进入本次拟合窗口，但可用于本日z值。ADF原假设是单位根；Engle–Granger原假设是不协整。普通残差ADF p值只做诊断，协整判定使用coint校正p值。I(1)诊断条件与0.05阈值是本课预设启发式，不构成完整单位根证明；反复滚动检验不能当作多次独立证据。

## 决策、成交与停止规则

|状态|预设行为|
|---|---|
|空仓且合格|$2\le |z|<4$时入场；正z做空A做多B，负z反向。|
|持仓且价差回归|进入中心带或直接越过均值时退出，不能因跳过中心带继续反向持有。|
|关系不合格|beta非正、半衰期无效、启用过滤后不满足协整／I(1)诊断时退出。|
|进一步恶化|向不利方向到达4个标准差时止损，或持仓40个交易日后退出。|
|不可借|执行时若任一目标空头不可借，整个组合归零并按逐腿成本关闭；恢复可借后由当时指令决定。|

周一收盘可得信号 → 周二开盘成交 → 周三开盘估值。退出也按下一交易日开盘执行，所以价格跳空可能使实际损失超出z止损。不可借分支是执行层覆盖，不会伪造历史借券数据。基线假设可借；2011-08到2012-07不可借区间只是显式压力假设。

## 对冲金额与共同账本

给定权益E、总敞口比例g和状态s：

$$w_A=gs/(1+|\beta|),\qquad w_B=-\beta w_A,\qquad q_i=w_iE_{after}/O_i$$

beta描述对数变动的近似金额对冲关系，不是两腿股数比；它也不保证美元中性或市场beta中性。借券、融资和风险暴露仍须单独计算。

$$E_{pre}=B+\sum_i q_iO_i,\qquad F_i=|\Delta q_i|O_i(c_{commission}+c_{slippage})$$
$$B'=B-\sum_i\Delta q_iO_i-\sum_iF_i-F_{borrow}-F_{funding}$$
$$E_{end}=B'+\sum_iq'_iO_{i,next},\qquad r=E_{end}/E_{pre}-1$$

算法同时求解扣交易费后的权益与目标持仓，避免把成本扣两次。借券按各腿空头开盘市值×假设年费率×日历天数/365计提，周末也计；担保现金=空头市值×1.5，现金不足担保部分按假设融资利率计费。担保只是现金分类，不能再次从账户权益减去。模型假定该融资额度可用，不实现真实券商保证金追缴和强平；总敞口有配置上限。

## 数据与记账口径

Yahoo教学下载保留供应商Open/Close、Adj Close、调整因子、分红拆股事件和时间字段，记录版本、覆盖、获取时点与SHA256。双腿只取共同交易日；不靠未来值填缺失，失败时停止。Yahoo非股息调整价可能已按拆股处理，不等于交易所原始逐笔报价。公开发布前需核对数据许可。

本账本用调整价与模型单位，复用第一课自融资恒等式的多资产扩展，并用单资产解析参考与第一课引擎交叉核验。公司行动不能再重复入账。真实空头股息补偿、支付日期、税、整数股、冲击和真实借券限制仍需原始价格账本验证；本课输出不能称为可执行套利。

## 实验路线与产物

|单元格|内容与完成证据|
|---|---|
|02.1|固定版本、种子、所有研究参数与运行日志。|
|02.2|两腿统一价格表、事件、数据清单和可得时间。|
|02.3|滚动OLS、协整／ADF、半衰期、状态机及冻结模型。|
|02.4|共用多资产账本、逐腿费用、现金担保、净／总敞口和信号接口。|
|02.5|费用、借券、融资、不可借、损益守恒、因果性和非法行情检查。|
|02.6|单资产、无对冲、固定对冲、滚动对冲、等权再平衡、冻结模型及图表。|
|02.7|一次改一因素的成本／借券／不可借／过滤对照与真实退出案例。|
|02.8|可运行双腿手算，与扣费后目标求解对照。|
|02.9|固定种子的结构变化合成案例，明确区别于历史行情与业绩。|
|02.10|练习参考、实际结果差异、全部最终验收和完整实验台账。|

主对照fixed只固定初始beta，其他信号相同，用于隔离对冲方式变化；frozen_model冻结alpha、beta和残差分布，是额外的整套模型对照。equal_weight是双腿等权日频再平衡，不称为纯粹一次买入持有。所有模型用相同本金、预热、执行起点和估值终点；每个规则使用相同账本。

## 手算与练习参考

本金10000，总敞口8000，beta=1.5：A金额3200、B金额−4800；A价100、B价200时，模型单位32和−24。1bp佣金加4bp滑点，总交易费用4。现金=10000−3200+4800−4=11596，权益=现金+3200−4800=9996。短卖收到的现金不是利润。A次日103、B次日201，市场损益=32×3−24×1=72；周五到周一借券费=4800×3%×3/365，再从账户扣除。02.8运行给出精确答案。固定份额手算与引擎的扣费后权重求解略有差异，课件同时展示两种口径。

1. 基础实验：全部运行，核对手算、信号延迟、每腿费用和账本。参考答案在02.5与02.8的断言和输出中。
2. 个人对照：选择成本、借券或过滤一项，先写机制假设，再报告02.7实际指标、订单和敞口，不预先宣称改善。参考差异由02.10读取本次结果计算。
3. 调优问题：什么机制使价差回归？半衰期是否短到覆盖费用？相关性是否只来自共同市场？多个候选中选显著配对是否造成选择偏差？答案必须结合开发证据、完整试验记录和后续新数据，不用“p小于0.05”替代收益验证。
4. 进阶：OU、Kalman或因子残差任选一项，先登记新增参数和否决条件，沿用相同目标接口和账本；本课不以实现所有进阶模型作为通过标准。

## 失败条件与验收

价差可能不回归、结构变化可能暂时被滚动估计吸收、借券可能召回、滑点与融资可能吞噬收益、调整数据可能被修订。过滤后无交易、模型亏损、成本敏感和关系断裂都应保留。结构变化示例是机制演示，不能充当真实回测证据。

标准课件验收：干净会话自上而下运行；全部10个代码单元格有实际输出；未来改写不改变过去；逐腿损益减全部费用等于账户权益变化；所有模型窗口一致；不可借时无持仓；手算正确；价格、价差、净值、回撤、净／总敞口、交易记录与失败案例均可查；最终日志状态为completed_and_verified。个人自主实验仍需学习者独立完成，课件提供任务和参考。

## 官方资料与研究入口

- [statsmodels coint：原假设与I(1)条件](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html)
- [statsmodels ADF：单位根诊断](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html)
- [yfinance下载与使用说明](https://github.com/ranaroussi/yfinance)
- [Backtesting.py：订单、成本与成交时间参考](https://kernc.github.io/backtesting.py/doc/backtesting/backtesting.html)

**本次标准基线未入场的处理**：严格过滤下的零交易保留为否决结果。02.6另给相同未过滤指令下的无对冲／固定／滚动对冲诊断；02.7的diagnostic情景再隔离成本、借券和不可借因素。它们都明确标为过滤消融，不能借此宣称满足协整或得到有效套利。

所有代码在下方展开；最终输出保留开发实验与模拟案例的明确标注。


In [ ]:
# 02.1 环境、预设候选与实验登记（可从此处独立运行）
%pip -q install numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.3 yfinance==0.2.66 statsmodels==0.14.5
import yfinance as yf
import json, hashlib, warnings, sys
from datetime import datetime, timezone
from importlib.metadata import version
for package in ["numpy", "pandas", "matplotlib", "statsmodels"]:
    loaded=sys.modules.get(package)
    if loaded is not None and getattr(loaded,"__version__",version(package)) != version(package):
        raise RuntimeError("核心依赖已更新：使用 重启会话并运行所有单元，再继续。")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from IPython.display import display
L2_CONFIG={"assets":["IWM","SPY"],"start":"2008-01-01","end":"2023-01-01",
    "window":252,"refit_every":21,"entry_z":2.,"exit_z":.5,"stop_z":4.,
    "max_holding":40,"gross_target":.8,"commission_bps":1.,"slippage_bps":4.,
    "borrow_annual":.03,"funding_annual":.05,"short_collateral":1.5,
    "initial_cash":10000.,"p_threshold":.05,"adf_maxlag":4,"single_window":20,
    "max_gross":2.,"seed":42,
    "borrow_status":"assumed availability, not historical observations"}
np.random.seed(42)
L2_LOG={"started_utc":datetime.now(timezone.utc).isoformat(),"python":sys.version,
    "versions":{p:version(p) for p in ["numpy","pandas","matplotlib","yfinance","statsmodels"]},
    "status":"started","accounting":"adjusted-price synthetic units"}
L2_EXPERIMENTS=[]
print(json.dumps(L2_CONFIG,ensure_ascii=False,indent=2))
print(L2_LOG["versions"])


## 02A 两腿数据与关系估计

02.2先建立一致价格表并保存事件与来源。02.3在每个决策日前的252行拟合OLS：alpha是截距，beta是金额对冲比的近似；用同一过去窗口的残差分布标准化当前z值。fit_last_session必须早于信号日。检验失败、半衰期无效或beta非正时，合格基线不入场。

固定对冲fixed只固定最初beta，整套冻结模型另称frozen_model。后者不能通过未来数据重新估计参数。

In [ ]:
# 02.2 两腿数据与事件；不使用后期样本，不自动填补行情
L2_RAW={}; L2_MANIFEST=[]
try:
    for asset in L2_CONFIG["assets"]:
        d=yf.download(asset,start=L2_CONFIG["start"],end=L2_CONFIG["end"],
            auto_adjust=False,actions=True,multi_level_index=False,progress=False,threads=False)
        if d is None or d.empty: raise RuntimeError(f"{asset}下载失败；停止，不伪造数据")
        if isinstance(d.columns,pd.MultiIndex): d.columns=d.columns.get_level_values(0)
        d.index=pd.DatetimeIndex(d.index).tz_localize(None).normalize()
        d=d.sort_index()
        if d.index.has_duplicates: raise ValueError(f"{asset}重复交易日")
        cols=["Open","Close","Adj Close"]
        d[cols]=d[cols].apply(pd.to_numeric,errors="coerce")
        if d[cols].isna().any().any() or (d[cols]<=0).any().any(): raise ValueError("价格缺失/非正")
        L2_RAW[asset]=d
        L2_MANIFEST.append({"instrument":asset,"source":"Yahoo/yfinance",
            "first":str(d.index.min().date()),"last":str(d.index.max().date()),"rows":len(d),"retrieved_utc":datetime.now(timezone.utc).isoformat(),
            "price_basis":"vendor OHLC and dividend-adjusted OHLC retained",
            "usage":"personal educational research; do not assume redistribution rights",
            "events_available":[x for x in ["Dividends","Stock Splits"] if x in d],
            "sha256":hashlib.sha256(d.to_csv(float_format="%.10g").encode()).hexdigest()})
except Exception as exc:
    L2_LOG.update(status="data_failed",failure=str(exc)); print(L2_LOG); raise
l2_open=pd.concat({a:d.Open*d["Adj Close"]/d.Close for a,d in L2_RAW.items()},axis=1)
l2_close=pd.concat({a:d["Adj Close"] for a,d in L2_RAW.items()},axis=1)
l2_common=l2_open.dropna().index.intersection(l2_close.dropna().index)
l2_dropped=len(l2_open.index)-len(l2_common)
l2_open=l2_open.loc[l2_common]; l2_close=l2_close.loc[l2_common]
assert len(l2_open)>500
l2_logprice=np.log(l2_close)
print("只保留双腿都有价格的交易日；丢弃不共同的行数：",l2_dropped)
display(pd.DataFrame(L2_MANIFEST),l2_close.head())
L2_PRICES=pd.concat({a:pd.DataFrame({"vendor_open":d.Open,"vendor_close":d.Close,
    "adjustment_factor":d["Adj Close"]/d.Close,"adjusted_open":d.Open*d["Adj Close"]/d.Close,
    "adjusted_close":d["Adj Close"],"dividend":d.get("Dividends",0.),"split":d.get("Stock Splits",0.)})
    for a,d in L2_RAW.items()},names=["instrument","session"]).reset_index()
L2_PRICES["available_at"]=(pd.to_datetime(L2_PRICES.session)+pd.Timedelta(hours=16)).dt.tz_localize("America/New_York")
print("原始供应商口径、调整口径、公司行动与时间字段：")
display(L2_PRICES.head(6))
print("复权已处理公司行动，不在模型单位账本重复记分红/拆股；事件仍保存在L2_RAW中。")


In [ ]:
# 02.3 滚动OLS、Engle-Granger、ADF与半衰期；只用t之前252行估计
# IWM为A，SPY为B；普通残差ADF只作诊断，协整采用coint的校正p值。
def l2_fit_features(logprice,window=None,refit_every=None):
    window=L2_CONFIG["window"] if window is None else window
    refit_every=L2_CONFIG["refit_every"] if refit_every is None else refit_every
    if window<30 or refit_every<1 or len(logprice)<=window:
        raise ValueError("估计窗口/重估频率/样本长度无效")
    rows=[]; estimate=None
    for t in range(window,len(logprice)):
        if (t-window)%refit_every==0:
            past=logprice.iloc[t-window:t]; y=past.iloc[:,0].to_numpy(); x=past.iloc[:,1].to_numpy()
            alpha,beta=np.linalg.lstsq(np.c_[np.ones(len(x)),x],y,rcond=None)[0]
            residual=y-alpha-beta*x
            phi=np.linalg.lstsq(np.c_[np.ones(len(residual)-1),residual[:-1]],residual[1:],rcond=None)[0][1]
            half=-np.log(2)/np.log(phi) if 0<phi<1 else np.nan
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                p_coint=coint(y,x,trend="c",maxlag=L2_CONFIG["adf_maxlag"],autolag="aic")[1]
                p_adf=adfuller(residual,maxlag=L2_CONFIG["adf_maxlag"],autolag="AIC")[1]
                level_p=[adfuller(v,maxlag=L2_CONFIG["adf_maxlag"],autolag="AIC")[1] for v in [y,x]]
                diff_p=[adfuller(np.diff(v),maxlag=L2_CONFIG["adf_maxlag"],autolag="AIC")[1] for v in [y,x]]
            estimate={"alpha":alpha,"beta":beta,"mean":residual.mean(),"std":residual.std(ddof=1),
                "coint_p":p_coint,"residual_adf_p":p_adf,"half_life":half,
                "i1_diagnostic":min(level_p)>L2_CONFIG["p_threshold"] and max(diff_p)<L2_CONFIG["p_threshold"],
                "fit_last_session":past.index[-1]}
        current=logprice.iloc[t]
        spread=current.iloc[0]-estimate["alpha"]-estimate["beta"]*current.iloc[1]
        z=(spread-estimate["mean"])/estimate["std"] if estimate["std"]>1e-10 else np.nan
        rows.append({"session":logprice.index[t],**estimate,"spread":spread,"z":z})
    return pd.DataFrame(rows).set_index("session")
l2_features=l2_fit_features(l2_logprice)
# 状态机：不在同一日退出后重新入场；显式保留退出原因和最长持仓。
def l2_states(features,entry=None,exit_z=None,stop=None,max_holding=None,require_coint=True):
    entry=L2_CONFIG["entry_z"] if entry is None else entry
    exit_z=L2_CONFIG["exit_z"] if exit_z is None else exit_z
    stop=L2_CONFIG["stop_z"] if stop is None else stop
    max_holding=L2_CONFIG["max_holding"] if max_holding is None else max_holding
    if not 0<=exit_z<entry<stop or max_holding<1: raise ValueError("状态机参数无效")
    state=0; age=0; rows=[]
    for date,r in features.iterrows():
        quality=(np.isfinite(r.z) and r.beta>0 and np.isfinite(r.half_life) and
                 (not require_coint or (r.coint_p<L2_CONFIG["p_threshold"] and bool(r.i1_diagnostic))))
        reason="hold"; old=state
        if state:
            age+=1
            if not quality: state=0; reason="relationship_failed"
            elif state*r.z>=-exit_z: state=0; reason="mean_reversion_exit"
            elif state*r.z<=-stop: state=0; reason="z_stop"
            elif age>=max_holding: state=0; reason="max_holding_exit"
        elif quality and entry<=abs(r.z)<stop:
            state=-int(np.sign(r.z)); age=0; reason="entry"
        rows.append({"session":date,"state":state,"age":age,"reason":reason,"changed":state!=old})
    return pd.DataFrame(rows).set_index("session")
l2_state=l2_states(l2_features)
l2_fixed_beta=float(l2_features.beta.iloc[0])
# 单资产z-score用于入门对照，均值/标准差来自上一日结束的20日窗口。
l2_prior=l2_close.iloc[:,0].shift(1)
l2_single_z=(l2_close.iloc[:,0]-l2_prior.rolling(L2_CONFIG["single_window"]).mean())/l2_prior.rolling(L2_CONFIG["single_window"]).std()
display(l2_features.head(),l2_state.loc[l2_state.changed].head(10))
print("固定beta（仅首个估计窗口）：",l2_fixed_beta)

# 冻结整个价差模型的附加对照：alpha/beta/均值/标准差只来自首个252日。
# 只固定beta的主对照仍复用滚动信号，故能隔离“对冲比率”这一个因素。
l2_frozen_features=l2_features.copy()
l2_initial_estimate=l2_features.iloc[0]
for field in ["alpha","beta","mean","std","coint_p","residual_adf_p","half_life","i1_diagnostic","fit_last_session"]:
    l2_frozen_features[field]=l2_initial_estimate[field]
l2_frozen_features["spread"]=(l2_logprice.iloc[:,0]-l2_initial_estimate.alpha-
    l2_initial_estimate.beta*l2_logprice.iloc[:,1]).reindex(l2_features.index)
l2_frozen_features["z"]=(l2_frozen_features.spread-l2_initial_estimate["mean"])/l2_initial_estimate["std"]
print("所有阈值、窗口和重估频率已接入L2_CONFIG；冻结模型只用第一个估计窗口。")


## 02B 指令转成持仓与现金账本

02.4把收盘状态移到下一开盘，用目标金额／开盘价换算两腿模型单位。每腿成交费、借券费与市场损益都单列。卖空收到现金，同时产生负资产市值；受限担保现金仍属于账户，不能再次扣减权益。融资只针对担保不足或借入现金部分。

这里的融资额度与借券可得性是情景假设。基线总敞口0.8，单资产测试允许预设1.5融资情景；本课不模拟真实券商强平。

In [ ]:
# 02.4 统一多资产账本：第一课自融资方程的多腿扩展
# 所有第二课模型复用此引擎；费用按每腿成交额，不用“价差涨幅”充当账户收益。
def l2_simulate_multi(opens,target,commission=None,slippage=None,borrow_rate=None,
                      funding=None,available=None,initial=None,collateral=None):
    commission=L2_CONFIG["commission_bps"] if commission is None else commission
    slippage=L2_CONFIG["slippage_bps"] if slippage is None else slippage
    borrow_rate=L2_CONFIG["borrow_annual"] if borrow_rate is None else borrow_rate
    funding=L2_CONFIG["funding_annual"] if funding is None else funding
    initial=L2_CONFIG["initial_cash"] if initial is None else initial
    collateral=L2_CONFIG["short_collateral"] if collateral is None else collateral
    if initial<=0 or collateral<1: raise ValueError("本金/担保比例无效")
    if not opens.columns.equals(target.columns): raise ValueError("价格与权重资产列必须同名同序")
    if not opens.index.is_monotonic_increasing or opens.index.has_duplicates:
        raise ValueError("价格索引需递增且唯一")
    if not target.index.equals(opens.loc[target.index[0]:target.index[-1]].index):
        raise ValueError("权重必须覆盖连续的共同交易日，不能跳过资金费用")
    if commission<0 or slippage<0 or borrow_rate<0 or funding<0: raise ValueError("费率不得为负")
    if target.isna().any().any() or (target.abs().sum(axis=1)>L2_CONFIG["max_gross"]).any(): raise ValueError("目标缺失/总敞口超限")
    p=opens.reindex(target.index); nxt=opens.shift(-1).reindex(target.index)
    ends=pd.Series(opens.index,index=opens.index).shift(-1).reindex(target.index)
    if (not np.isfinite(p.to_numpy()).all() or not np.isfinite(nxt.to_numpy()).all()
        or (p<=0).any().any() or (nxt<=0).any().any()):
        raise ValueError("执行和估值价格必须有限且为正")
    if available is None: available=pd.DataFrame(True,index=target.index,columns=target.columns)
    available=available.reindex(index=target.index,columns=target.columns)
    if available.isna().any().any(): raise ValueError("借券可得性缺失，不能默认为可借")
    cash=float(initial); units=np.zeros(len(target.columns)); rows=[]; trades=[]
    rate=(commission+slippage)/10000
    for date,w in target.iterrows():
        price=p.loc[date].to_numpy(float); future=nxt.loc[date].to_numpy(float)
        weights=w.to_numpy(float); blocked=bool(((weights<0)&~available.loc[date].to_numpy(bool)).any())
        if blocked: weights=np.zeros_like(weights)  # 不可借则关闭整个组合，已有空头买回。
        before=cash+units@price
        if before<=0: raise RuntimeError("权益非正，停止策略，不继续产生无效权重")
        old_value=units*price; post=before
        # 同时求解扣费后权益与各腿目标；成本在账户内只扣一次。
        for _ in range(40):
            delta=weights*post-old_value
            new_post=before-np.abs(delta).sum()*rate
            if abs(new_post-post)<1e-9: post=new_post; break
            post=new_post
        else: raise RuntimeError("持仓/费用求解未收敛")
        delta=weights*post-old_value
        delta[np.abs(delta)<1e-8]=0
        commission_cash=np.abs(delta)*commission/10000
        slippage_cash=np.abs(delta)*slippage/10000
        fees=commission_cash+slippage_cash
        units=units+delta/price; cash=cash-delta.sum()-fees.sum()
        after=cash+units@price
        assert np.isclose(after,before-fees.sum(),atol=1e-6)
        short_value=np.maximum(-units*price,0)
        restricted=collateral*short_value.sum()
        days=(ends.loc[date]-date).days
        borrow=short_value*borrow_rate*days/365
        finance=max(restricted-cash,0)*funding*days/365
        cash-=borrow.sum()+finance
        market_pnl=units*(future-price)
        end=cash+units@future
        assert np.isclose(end-before,market_pnl.sum()-fees.sum()-borrow.sum()-finance,atol=1e-6)
        if end<=0: raise RuntimeError("组合发生权益耗尽，停止并保留失败原因")
        row={"session":date,"end_session":ends.loc[date],"equity_before":before,
            "equity_after_trade":after,"cash":cash,"cash_restricted":restricted,
            "cash_free":cash-restricted,"equity_end":end,"net_return":end/before-1,
            "trade_notional":np.abs(delta).sum(),"transaction_cost":fees.sum(),
            "borrow_cost":borrow.sum(),"funding_cost":finance,"turnover":np.abs(delta).sum()/before,
            "gross_exposure":np.abs(units*price).sum()/after,"net_exposure":(units*price).sum()/after,
            "blocked_by_borrow":blocked,"market_pnl":market_pnl.sum(),
            "net_pnl":end-before,"long_value":np.maximum(units*price,0).sum(),
            "short_value":short_value.sum(),"holding":bool(np.abs(units).sum()>1e-8)}
        for j,a in enumerate(target.columns):
            row[f"units_{a}"]=units[j]; row[f"borrow_{a}"]=borrow[j]
            row[f"market_pnl_{a}"]=market_pnl[j]; row[f"position_value_{a}"]=units[j]*price[j]
            row[f"commission_{a}"]=commission_cash[j]; row[f"slippage_{a}"]=slippage_cash[j]
            if abs(delta[j])>1e-8:
                trades.append({"session":date,"instrument":a,"delta_model_units":delta[j]/price[j],
                    "notional":abs(delta[j]),"commission":commission_cash[j],"slippage":slippage_cash[j],
                    "model_price":price[j],"action":"buy/cover" if delta[j]>0 else "sell/short"})
        assert np.isclose(end,cash+units@future,atol=1e-6)
        rows.append(row)
    ledger=pd.DataFrame(rows).set_index("session")
    ledger["wealth"]=ledger.equity_end/initial
    ledger["drawdown"]=ledger.wealth/ledger.wealth.cummax().clip(lower=1)-1
    return ledger,pd.DataFrame(trades)

# 共同目标接口，available_at不晚于执行日；回归beta仅近似金额对冲，不等于股数比。
def l2_targets(model="rolling",require_coint=True):
    source_features=l2_frozen_features if model=="frozen_model" else l2_features
    state=l2_states(source_features,require_coint=require_coint).state.reindex(l2_open.index).fillna(0)
    beta=l2_features.beta.reindex(l2_open.index).fillna(0)
    w=pd.DataFrame(0.,index=l2_open.index,columns=l2_open.columns)
    if model in ["rolling","fixed","frozen_model"]:
        b=beta if model=="rolling" else l2_fixed_beta
        w.iloc[:,0]=L2_CONFIG["gross_target"]*state/(1+np.abs(b))
        w.iloc[:,1]=-b*w.iloc[:,0]
    elif model=="unhedged": w.iloc[:,0]=L2_CONFIG["gross_target"]*state
    elif model=="single_z":
        f=pd.DataFrame({"z":l2_single_z,"beta":1.,"half_life":1.},index=l2_open.index)
        w.iloc[:,0]=L2_CONFIG["gross_target"]*l2_states(f,require_coint=False).state
    elif model=="equal_weight": w[:]=L2_CONFIG["gross_target"]/len(w.columns)
    else: raise ValueError("未知模型")
    return w.shift(1).iloc[L2_CONFIG["window"]+1:-1].copy()
l2_target=l2_targets()
l2_interface=l2_target.stack().rename("target_weight").reset_index()
l2_interface.columns=["execution_session","instrument","target_weight"]
l2_prev=pd.Series(l2_open.index,index=l2_open.index).shift(1)
l2_interface["available_at"]=pd.to_datetime(l2_interface.execution_session.map(l2_prev))+pd.Timedelta(hours=16)
l2_interface["available_at"]=l2_interface.available_at.dt.tz_localize("America/New_York")
l2_interface["execution_at"]=(pd.to_datetime(l2_interface.execution_session)+pd.Timedelta(hours=9,minutes=30)).dt.tz_localize("America/New_York")
l2_interface["model_id"]="rolling_pair"; l2_interface["horizon"]="next_open"
l2_interface["signal"]=l2_interface.execution_session.map(l2_prev).map(l2_states(l2_features).state)
assert (l2_interface.available_at<l2_interface.execution_at).all()
display(l2_interface.head())


In [ ]:
# 02.5 关键测试：双腿费用、借券/融资、不可借、守恒与时间因果性
l2_test_dates=pd.date_range("2020-01-06",periods=3,freq="B")
l2_test_open=pd.DataFrame({"A":[100.,102.,101.],"B":[200.,202.,201.]},index=l2_test_dates)
l2_test_w=pd.DataFrame([[.4,-.4],[0.,0.]],index=l2_test_dates[:2],columns=["A","B"])
l2_t,l2_tt=l2_simulate_multi(l2_test_open,l2_test_w)
assert (l2_t.transaction_cost>=0).all() and (l2_t.borrow_cost>=0).all()
assert np.isclose(l2_tt.commission.sum()+l2_tt.slippage.sum(),l2_t.transaction_cost.sum())
assert np.allclose(l2_t.equity_end,l2_t.cash+l2_t.units_A*l2_test_open.A.shift(-1).loc[l2_t.index]+l2_t.units_B*l2_test_open.B.shift(-1).loc[l2_t.index])
assert np.allclose((1+l2_t.net_return).cumprod(),l2_t.wealth)
l2_free,_=l2_simulate_multi(l2_test_open,l2_test_w,0,0,0,0)
assert l2_t.equity_end.iloc[-1]<l2_free.equity_end.iloc[-1]
l2_unavailable=pd.DataFrame(False,index=l2_test_w.index,columns=l2_test_w.columns)
l2_blocked,l2_bt=l2_simulate_multi(l2_test_open,l2_test_w,available=l2_unavailable)
assert np.allclose(l2_blocked.equity_end,10000) and l2_bt.empty
# 融资分支用明确的假设杠杆例子验证，实际研究基线总敞口仍为0.8。
l2_constant=pd.DataFrame({"A":[100.,100.,100.]},index=l2_test_dates)
l2_leveraged=pd.DataFrame({"A":[1.5,0.]},index=l2_test_dates[:2])
l2_finance,_=l2_simulate_multi(l2_constant,l2_leveraged,0,0,0,.05,initial=1000)
assert np.isclose(l2_finance.funding_cost.iloc[0],500*.05/365)
# 与第一课引擎交叉核验0/1单资产：相同输入必须得到相同权益。
if "simulate" in globals():
    l2_binary=pd.Series([1.,0.],index=l2_test_dates[:2])
    l2_old=simulate(l2_test_open.A,l2_binary,5,1000)
    l2_new,_=l2_simulate_multi(l2_test_open[["A"]],l2_binary.to_frame("A"),1,4,0,0,initial=1000)
    assert np.allclose(l2_old.equity_end,l2_new.equity_end)
# 改写未来的两腿价格；过去拟合、信号和状态必须保持不变。
l2_cut=len(l2_logprice)//2
l2_changed=l2_logprice.copy(); l2_changed.iloc[l2_cut:]+=np.array([.5,-.2])
l2_changed_features=l2_fit_features(l2_changed.iloc[:l2_cut+25])
l2_before=l2_logprice.index[l2_cut]
pd.testing.assert_frame_equal(l2_features.loc[l2_features.index<l2_before],
                              l2_changed_features.loc[l2_changed_features.index<l2_before])
pd.testing.assert_frame_equal(l2_states(l2_features).loc[:l2_before].iloc[:-1],
                              l2_states(l2_changed_features).loc[:l2_before].iloc[:-1])
assert (l2_features.fit_last_session<l2_features.index).all()
l2_jump=pd.DataFrame({"z":[2.5,-2.],"beta":1.,"half_life":1.},index=l2_test_dates[:2])
l2_jump_state=l2_states(l2_jump,require_coint=False)
assert list(l2_jump_state.state)==[-1,0] and l2_jump_state.reason.iloc[-1]=="mean_reversion_exit"
print("L2 PASS：费用、借券、融资、不可借关闭、账本守恒、复利、单资产兼容与因果性")

# 额外核验：每腿市场PnL减去所有费用，严格等于账户权益变化。
assert np.allclose(l2_t.net_pnl,l2_t.market_pnl-l2_t.transaction_cost-l2_t.borrow_cost-l2_t.funding_cost)
assert np.allclose(l2_t.market_pnl,l2_t.market_pnl_A+l2_t.market_pnl_B)
assert np.allclose(l2_blocked.gross_exposure,0)
# 非法报价/总敞口在交易前拒绝，不能产生看似有效结果。
for bad_price in [0.,np.nan,np.inf]:
    invalid=l2_test_open.copy(); invalid.iloc[0,0]=bad_price
    try: l2_simulate_multi(invalid,l2_test_w)
    except ValueError: pass
    else: raise AssertionError("非法行情未被拒绝")
# 单资产满仓/现金的解析参考，第二课独立运行也必须执行。
reference_cash=1000/(100*1.0005)*102*(1-.0005)
reference_target=pd.DataFrame({"A":[1.,0.]},index=l2_test_dates[:2])
reference_ledger,_=l2_simulate_multi(l2_test_open[["A"]],reference_target,1,4,0,0,initial=1000)
assert np.isclose(reference_ledger.equity_end.iloc[-1],reference_cash)
print("L2 CORE CHECKS COMPLETE：含逐腿PnL归因、非法报价拒绝与独立解析参考")


## 02C 基线与诊断实验

02.6的主要比较使用相同协整过滤；若均空仓，结果说明候选没有满足预设规则。附加未过滤对冲比较明确标为诊断，隔离对冲比率的作用。02.7的严格情景保留零结果，活跃诊断则以no_coint_filter为共同参考，一次只改变费用、借券与可得性。

图表包括价差水平、z值、beta、半衰期、净值、回撤和净／总市场敞口。逐腿订单数与完整往返交易次数不同。

In [ ]:
# 02.6 无对冲/固定/滚动/单资产/简单基准，全部共用账本
L2_LEDGERS={}; L2_TRADES={}
for model in ["rolling","fixed","unhedged","single_z","equal_weight","frozen_model"]:
    w=l2_targets(model)
    L2_LEDGERS[model],L2_TRADES[model]=l2_simulate_multi(l2_open,w)
    assert w.index.equals(l2_target.index)
def l2_metrics(x):
    r=x.net_return; vol=r.std(ddof=1)
    return {"total_return":x.wealth.iloc[-1]-1,"annual_return":x.wealth.iloc[-1]**(252/len(x))-1,
        "sharpe_rf0":r.mean()/vol*np.sqrt(252) if vol>1e-12 else np.nan,
        "max_drawdown":x.drawdown.min(),"mean_gross":x.gross_exposure.mean(),
        "mean_net":x.net_exposure.mean(),"turnover":x.turnover.sum(),
        "trade_cost":x.transaction_cost.sum(),"borrow_cost":x.borrow_cost.sum(),
        "funding_cost":x.funding_cost.sum(),"borrow_block_days":int(x.blocked_by_borrow.sum()),
        "annual_vol":vol*np.sqrt(252),"holding_fraction":x.holding.mean(),
        "min_cash_free":x.cash_free.min(),"market_pnl":x.market_pnl.sum(),"net_pnl":x.net_pnl.sum()}
L2_SUMMARY=pd.DataFrame({k:l2_metrics(v) for k,v in L2_LEDGERS.items()}).T
l2_fmt={x:"{:.2%}" for x in ["total_return","annual_return","annual_vol","max_drawdown","mean_gross","mean_net","holding_fraction"]}
l2_fmt.update({x:"{:.2f}" for x in ["sharpe_rf0","turnover","trade_cost","borrow_cost","funding_cost"]})
display(L2_SUMMARY.style.format(l2_fmt))
display(L2_TRADES["rolling"].tail(12))
display(L2_LEDGERS["rolling"].tail(5))
print("滚动策略逐腿订单数：",len(L2_TRADES["rolling"]))
print("主对照fixed只固定金额对冲比，冻结整个价差的附加对照叫frozen_model。")
print("通过I(1)诊断与协整阈值的收盘占比：",((l2_features.coint_p<L2_CONFIG["p_threshold"])&l2_features.i1_diagnostic).mean())
# 简单基准为双腿等权日频再平衡，佣金滑点也照记，不冒称纯粹一次买入持有。
fig,axes=plt.subplots(3,1,figsize=(13,10))
axes[0].plot(l2_features.index,l2_features.z,label="Rolling residual z")
for v in [-4,-2,-.5,.5,2,4]: axes[0].axhline(v,color="gray",alpha=.4,lw=.7)
axes[0].set(title="Residual z-score: entry, exit and failure thresholds")
for k,x in L2_LEDGERS.items():
    axes[1].plot(pd.DatetimeIndex([x.index[0],*x.end_session.tolist()]),np.r_[1.,x.wealth.to_numpy()],label=k)
axes[1].set(title="Same window and account engine: net wealth",ylabel="Wealth")
for k in ["rolling","fixed","unhedged"]:
    x=L2_LEDGERS[k]; axes[2].plot(x.end_session,100*x.drawdown,label=k)
axes[2].set(title="Hedge comparison: drawdowns",ylabel="Drawdown (%)")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
axes[0].plot(l2_features.index,l2_features.coint_p,label="coint p"); axes[0].axhline(.05,color="red")
axes[1].plot(l2_features.index,l2_features.beta,label="rolling beta")
axes[1].plot(l2_features.index,l2_features.half_life,label="local half-life (sessions)",alpha=.6)
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

# 价差和市场净/总敞口是课件必需产物，不只展示z值与净值。
fig,axes=plt.subplots(4,1,figsize=(13,11),sharex=True)
axes[0].plot(l2_features.index,l2_features.spread,label="Rolling log residual")
axes[0].plot(l2_frozen_features.index,l2_frozen_features.spread,label="Frozen-model log residual",alpha=.7)
axes[0].set(title="Spread levels: do not treat these as directly tradable prices")
axes[1].plot(l2_features.index,l2_features.beta,label="Rolling amount hedge beta")
axes[1].axhline(l2_fixed_beta,color="gray",label="Initial fixed beta")
axes[2].plot(l2_features.index,l2_features.half_life,label="Local AR(1) half-life")
axes[2].set(ylabel="Trading sessions")
l2_base=L2_LEDGERS["rolling"]
axes[3].plot(l2_base.index,l2_base.gross_exposure,label="Gross exposure")
axes[3].plot(l2_base.index,l2_base.net_exposure,label="Net exposure")
axes[3].set(ylabel="Fraction of equity",xlabel="Session")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
# 所有基线登记到同一试验台账，使用实际运行指标，不遗漏负结果。
for model,x in L2_LEDGERS.items():
    L2_EXPERIMENTS.append({"id":"model_"+model,"kind":"baseline_comparison",
        "config":L2_CONFIG.copy(),"metrics":l2_metrics(x),
        "data_hashes":[d["sha256"] for d in L2_MANIFEST]})

# 补充诊断：基线过滤未入场时，三种对冲在同一未过滤指令下比较。
# 这是已登记的过滤消融，不改变合格基线，不声称满足协整条件或可交易。
L2_HEDGE_DIAGNOSTIC={}; L2_HEDGE_TRADES={}
for model in ["rolling","fixed","unhedged"]:
    diagnostic_w=l2_targets(model,require_coint=False)
    dx,dt=l2_simulate_multi(l2_open,diagnostic_w)
    L2_HEDGE_DIAGNOSTIC[model]=dx; L2_HEDGE_TRADES[model]=dt
    L2_EXPERIMENTS.append({"id":"diagnostic_hedge_"+model,"kind":"unfiltered_hedge_ablation",
        "coint_filter":False,"metrics":l2_metrics(dx),"config":L2_CONFIG.copy(),
        "data_hashes":[d["sha256"] for d in L2_MANIFEST]})
print("未过滤信号的对冲诊断：这些结果不构成协整套利证据。")
L2_HEDGE_TABLE=pd.DataFrame({k:l2_metrics(v) for k,v in L2_HEDGE_DIAGNOSTIC.items()}).T
display(L2_HEDGE_TABLE.style.format(l2_fmt))
print("未过滤滚动对冲的真实逐腿交易例子：")
display(L2_HEDGE_TRADES["rolling"].head(10))
fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
for k,x in L2_HEDGE_DIAGNOSTIC.items():
    axes[0].plot(pd.DatetimeIndex([x.index[0],*x.end_session.tolist()]),np.r_[1.,x.wealth.to_numpy()],label=k)
    axes[1].plot(x.end_session,100*x.drawdown,label=k)
axes[0].set(title="Diagnostic only: hedge comparison without cointegration filter",ylabel="Wealth")
axes[1].set(ylabel="Drawdown (%)")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()


In [ ]:
# 02.7 一次改变一个因素：交易费、借券、不可借及取消协整过滤
l2_scenarios={"baseline":{},"cost20":{"slippage":20.-L2_CONFIG["commission_bps"]},"borrow15pct":{"borrow_rate":.15},
              "unavailable_stress":{},"no_coint_filter":{},
    "diagnostic_cost20":{"slippage":20.-L2_CONFIG["commission_bps"]},
    "diagnostic_borrow15pct":{"borrow_rate":.15},"diagnostic_unavailable":{},
    "diagnostic_no_cost":{"commission":0.,"slippage":0.,"borrow_rate":0.,"funding":0.}}
l2_assumed_available=pd.DataFrame(True,index=l2_target.index,columns=l2_target.columns)
l2_stress_available=l2_assumed_available.copy()
# 明确标注的假设不可借区间，不是历史借券事件。
l2_stress_available.loc["2011-08-01":"2012-07-31",:]=False
L2_COMPARISONS={}; l2_results=[]
for name,kw in l2_scenarios.items():
    filtered=(name!="no_coint_filter" and not name.startswith("diagnostic_"))
    w=l2_targets(require_coint=filtered)
    availability=l2_stress_available if name in ["unavailable_stress","diagnostic_unavailable"] else l2_assumed_available
    x,tr=l2_simulate_multi(l2_open,w,available=availability,**kw)
    L2_COMPARISONS[name]=x
    l2_results.append({"scenario":name,"leg_orders":len(tr),**l2_metrics(x)})
    L2_EXPERIMENTS.append({"id":name,"pair":L2_CONFIG["assets"],"scope":"2008-2022 development",
        "metrics":l2_metrics(x),"config":L2_CONFIG.copy(),
        "overrides":kw,"coint_filter":filtered,
        "borrow_availability":"synthetic unavailable interval" if name=="unavailable_stress" else "assumed true",
        "data_hashes":[d["sha256"] for d in L2_MANIFEST]})
L2_COMPARISON_TABLE=pd.DataFrame(l2_results).set_index("scenario")
display(L2_COMPARISON_TABLE.style.format(l2_fmt))
fig,ax=plt.subplots(figsize=(13,4))
for name,x in L2_COMPARISONS.items(): ax.plot(x.end_session,x.wealth,label=name)
ax.set(title="Predeclared development experiments; borrowing is assumed",ylabel="Net wealth")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
l2_failures=l2_features.join(l2_state).loc[lambda d:d.reason.isin(["relationship_failed","z_stop","max_holding_exit"])]
print("失效/退出案例：")
display(l2_failures[["z","beta","coint_p","residual_adf_p","half_life","reason"]].head(12))
print("参考解答：相关性不保证协整；普通残差ADF p值不能替代协整校正p值。")
print("beta是近似金额对冲比：先分配总敞口，再按各腿价格转成模型单位。")
print("两腿交易费和空头借券分别计费；周末也计借券日历天；限制现金可能产生融资。")
print("无交易/低收益/过滤失败都是有效研究结果，不放宽规则凑业绩。")
print("作业：任选一个对照，提交预先假设、实际指标、机制解释和否决证据。")
L2_LOG.update(status="core_complete_waiting_for_handcalc_and_review",finished_utc=datetime.now(timezone.utc).isoformat(),
    checks="fees, borrow, funding, unavailable, conservation, compound, single_asset_compatibility, causality, timing passed",
    data_manifest=L2_MANIFEST,assumptions=L2_CONFIG,experiments=L2_EXPERIMENTS)
print(json.dumps(L2_LOG,ensure_ascii=False,indent=2))
# 不将开发实验称独立样本外，不据这份模型单位账本声称可执行套利。

print("真实开发实验自动解读：")
l2_baseline_metrics=L2_COMPARISON_TABLE.loc["baseline"]
for name,r in L2_COMPARISON_TABLE.iterrows():
    print(f"{name}: 累计收益{r.total_return:.2%}，夏普{r.sharpe_rf0:.2f}，最大回撤{r.max_drawdown:.2%}，"
          f"逐腿订单{r.leg_orders:.0f}，相对基线累计收益变化{r.total_return-l2_baseline_metrics.total_return:+.2%}")
if L2_TRADES["rolling"].empty:
    print("过滤后的预设候选未交易：这是当前规则与数据下的否决结果，不放宽阈值补业绩。")
print("上述差异限于开发期；同一pair中滚动检验不等于多次独立统计证据。")

print("活跃交易的费用诊断：固定未过滤信号，以no_coint_filter为共同参考。")
diagnostic_names=["no_coint_filter","diagnostic_cost20","diagnostic_borrow15pct",
    "diagnostic_unavailable","diagnostic_no_cost"]
active_cost_table=L2_COMPARISON_TABLE.loc[diagnostic_names].copy()
active_cost_table["delta_return_vs_same_signal"]=active_cost_table.total_return-L2_COMPARISON_TABLE.loc["no_coint_filter","total_return"]
display(active_cost_table.style.format({**l2_fmt,"delta_return_vs_same_signal":"{:+.2%}"}))
print("活跃诊断的成本差异只解释当前规则和模型费用；它不把未通过的配对升级为合格套利。")


## 02D 手算与结构变化

02.8用假设的固定32／−24单位核对双腿金额、现金、三天借券和净收益，再对照扣费后目标权重的共用引擎。固定份额与目标权重再平衡是两种不同指令，略有数值差异应能解释。

02.9使用已标注的合成数据：模拟前段AR(1)均值回归，后段关系持续偏移。即使两腿收益相关性仍高，残差也可能不再回归；过滤可能滞后。它验证机制，不证明真实套利表现。

In [ ]:
# 02.8 可运行双腿手算：明确标注假设价格，不当作真实行情
# A做多32单位，B做空24单位；固定金额3200/-4800，总敞口8000。
# 周五开盘入场，周一开盘估值；借券按3个日历日而非1个交易日计算。
manual_initial=10000.; manual_beta=1.5; manual_gross=8000.
manual_price=np.array([100.,200.]); manual_next=np.array([103.,201.])
manual_dollars=np.array([manual_gross/(1+manual_beta),-manual_beta*manual_gross/(1+manual_beta)])
manual_units=manual_dollars/manual_price
manual_commission=np.abs(manual_dollars)*.0001
manual_slippage=np.abs(manual_dollars)*.0004
manual_fee=(manual_commission+manual_slippage).sum()
manual_cash=manual_initial-manual_dollars.sum()-manual_fee
manual_short=-np.minimum(manual_dollars,0)
manual_borrow=manual_short*.03*3/365
manual_restricted=manual_short.sum()*1.5
manual_funding=max(manual_restricted-manual_cash,0)*.05*3/365
manual_pnl=manual_units*(manual_next-manual_price)
manual_cash_end=manual_cash-manual_borrow.sum()-manual_funding
manual_end=manual_cash_end+manual_units@manual_next
manual_table=pd.DataFrame({"instrument":["Synthetic A","Synthetic B"],
    "entry_price":manual_price,"signed_amount":manual_dollars,"units":manual_units,
    "commission":manual_commission,"slippage":manual_slippage,"borrow_3days":manual_borrow,
    "price_pnl":manual_pnl})
display(manual_table)
print(f"成交费{manual_fee:.2f}；买卖后现金{manual_cash:.2f}；担保现金{manual_restricted:.2f}")
print(f"双腿市场损益{manual_pnl.sum():.2f}；借券{manual_borrow.sum():.6f}；融资{manual_funding:.6f}")
print(f"周一权益{manual_end:.6f}；净收益{manual_end/manual_initial-1:.4%}")
assert np.allclose(manual_units,[32.,-24.]) and np.isclose(manual_fee,4.)
assert np.isclose(manual_end,manual_initial+manual_pnl.sum()-manual_fee-manual_borrow.sum()-manual_funding)
# 共用引擎按扣费后的权益求目标权重，故与固定32/-24手算略有差异；不是记账误差。
manual_dates=pd.to_datetime(["2020-01-10","2020-01-13","2020-01-14"])
manual_opens=pd.DataFrame([[100.,200.],[103.,201.],[103.,201.]],index=manual_dates,columns=["A","B"])
manual_targets=pd.DataFrame([[.32,-.48],[0.,0.]],index=manual_dates[:2],columns=["A","B"])
manual_ledger,manual_trades=l2_simulate_multi(manual_opens,manual_targets,
    commission=1,slippage=4,borrow_rate=.03,funding=.05,initial=10000,collateral=1.5)
exact_post_equity=manual_initial/(1+.8*.0005)
assert np.isclose(manual_ledger.equity_after_trade.iloc[0],exact_post_equity)
assert np.isclose(manual_ledger.units_A.iloc[0],.32*exact_post_equity/100)
assert np.isclose(manual_ledger.units_B.iloc[0],-.48*exact_post_equity/200)
print("引擎的扣费后目标与入场/平仓订单（第二行还要计两腿退出费）：")
display(manual_ledger[["equity_before","equity_after_trade","cash","cash_restricted",    "borrow_cost","funding_cost","market_pnl","equity_end"]],manual_trades)
L2_EXPERIMENTS.append({"id":"manual_two_leg","kind":"synthetic_accounting_check",    "days":3,"fixed_units_equity":float(manual_end),"target_weight_entry_equity":float(manual_ledger.equity_end.iloc[0])})
print("L2 HAND CALC PASS：双腿仓位、现金、逐腿费用、周末借券及扣费后权重")


In [ ]:
# 02.9 合成结构变化案例：机制验证，不声称历史套利收益
# 固定种子；前段价差AR(1)，后段加入持续偏移，检验关系变坏时是否停止。
rng=np.random.default_rng(L2_CONFIG["seed"])
n=750; synthetic_dates=pd.bdate_range("2010-01-04",periods=n)
synthetic_x=4.+np.cumsum(.001+rng.normal(0,.008,n))
synthetic_u=np.zeros(n)
for k in range(1,n): synthetic_u[k]=.90*synthetic_u[k-1]+rng.normal(0,.007)
synthetic_y=.2+1.1*synthetic_x+synthetic_u
synthetic_break_at=450
synthetic_broken_y=synthetic_y.copy()
synthetic_broken_y[synthetic_break_at:]+=.004*np.arange(n-synthetic_break_at)
synthetic_clean=pd.DataFrame({"A":synthetic_y,"B":synthetic_x},index=synthetic_dates)
synthetic_broken=pd.DataFrame({"A":synthetic_broken_y,"B":synthetic_x},index=synthetic_dates)
synthetic_features=l2_fit_features(synthetic_broken)
synthetic_states=l2_states(synthetic_features)
synthetic_reference_half=-np.log(2)/np.log(.90)
print(f"已知模拟AR系数0.90，理论半衰期{synthetic_reference_half:.2f}交易日。")
print("此案例只校验机制：滚动估计可能暂时吸收变化，过滤不会保证立刻识别断裂。")
synthetic_diagnostics=pd.DataFrame({"return_correlation":[synthetic_clean.diff().iloc[1:].corr().iloc[0,1],    synthetic_broken.diff().iloc[1:].corr().iloc[0,1]],    "sample_type":["synthetic stable relationship","synthetic structural break"]})
display(synthetic_diagnostics)
fig,axes=plt.subplots(3,1,figsize=(13,8),sharex=True)
known_residual=synthetic_broken.A-.2-1.1*synthetic_broken.B
axes[0].plot(synthetic_dates,known_residual,label="Known synthetic residual")
axes[0].set(title="Synthetic mechanism example: relationship break",ylabel="Residual")
axes[1].plot(synthetic_features.index,synthetic_features.z,label="Rolling z")
axes[1].axhline(L2_CONFIG["stop_z"],color="gray"); axes[1].axhline(-L2_CONFIG["stop_z"],color="gray")
axes[2].plot(synthetic_states.index,synthetic_states.state,label="Decision state")
for ax in axes:
    ax.axvline(synthetic_dates[synthetic_break_at],color="red",ls="--",label="Known break")
    ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
print("模拟退出/失效记录：")
synthetic_cases=synthetic_features.join(synthetic_states)
display(synthetic_cases.loc[synthetic_cases.reason.isin(["z_stop","relationship_failed","max_holding_exit"]),    ["z","beta","half_life","coint_p","state","reason"]].head(12))
L2_EXPERIMENTS.append({"id":"synthetic_regime_break","kind":"synthetic_mechanism",    "seed":L2_CONFIG["seed"],"ar_phi":.90,"break_index":synthetic_break_at,    "known_half_life":float(synthetic_reference_half),"not_historical_returns":True})


## 02E 练习参考与最终验收

02.10按本次实际指标生成参考差异，逐个模型核验窗口、费用、账户损益和不可借分支。只有全部断言通过后才打印L2 FINAL PASS并标记completed_and_verified。

标准版课件完成与个人自主研究分开：选择一个对照，先记录假设、只改一个因素，再填写实际指标、机制解释与否决证据。不得把未过滤诊断当作合格协整策略，也不得在历史留出期重新选配对。

In [ ]:
# 02.10 练习参考解答、完整验收与最终运行记录
# 此处只读本次真实输出；不编造个人研究结论，不把开发期称样本外。
reference_rows=[]
for name in ["fixed","unhedged","frozen_model","single_z","equal_weight"]:
    r=L2_SUMMARY.loc[name]; b=L2_SUMMARY.loc["rolling"]
    reference_rows.append({"comparison":name,"delta_total_return_vs_rolling":r.total_return-b.total_return,        "delta_max_drawdown_vs_rolling":r.max_drawdown-b.max_drawdown,        "mean_gross":r.mean_gross,"mean_net":r.mean_net,        "transaction_and_carry_cost":r.trade_cost+r.borrow_cost+r.funding_cost})
print("练习1 参考：保持同一滚动信号，仅改变金额对冲比，读取fixed与rolling；冻结全部模型另读frozen_model。")
display(pd.DataFrame(reference_rows).set_index("comparison").style.format({    "delta_total_return_vs_rolling":"{:+.2%}","delta_max_drawdown_vs_rolling":"{:+.2%}",    "mean_gross":"{:.2%}","mean_net":"{:.2%}","transaction_and_carry_cost":"{:.2f}"}))
print("练习2 参考：过滤基线若无交易，成本情景全为零；活跃诊断应与相同未过滤信号no_coint_filter比较。")
print("未过滤对冲诊断的真实结果：")
display(L2_HEDGE_TABLE.style.format(l2_fmt))
print("练习3 参考：不得按全样本最小p值挑配对；多个候选/多次试验需登记、在开发期选择、在新数据验证。")
print("练习4 参考：半衰期是历史局部估计；即使回归，市场损益仍可能小于交易和借券成本。")
print("进阶：预先登记一个OU、Kalman或因子残差假设，先验证因果性和基线，不在留出期选择自由度。")
# 最终核验逐腿与账户损益完全相等，并要求运行中的所有模型同一窗口。
validation=[]
def l2_final_check(name,ok):
    assert bool(ok),name
    validation.append({"check":name,"status":"PASS"})
l2_final_check("signal available before execution",(l2_interface.available_at<l2_interface.execution_at).all())
l2_final_check("OLS window ends before signal session",(l2_features.fit_last_session<l2_features.index).all())
l2_final_check("model accounting checks ran",np.isclose(reference_ledger.equity_end.iloc[-1],reference_cash))
l2_final_check("manual two-leg calculation",np.isclose(manual_fee,4.) and np.allclose(manual_units,[32.,-24.]))
for name,x in {**L2_LEDGERS,**{"scenario_"+k:v for k,v in L2_COMPARISONS.items()},
               **{"diagnostic_"+k:v for k,v in L2_HEDGE_DIAGNOSTIC.items()}}.items():
    l2_final_check(name+": same time window",x.index.equals(l2_target.index))
    l2_final_check(name+": PnL conservation",np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost-x.borrow_cost-x.funding_cost))
    l2_final_check(name+": cost signs",(x[["transaction_cost","borrow_cost","funding_cost"]]>=0).all().all())
    l2_final_check(name+": compound wealth",np.allclose((1+x.net_return).cumprod(),x.wealth))
    blocked=x.loc[x.blocked_by_borrow]
    l2_final_check(name+": unavailable closes portfolio",blocked.empty or np.allclose(blocked.gross_exposure,0))
L2_VALIDATION=pd.DataFrame(validation)
display(L2_VALIDATION)
L2_LOG.update(status="completed_and_verified",finished_utc=datetime.now(timezone.utc).isoformat(),    validation=L2_VALIDATION.to_dict("records"),data_manifest=L2_MANIFEST,    actual_execution_start=str(l2_target.index[0].date()),    actual_final_valuation=str(L2_LEDGERS["rolling"].end_session.iloc[-1].date()),    models=list(L2_LEDGERS),diagnostic_models=list(L2_HEDGE_DIAGNOSTIC),
    experiments=L2_EXPERIMENTS,configuration=L2_CONFIG.copy(),    personal_exercise_status="template and reference supplied; learner experiment remains independent",    limitations=["fixed ETF pair","historical borrow availability not observed",        "adjusted synthetic units","development data only","no taxes/market impact/real margin calls"])
print("L2 FINAL PASS：本课代码、手算、对照、结构变化示例、图表与运行记录已完成。")
print(json.dumps(L2_LOG,ensure_ascii=False,indent=2,default=str))
# 所有产物均保留在本Colab笔记本与输出；不导出本地文件。

# 简明验收卡：完整台账在上方，不需要下载额外文件。
from IPython.display import Markdown
display(Markdown("### 第二课课件验收完成"))
completion_card=pd.DataFrame({"项目":["02.1至02.10代码","时间因果与共同窗口","逐腿费用与损益守恒",
    "双腿手算及周末借券","对冲/成本/不可借诊断","结构变化示例","运行日志"],
    "状态":["已顺序执行","PASS","PASS","PASS","已完成并标注过滤消融","已完成并标注合成数据","completed_and_verified"]})
display(completion_card)
print(f"验收断言{len(L2_VALIDATION)}项全部通过；实际开发数据{L2_LOG['actual_execution_start']}至{L2_LOG['actual_final_valuation']}。")


# 第3课 多因子定价与选股

本课交付：FF5＋动量归因、真实 SEC 财务因子、公开时间审计、横截面去极值/标准化/行业与规模中性化、Rank IC、分组收益、单因子/等权/训练期加权对照、成本台账与验收。

**先运行第1、2课。** 本课共用第二课自融资多资产账本，所有方案统一成本和执行时间；教学数据窗为2014—2022，训练2014—2018、验证2019—2020、开发评估2021—2022。它们均属研究开发数据，不重新把已经看过的历史称为独立留出集。12只固定存续股票用于教学，存在存续与选择偏差；粗行业标签不是历史GICS。

风险因子：公开 FF5、动量组合用于解释已实现收益，不能当成当天可交易选股信号。选股因子：12−1月动量、年利润/最近资产（ROA）、账面权益/估计市值（B/M）；价值与质量来自实际报表。市值用最后已公开股数及其后拆股更新估计，无法反映所有回购/增发，须视为代理。

财务字段只在 filed 日纽约时间23:59以后可用，保守处理 companyfacts 只有日期的限制；按当时已公开版本选择最近报告期，记录 accession 与报告期，不用期末冒充公开日。拆股仅用于统一价格/股数单位；股息已经包含在复权收益中，不再重复加入现金。

方法顺序：03.01来源→03.02价格/报表→03.03因子面板→03.04归因→03.05 IC与组合→03.06诊断与验收。输出保留在本笔记本，缓存仅在内存，重启需重新下载。

来源：[SEC API](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、[SEC访问规则](https://www.sec.gov/about/developer-resources)、[French因子库](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)。French美国因子自2025年1月转用CRSP CIZ；此处保存当前版本哈希，回溯归因存在数据修订风险。


In [ ]:
# 03.01 共同环境与来源核验（03—05仅保存于Colab）
import sys, subprocess, importlib.metadata as im, time, requests, hashlib, io, zipfile, re, json
subprocess.check_call([sys.executable,'-m','pip','install','-q','yfinance==0.2.66','statsmodels==0.14.5','scikit-learn==1.6.1','lightgbm==4.6.0','beautifulsoup4==4.13.4'])
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yfinance as yf
from bs4 import BeautifulSoup
from IPython.display import display, Markdown
CACHE=globals().get('CACHE',{}); SOURCE_LOG=globals().get('SOURCE_LOG',[])
SESSION=requests.Session(); SESSION.headers.update({'User-Agent':'QuantResearchTeachingNotebook/1.0 academic research','Accept-Encoding':'gzip, deflate'})
def fetch(url):
    if url in CACHE: return CACHE[url]
    if 'sec.gov' in url: time.sleep(0.7) # serial requests, below SEC 10/s limit
    r=SESSION.get(url,timeout=40)
    SOURCE_LOG.append({'url':url,'status':r.status_code,'bytes':len(r.content),'sha256':hashlib.sha256(r.content).hexdigest(),'retrieved_utc':pd.Timestamp.now(tz='UTC').isoformat()})
    r.raise_for_status(); CACHE[url]=r.content; return r.content
PROBE_URLS={
 'facts':'https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json',
 'submissions':'https://data.sec.gov/submissions/CIK0000320193.json',
 'french':'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip',
 'LM':'https://drive.google.com/uc?export=download&id=1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF'}
for name,url in PROBE_URLS.items():
    if name=='LM': continue # full dictionary schema and fallback checked in 04.01
    try: print(name,'OK',len(fetch(url)))
    except Exception as e: print(name,type(e).__name__,str(e)[:160])
try:
    meta=json.loads(fetch(PROBE_URLS['submissions'])); q=pd.DataFrame(meta['filings']['recent']); row=q[q.form.eq('10-K')].iloc[0]
    ARCHIVE_PROBE='https://www.sec.gov/Archives/edgar/data/320193/'+row.accessionNumber.replace('-','')+'/'+row.primaryDocument
    print('filing',len(fetch(ARCHIVE_PROBE)))
except Exception as e: print('filing',type(e).__name__,str(e)[:160])
display(pd.DataFrame(SOURCE_LOG))
print('环境',sys.version.split()[0],{p:im.version(p) for p in ['numpy','pandas','yfinance','scikit-learn','lightgbm']})


In [ ]:
# 03.02 固定股票样本、复权价格与真实SEC报表缓存
from urllib.parse import urljoin
subprocess.check_call([sys.executable,'-m','pip','install','-q','pypdf==5.6.1'])
UNIVERSE={'AAPL':(320193,'tech'),'MSFT':(789019,'tech'),'IBM':(51143,'tech'),'INTC':(50863,'tech'),
 'KO':(21344,'consumer'),'PEP':(77476,'consumer'),'MCD':(63908,'consumer'),'PG':(80424,'consumer'),
 'XOM':(34088,'energy'),'CVX':(93410,'energy'),'CAT':(18230,'industrial'),'MMM':(66740,'industrial')}
ASSETS=list(UNIVERSE); NY='America/New_York'; SEED=42
START='2012-01-01'; END='2023-01-01'; TRAIN_END=pd.Timestamp('2019-01-01'); VALID_END=pd.Timestamp('2021-01-01')
raw=yf.download(ASSETS+['SPY'],start=START,end=END,auto_adjust=False,actions=True,progress=False,threads=False)
assert not raw.empty, 'Yahoo source empty: stop, do not fabricate prices'
CLOSE=raw['Adj Close'].reindex(columns=ASSETS+['SPY']).dropna(how='any')
QUOTE=raw['Close'].reindex(index=CLOSE.index,columns=CLOSE.columns)
OPEN=raw['Open'].reindex(index=CLOSE.index,columns=CLOSE.columns)*(CLOSE/QUOTE)
assert (OPEN>0).all().all() and (CLOSE>0).all().all()
SPLITS={}; FACTS={}
for t,(cik,industry) in UNIVERSE.items():
    sp=yf.Ticker(t).get_splits(period='max')
    sp.index=pd.DatetimeIndex(sp.index).tz_localize(None).normalize(); SPLITS[t]=sp[sp.ne(0)].copy()
    FACTS[t]=json.loads(fetch(f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json'))
PRICE_AUDIT={'provider':'Yahoo via yfinance','start':str(CLOSE.index.min()),'end':str(CLOSE.index.max()),
 'sha256':hashlib.sha256(pd.util.hash_pandas_object(raw,index=True).values.tobytes()).hexdigest(),
 'scope':'fixed survivors; adjusted synthetic units; no delisted universe','missing_removed':len(raw)-len(CLOSE)}
# 公司官网公开镜像：仅链接发现，不猜测/绕过SEC封禁。
IBM_IR='https://www.ibm.com/investor/services/investor-information'
IR_HTML=BeautifulSoup(fetch(IBM_IR),'html.parser')
IBM_LINKS=[{'label':a.get_text(' ',strip=True),'url':urljoin(IBM_IR,a['href'])} for a in IR_HTML.select('a[href]')
           if re.search(r'10.?[KQ]',a.get_text(' ',strip=True),re.I)]
print('价格审计',PRICE_AUDIT); display(pd.DataFrame(IBM_LINKS).tail(18))
# Google Drive公开下载端点；若返回HTML而非CSV，验收会失败并显示真实状态。
LM_URL='https://drive.usercontent.google.com/download?id=1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF&export=download&authuser=0&confirm=t'
try:
    LM_BYTES=fetch(LM_URL); lm_probe=pd.read_csv(io.BytesIO(LM_BYTES)); assert {'Word','Negative','Positive'}.issubset(lm_probe.columns) and len(lm_probe)>10000,'Download returned HTML/quota page, not dictionary'; print('LM CSV schema verified',lm_probe.shape)
except Exception as e: LM_BYTES=None; print('LM_UNAVAILABLE',type(e).__name__,str(e)[:120])
print('03.02 数据读取结束；所有缓存仅在Colab内存')


In [ ]:
# 03.03 点时财务因子、拆股单位与行业/规模中性化
TAGS={'assets':('us-gaap',['Assets'],'USD'),
 'equity':('us-gaap',['StockholdersEquity','StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest'],'USD'),
 'income':('us-gaap',['NetIncomeLoss','ProfitLoss'],'USD'),
 'shares':('dei',['EntityCommonStockSharesOutstanding'],'shares')}
FACT_AUDIT=[]
def fact_events(t,key):
    ns,tags,unit=TAGS[key]; records=[]
    for priority,tag in enumerate(tags):
        for x in FACTS[t]['facts'].get(ns,{}).get(tag,{}).get('units',{}).get(unit,[]):
            if x.get('form') not in ['10-K','10-Q','10-K/A','10-Q/A']: continue
            if key=='income':
                if 'start' not in x or not 330<=(pd.Timestamp(x['end'])-pd.Timestamp(x['start'])).days<=380: continue
            records.append(dict(x,tag=tag,priority=priority))
    assert records, (t,key,'no usable SEC tag')
    f=pd.DataFrame(records); f['end']=pd.to_datetime(f.end); f['filed']=pd.to_datetime(f.filed)
    f=f.sort_values(['filed','end','priority','accn'],ascending=[True,True,False,True])
    current=None; updates=[]
    for _,r in f.iterrows():
        if current is None or r.end>current.end or (r.end==current.end and r.filed>=current.filed): current=r
        else: continue # later comparative old periods do not replace the latest known period
        updates.append({'available_at':(r.filed+pd.Timedelta(hours=23,minutes=59)).tz_localize(NY),
                        key:float(r.val),key+'_end':r.end,key+'_accn':r.accn,key+'_tag':r.tag})
    return pd.DataFrame(updates).drop_duplicates('available_at',keep='last').sort_values('available_at')
FUND={}; dates=CLOSE.index
for t in ASSETS:
    p=pd.DataFrame({'date':dates,'decision_at':(dates+pd.Timedelta(hours=16)).tz_localize(NY)})
    for key in TAGS:
        ev=fact_events(t,key).rename(columns={'available_at':key+'_available_at'})
        p=pd.merge_asof(p.sort_values('decision_at'),ev,left_on='decision_at',right_on=key+'_available_at',direction='backward')
    p=p.set_index('date'); sp=SPLITS[t]
    # Yahoo Close统一到当前拆股股数基准；还原当日名义报价后与当时公开股数匹配。
    future_split=pd.Series([sp.loc[sp.index>d].prod() for d in dates],index=dates)
    nominal_close=QUOTE[t]*future_split
    share_update=pd.Series([sp.loc[(sp.index>e)&(sp.index<=d)].prod() if pd.notna(e) else np.nan for d,e in zip(dates,p.shares_end)],index=dates)
    p['marketcap_est']=nominal_close*p.shares*share_update
    p['value']=p.equity/p.marketcap_est.where(p.marketcap_est>0)
    p['quality']=p.income/p.assets.where(p.assets>0)
    p['momentum']=CLOSE[t].shift(21)/CLOSE[t].shift(252)-1
    p['low_vol']=-CLOSE[t].pct_change(fill_method=None).rolling(63).std()
    p['size']=np.log(p.marketcap_est.where(p.marketcap_est>0))
    for key in TAGS:
        p[key+'_age_days']=(pd.Series(dates,index=dates)-p[key+'_end']).dt.days
        assert (p[key+'_available_at'].dropna()<=p.loc[p[key+'_available_at'].notna(),'decision_at']).all()
    stale=p[[k+'_age_days' for k in TAGS]].max(axis=1)>550
    p.loc[stale,['quality','value','size']]=np.nan
    p['ticker']=t; p['industry']=UNIVERSE[t][1]; FUND[t]=p
PANEL=pd.concat(FUND.values()).reset_index().sort_values(['date','ticker']).reset_index(drop=True)
PANEL=PANEL[PANEL.date.between('2014-01-01','2022-12-31')].copy()
FACTOR_NAMES=['momentum','quality','value']
def neutralize(g,col):
    ok=g[col].notna()&g['size'].notna(); out=pd.Series(np.nan,index=g.index)
    if ok.sum()<8: return out
    y=g.loc[ok,col].clip(g.loc[ok,col].quantile(.05),g.loc[ok,col].quantile(.95))
    y=(y-y.mean())/max(y.std(ddof=0),1e-12)
    industry=pd.get_dummies(g.loc[ok,'industry'],dtype=float)
    sz=g.loc[ok,'size']; sz=(sz-sz.mean())/max(sz.std(ddof=0),1e-12)
    X=np.column_stack([industry.to_numpy(),sz.to_numpy()]); residual=y.to_numpy()-X@np.linalg.lstsq(X,y,rcond=None)[0]
    out.loc[ok]=residual/max(residual.std(),1e-12); return out
for col in FACTOR_NAMES:
    PANEL[col+'_z']=pd.concat([neutralize(g,col) for _,g in PANEL.groupby('date',sort=False)]).reindex(PANEL.index)
MONTH_DATES=pd.DatetimeIndex(PANEL.groupby(PANEL.date.dt.to_period('M')).date.max().values)
MONTH=PANEL[PANEL.date.isin(MONTH_DATES)].dropna(subset=[f+'_z' for f in FACTOR_NAMES]).copy()
L3_DICTIONARY=pd.DataFrame([
 ['momentum','adjusted close[t-21]/close[t-252]-1','t close','skip latest month'],
 ['quality','annual net income/latest assets','filed day 23:59 NY','ROA proxy, mixed balance sheet date'],
 ['value','equity/estimated nominal marketcap','filed day 23:59 NY','lagged disclosed shares; split correction'],
 ['neutralization','5/95% clip -> z -> industry/size OLS residual','same date cross-section','fixed study industry tags, small universe']],columns=['feature','definition','available','limitation'])
display(L3_DICTIONARY); display(MONTH[['date','ticker','momentum','quality','value','shares_end','income_available_at','equity_accn']].tail(12))
print('03.03 PASS',len(PANEL),'daily rows',len(MONTH),'monthly rows; missing rates'); display(PANEL[FACTOR_NAMES+['size']].isna().mean())


In [ ]:
# 03.04 French研究因子、FF3/FF5+动量与共同策略归因
import statsmodels.api as sm
def french_monthly(url):
    z=zipfile.ZipFile(io.BytesIO(fetch(url))); txt=z.read(z.namelist()[0]).decode('utf-8-sig')
    lines=txt.splitlines(); start=next(i for i,x in enumerate(lines) if re.match(r'^\s*\d{6},',x)); head=lines[start-1]
    rows=[]
    for x in lines[start:]:
        if not re.match(r'^\s*\d{6},',x): break
        rows.append(x)
    f=pd.read_csv(io.StringIO(head+'\n'+'\n'.join(rows)),index_col=0)
    f.columns=f.columns.str.strip(); f.index=pd.PeriodIndex(f.index.astype(str),freq='M')
    return f.replace([-99.99,-999],np.nan)/100
FF5=french_monthly(PROBE_URLS['french'])
MOM=french_monthly('https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip')
FF=FF5.join(MOM.rename(columns={MOM.columns[0]:'Mom'}),how='inner')
# 第一课风格的SPY趋势策略重建。归因用close-to-close收益，与月度French口径对齐。
trend=(CLOSE.SPY.rolling(50).mean()>CLOSE.SPY.rolling(200).mean()).astype(float)
close_ret=CLOSE.SPY.pct_change(fill_method=None)
strategy_daily=.8*trend.shift(1).fillna(0)*close_ret # descriptive frictionless close-based research proxy
strategy_month=(1+strategy_daily).groupby(strategy_daily.index.to_period('M')).prod()-1
research=pd.concat([strategy_month.rename('strategy'),FF],axis=1).dropna().loc['2014-01':'2022-12']
ATTR={}; attr_rows=[]
for name,cols in {'FF3':['Mkt-RF','SMB','HML'],'FF5':['Mkt-RF','SMB','HML','RMW','CMA'],'FF5+Mom':['Mkt-RF','SMB','HML','RMW','CMA','Mom']}.items():
    m=sm.OLS(research.strategy-research.RF,sm.add_constant(research[cols])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
    ATTR[name]=m
    for term in m.params.index: attr_rows.append({'model':name,'term':term,'coefficient':m.params[term],'HAC_t':m.tvalues[term],'R2':m.rsquared})
display(pd.DataFrame(attr_rows)); print('月alpha为回归截距，不是alpha×12保证收益；该归因为无成本close代理，不混入open账本。')
fig,ax=plt.subplots(1,2,figsize=(13,4)); research[['strategy','Mkt-RF','HML','RMW','Mom']].cumsum().plot(ax=ax[0],title='Monthly arithmetic cumulative factor diagnostics')
FF.loc['2014-01':'2022-12',['Mkt-RF','SMB','HML','RMW','CMA','Mom']].corr().plot.bar(ax=ax[1],title='Published factor correlations'); plt.tight_layout(); plt.show()
assert len(research)==108 and np.isfinite(research.to_numpy()).all()
print('03.04 PASS: FF3, FF5, FF5+Mom; percent converted once; 108 aligned months')


In [ ]:
# 03.05 Rank IC、训练期加权、分组与统一成本组合
assert 'l2_simulate_multi' in globals(),'请先顺序运行第1、2课'
def forward_open(h): return OPEN[ASSETS].shift(-(h+1))/OPEN[ASSETS].shift(-1)-1
def attach_forward(df,h):
    y=forward_open(h).stack().rename('forward_return').rename_axis(['date','ticker']).reset_index()
    out=df.merge(y,on=['date','ticker'],how='left'); out['label_end']=out.date.map(pd.Series(CLOSE.index,index=CLOSE.index).shift(-(h+1)))
    return out
M21=attach_forward(MONTH,21).dropna(subset=['forward_return'])
IC_ROWS=[]
for date,g in M21.groupby('date'):
    for f in FACTOR_NAMES: IC_ROWS.append({'date':date,'factor':f,'RankIC':g[f+'_z'].corr(g.forward_return,method='spearman'),'raw_RankIC':g[f].corr(g.forward_return,method='spearman'),'n':len(g),'label_end':g.label_end.max()})
IC=pd.DataFrame(IC_ROWS)
IC['fold']=np.select([IC.date<TRAIN_END,IC.date<VALID_END],['train','validation'],default='development_eval')
print('同一日期/同一股票集合：中性化前后Rank IC对照')
display(IC.groupby(['fold','factor'])[['raw_RankIC','RankIC']].mean())
train_ic=IC[(IC.date<TRAIN_END)&(IC.label_end<TRAIN_END)].groupby('factor').RankIC.mean().reindex(FACTOR_NAMES)
weights=train_ic.clip(lower=0)
FACTOR_WEIGHTS=weights/weights.sum() if weights.sum()>0 else pd.Series(1/3,index=FACTOR_NAMES)
MONTH['equal']=MONTH[[f+'_z' for f in FACTOR_NAMES]].mean(axis=1)
MONTH['trained']=sum(MONTH[f+'_z']*FACTOR_WEIGHTS[f] for f in FACTOR_NAMES)
SCORE_COLUMNS={f:f+'_z' for f in FACTOR_NAMES}|{'equal':'equal','trained':'trained'}
exec_dates=OPEN.loc['2014-02-01':'2022-12-29'].index
L3_TARGETS={}; L3_LEDGERS={}; L3_TRADES={}
def monthly_target(score_col,quantile=None):
    decisions=pd.DataFrame(0.,index=MONTH_DATES,columns=ASSETS)
    for date,g in MONTH.groupby('date'):
        ranked=g.sort_values([score_col,'ticker']); chosen=ranked.tail(3)
        if quantile is not None: chosen=np.array_split(ranked,3)[quantile]
        if len(chosen): decisions.loc[date,chosen.ticker]=.8/len(chosen)
    # 收盘信号只能从下个交易日开盘启用；月内每日维护同一目标权重，费用覆盖维护交易。
    sparse=decisions.reindex(OPEN.index)
    return sparse.shift(1).ffill().fillna(0).reindex(exec_dates)
for name,col in SCORE_COLUMNS.items():
    target=monthly_target(col); L3_TARGETS[name]=target
    ledger,trades=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    L3_LEDGERS[name]=ledger; L3_TRADES[name]=trades
for k in range(3):
    target=monthly_target('equal',k); name='Q'+str(k+1); L3_TARGETS[name]=target
    L3_LEDGERS[name],L3_TRADES[name]=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
def ledger_stats(x):
    growth=(1+x.net_return).cumprod(); dd=growth/growth.cummax().clip(lower=1)-1
    return {'net_return':growth.iloc[-1]-1,'annualized':growth.iloc[-1]**(252/len(x))-1,'max_drawdown':dd.min(),
            'Sharpe_0rf':x.net_return.mean()/max(x.net_return.std(),1e-12)*np.sqrt(252),
            'turnover_sum':x.turnover.sum(),'fees':x.transaction_cost.sum()}
L3_RESULT=pd.DataFrame([{'model':n,'fold':fold,**ledger_stats(x.loc[a:b])} for n,x in L3_LEDGERS.items()
    for fold,a,b in [('train','2014-02-01','2018-12-31'),('validation','2019-01-01','2020-12-31'),('development_eval','2021-01-01','2022-12-29')]]).set_index(['fold','model'])
display(pd.DataFrame({'train_mean_IC':train_ic,'frozen_weight':FACTOR_WEIGHTS})); display(L3_RESULT)
print('权重仅训练期估计；验证与开发评估报告全部对照，未按最后一段结果挑选赢家。')
fig,ax=plt.subplots(1,2,figsize=(13,4)); pd.DataFrame({n:x.wealth for n,x in L3_LEDGERS.items() if not n.startswith('Q')}).plot(ax=ax[0],title='Same cost, daily target maintenance')
pd.DataFrame({n:x.wealth for n,x in L3_LEDGERS.items() if n.startswith('Q')}).plot(ax=ax[1],title='Tercile portfolios, net of costs'); plt.tight_layout(); plt.show()
print('03.05 PASS',len(IC),'IC observations;',len(L3_LEDGERS),'shared-ledger portfolios')


In [ ]:
# 03.06 IC衰减、相关性、行业敞口与因果性验收
DECAY=[]
for h in [5,21,63]:
    joined=attach_forward(MONTH,h).dropna(subset=['forward_return'])
    for f in FACTOR_NAMES:
        vals=[g[f+'_z'].corr(g.forward_return,method='spearman') for _,g in joined.groupby('date')]
        DECAY.append({'horizon_sessions':h,'factor':f,'mean_RankIC':np.nanmean(vals),'months':len(vals)})
display(pd.DataFrame(DECAY)); display(IC.groupby('factor').RankIC.agg(['mean','std','count']))
corr=MONTH[[f+'_z' for f in FACTOR_NAMES]].corr(); display(corr)
industry_weights=pd.DataFrame({group:L3_TARGETS['equal'][[t for t in ASSETS if UNIVERSE[t][1]==group]].sum(axis=1) for group in sorted({v[1] for v in UNIVERSE.values()})})
display(industry_weights.mean().rename('average_target_weight').to_frame())
fig,ax=plt.subplots(2,2,figsize=(14,8)); IC.pivot(index='date',columns='factor',values='RankIC').rolling(12,min_periods=3).mean().plot(ax=ax[0,0],title='12-month rolling mean Rank IC')
for f,g in pd.DataFrame(DECAY).groupby('factor'): ax[0,1].plot(g.horizon_sessions,g.mean_RankIC,marker='o',label=f)
ax[0,1].legend(); ax[0,1].set_title('Horizon decay diagnostic')
industry_weights.plot.area(ax=ax[1,0],title='Study industry exposure'); L3_LEDGERS['equal'].turnover.rolling(21).sum().plot(ax=ax[1,1],title='21-session turnover including maintenance'); plt.tight_layout(); plt.show()
CHECKS=[]
def check(name,ok):
    assert bool(ok),name
    CHECKS.append({'check':name,'status':'PASS'})
check('SEC PIT availability audit',all((PANEL[k+'_available_at'].dropna()<=PANEL.loc[PANEL[k+'_available_at'].notna(),'decision_at']).all() for k in TAGS))
check('training IC labels matured before validation',IC[(IC.date<TRAIN_END)&(IC.label_end<TRAIN_END)].label_end.max()<TRAIN_END)
check('daily asset-date unique',not PANEL.duplicated(['date','ticker']).any())
check('French six factors and RF',set(['Mkt-RF','SMB','HML','RMW','CMA','Mom','RF']).issubset(FF.columns))
# 中性化的行业残差均值应近零；规模残差正交。
for f in FACTOR_NAMES:
    valid=PANEL.dropna(subset=[f+'_z','size'])
    check(f+' industry orthogonality',valid.groupby(['date','industry'])[f+'_z'].mean().abs().max()<1e-7)
    dots=valid.groupby('date').apply(lambda g: np.dot(g[f+'_z'],g['size']-g['size'].mean()),include_groups=False)
    check(f+' size orthogonality',dots.abs().max()<1e-6)
for name,x in L3_LEDGERS.items():
    check(name+' common window',x.index.equals(exec_dates))
    check(name+' equity conservation',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
    check(name+' long-only and gross cap',(L3_TARGETS[name]>=0).all().all() and (x.gross_exposure<=.800001).all())
    check(name+' compounded wealth',np.allclose(x.wealth,(1+x.net_return).cumprod()))
# 有效报表公开时间与信号时间各自保存；12-1动量不能读当日未来价格。
probe_date=MONTH.date.iloc[len(MONTH)//2]; probe_t=MONTH.ticker.iloc[len(MONTH)//2]
past=CLOSE[probe_t].loc[:probe_date]; expected=past.iloc[-22]/past.iloc[-253]-1
check('momentum prefix recomputation',np.isclose(MONTH.loc[(MONTH.date==probe_date)&(MONTH.ticker==probe_t),'momentum'].iloc[0],expected))
# 对03.05同一有成本账户做close-to-close因子归因，避免把开盘收益直接对上收盘因子。
net_account=L3_LEDGERS['equal']
account_close=net_account.cash+sum(net_account['units_'+a]*CLOSE[a].reindex(net_account.index) for a in ASSETS)
month_equity=account_close.groupby(account_close.index.to_period('M')).last()
month_equity.loc[pd.Period('2014-01',freq='M')]=10000.; month_equity=month_equity.sort_index()
actual_month=month_equity.pct_change(fill_method=None).rename('account_net')
# 最后2022-12仅到29日收盘，排除这个不完整月份。
actual_factor_data=pd.concat([actual_month,FF],axis=1).loc['2014-02':'2022-11'].dropna()
ACTUAL_ATTR=sm.OLS(actual_factor_data.account_net-actual_factor_data.RF,
                  sm.add_constant(actual_factor_data[['Mkt-RF','SMB','HML','RMW','CMA','Mom']])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
print('同一个等权因子有成本账户：完整月份收盘权益，FF5+Mom归因')
display(pd.DataFrame({'coefficient':ACTUAL_ATTR.params,'HAC_t':ACTUAL_ATTR.tvalues,'p_value':ACTUAL_ATTR.pvalues}))
check('actual portfolio FF5+Mom aligns full months',len(actual_factor_data)==106 and actual_factor_data.index.max()==pd.Period('2022-11',freq='M'))
L3_VALIDATION=pd.DataFrame(CHECKS); display(L3_VALIDATION)
L3_CARD={'course':3,'status':'verified','data_card':PRICE_AUDIT,'weights':FACTOR_WEIGHTS.to_dict(),'checks':len(CHECKS),
 'limitations':['12 fixed survivors','coarse static industry','marketcap uses lagged shares','SEC revised taxonomy/vendor revised prices','daily target maintenance, not literal monthly buy-and-hold','HAC descriptive inference; separate close proxy and same-cost account attribution','no pristine holdout claim']}
print(json.dumps(L3_CARD,ensure_ascii=False,indent=2)); print('L3 FINAL PASS')


# 第4课 替代数据与金融NLP

交付：真实10-K/10-Q文档清单、SEC acceptance/accession/报告期/版本映射、全文提取与段落质量检查、LM情绪及不确定性、否定处理、五交易日超额方向标签、事前市场模型事件研究、价格/价格+词典/价格+TF-IDF Logistic 对照和增量评价。

SEC全文端点本次在Colab返回403，遵守访问限制；使用公司官网列出的IBM原始申报PDF镜像，不改变申报元数据。以报告期、表单、CIK匹配SEC accession；缺失、重复链接、报告期不符、版本歧义均记录并剔除。保存完整下载字节哈希；主文件包含的引用外部附件不自动假定已经下载。镜像历史内容可能更新，无法保证与当年SEC字节完全相同，这是实际PIT局限。

LM官方[词典](https://sraf.nd.edu/loughranmcdonald-master-dictionary/)用于学术教学；官方CSV/XLSX因下载配额不可用时，使用[pysentiment2 0.1.1随附的LM.csv快照](https://nickderobertis.github.io/pysentiment/_modules/pysentiment2/lm.html)，显示实际来源与哈希，不冒充当前官网版本。保存实际词典版本，主实验冻结2011年已存在的情绪词标记，仅用基础词源，不使用2025词频。该重建不是完整2011字典快照，保留修订偏差。TF-IDF词表、数值填补和标准化均只fit训练集。FinBERT仅列为冻结推断扩展，不微调大模型。

时间链：保留acceptance原字段；执行下界取精确时间与filed日纽约时间23:59较晚者→加60分钟处理延迟→下一个可交易开盘→第五个后续开盘退出。即使有精确acceptance也采用保守filed日末下界；旧元数据缺失精确时间时只使用filed日末，并保留原字段缺失状态。120个已经结束的开盘到开盘收益估计市场模型，未来五日仅构造标签；同一报告组不跨集合，切分边界清除未成熟标签。

IBM单公司2014—2022小样本仅用于流程、数据质量和增量诊断，不能声称已经证明文本有交易价值。训练≤2018、验证2019—2020、开发评估2021—2022；不随机打乱，不根据最后一段挑词、挑事件或挑模型。事件研究用市场模型异常收益；分类标签使用股票与SPY同窗复合收益之差，两者定义分别输出。

参考：[SEC API](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、[IBM原始报告镜像目录](https://www.ibm.com/investor/services/investor-information)、[sklearn Pipeline防泄漏](https://scikit-learn.org/stable/common_pitfalls.html)。


In [ ]:
# 04.01 真实申报元数据、公司官网镜像映射与LM词典
from pypdf import PdfReader
IBM_META=json.loads(fetch('https://data.sec.gov/submissions/CIK0000051143.json'))
parts=[pd.DataFrame(IBM_META['filings']['recent'])]
for file in IBM_META['filings'].get('files',[]):
    if file['filingTo']>='2014-01-01' and file['filingFrom']<='2022-12-31':
        parts.append(pd.DataFrame(json.loads(fetch('https://data.sec.gov/submissions/'+file['name']))))
FILINGS=pd.concat(parts,ignore_index=True).drop_duplicates('accessionNumber')
FILINGS=FILINGS[FILINGS.form.isin(['10-K','10-Q','10-K/A','10-Q/A'])].copy()
FILINGS['reportDate']=pd.to_datetime(FILINGS.reportDate); FILINGS['filingDate']=pd.to_datetime(FILINGS.filingDate)
FILINGS['acceptance_raw']=FILINGS.get('acceptanceDateTime',pd.Series(index=FILINGS.index,dtype=str))
FILINGS['acceptance_at']=pd.to_datetime(FILINGS.acceptance_raw,utc=True,errors='coerce').dt.tz_convert(NY)
FILINGS['filed_end_at']=(FILINGS.filingDate+pd.Timedelta(hours=23,minutes=59)).dt.tz_localize(NY)
# 保守事件版本：有精确时间也不早于filed日末；避免旧API时区或当天处理能力的乐观假设。
FILINGS['public_bound']=FILINGS[['acceptance_at','filed_end_at']].max(axis=1)
FILINGS['ready_at']=FILINGS.public_bound+pd.Timedelta(hours=1)
FILINGS=FILINGS[FILINGS.filingDate.between('2014-01-01','2022-12-31')]
LM_XLSX_URL='https://drive.google.com/uc?export=download&id=1P3KtWuDyQT2Eu_VwTU3D2sEeQxR9dWCS'
subprocess.check_call([sys.executable,'-m','pip','install','-q','openpyxl==3.1.5'])
try:
    LM=pd.read_excel(io.BytesIO(fetch(LM_XLSX_URL)))
    LM_SOURCE='official 1993-2025 workbook'
    LM_BLOB=fetch(LM_XLSX_URL)
except Exception as e:
    print('Official workbook unavailable:',str(e)); print('Use declared pinned package snapshot; do not claim latest official file')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps','pysentiment2==0.1.1'])
    LM_BLOB=im.distribution('pysentiment2').locate_file('pysentiment2/static/LM.csv').read_bytes()
    LM=pd.read_csv(io.BytesIO(LM_BLOB)); LM_SOURCE='pysentiment2 0.1.1 bundled LM.csv (package released 2020)'
print('Dictionary source',LM_SOURCE,'sha256',hashlib.sha256(LM_BLOB).hexdigest())
assert {'Word','Negative','Positive','Uncertainty','Source'}.issubset(LM.columns) and len(LM)>10000,'Official LM schema failed'
LM.columns=LM.columns.str.strip(); LM['Word']=LM.Word.astype(str).str.upper()
base=LM[LM.Source.astype(str).eq('12of12inf')]
# 只取当时已有的词类标签，正负年标记是加入/移除年，不把negative年当情绪负值。
LEX={c:set(base.loc[base[c].between(1,2011),'Word']) for c in ['Positive','Negative','Uncertainty']}
EFFECTIVE_VOCAB=set(base.Word); NEGATORS={'NO','NOT','NEVER','NEITHER','NOR','WITHOUT','CANNOT'}
print('LM snapshot rows',len(LM),'frozen vocab',len(EFFECTIVE_VOCAB),'categories',{k:len(v) for k,v in LEX.items()})
# 来源仅在官网可见链接中选择；没有精确映射的链接不会进入实验。
LINK_MANIFEST=[]
for link in IBM_LINKS:
    url=link['url']; label=link['label']; name=url.rsplit('/',1)[-1]
    year_match=re.search(r'(20\d{2})',name)
    year=int(year_match[1]) if year_match else (2000+int(re.search(r'^[123]q(\d{2})_',name,re.I)[1]) if re.search(r'^[123]q(\d{2})_',name,re.I) else None)
    form='10-K' if label=='10-K' else '10-Q'
    qm=re.search(r'([123])(?:st|nd|rd) Quarter',label); q=int(qm[1]) if qm else 4
    if year is None or not 2013<=year<=2022: continue
    report_end=pd.Timestamp(year=year,month=3*q,day=1)+pd.offsets.MonthEnd(0)
    m=FILINGS[(FILINGS.form==form)&(FILINGS.reportDate==report_end)]
    status='mapped' if len(m)==1 else 'missing_or_ambiguous'
    record={'ticker':'IBM','CIK':51143,'form':form,'report_end':report_end,'url':url,'status':status,'directory_label':label}
    if len(m)==1: record.update(m.iloc[0][['accessionNumber','acceptance_raw','acceptance_at','filed_end_at','ready_at','filingDate']].to_dict())
    LINK_MANIFEST.append(record)
LINK_MANIFEST=pd.DataFrame(LINK_MANIFEST)
LINK_MANIFEST.loc[LINK_MANIFEST.url.duplicated(keep=False),'status']='duplicate_directory_url'
print('Amendments kept in separate audit, never merged into original event'); display(FILINGS[FILINGS.form.str.endswith('/A')][['form','reportDate','accessionNumber','filingDate']])
display(LINK_MANIFEST[['form','report_end','status','accessionNumber','url']]); print('04.01 manifest prepared')


In [ ]:
# 04.02 原始PDF全文提取、质量筛选、段落与否定词典特征
TEXTS={}; DOC_ROWS=[]; EXTRACT_CACHE=globals().get('EXTRACT_CACHE',{})
def text_features(text):
    seq=re.findall(r'[A-Za-z]+|[.!?;:]',text.upper()); counts=dict(positive=0,negative=0,uncertainty=0,negated=0); history=[]; effective=0
    for token in seq:
        if token in '.!?;:': history=[]; continue
        effective+=int(token in EFFECTIVE_VOCAB)
        neg=any(x in NEGATORS for x in history[-3:])
        if token in LEX['Positive']:
            counts['negative' if neg else 'positive']+=1; counts['negated']+=int(neg)
        if token in LEX['Negative']:
            if not neg: counts['negative']+=1
            else: counts['negated']+=1
        if token in LEX['Uncertainty']: counts['uncertainty']+=1
        history.append(token)
    den=max(effective,1)
    return {'sentiment':(counts['positive']-counts['negative'])/den,'negative':counts['negative']/den,
            'uncertainty':counts['uncertainty']/den,'negated_rate':counts['negated']/den,'effective_tokens':effective,'log_length':np.log1p(effective)}
for _,r in LINK_MANIFEST.iterrows():
    row=r.to_dict(); row['text_scope']='complete downloaded main filing PDF; external incorporated exhibits not collected'
    if r.status!='mapped': DOC_ROWS.append(row); continue
    try:
        blob=fetch(r.url); assert blob.startswith(b'%PDF'),'not a PDF'
        blob_hash=hashlib.sha256(blob).hexdigest()
        if blob_hash in EXTRACT_CACHE: pages=EXTRACT_CACHE[blob_hash]
        else:
            reader=PdfReader(io.BytesIO(blob)); pages=[page.extract_text() or '' for page in reader.pages]; EXTRACT_CACHE[blob_hash]=pages
        full='\n\n'.join(pages); header=re.sub(r'\s+',' ',full[:7000]).upper()
        expected=r.report_end.strftime('%B').upper()+r.report_end.strftime(' %d, %Y')
        # 不仅依赖官网链接标签：验证文本中的表单、年份、报告期和发行人。
        dates_found=[pd.to_datetime(m.group(0)).normalize() for m in re.finditer(r'(?:MARCH|JUNE|SEPTEMBER|DECEMBER)\s+\d{1,2},?\s+20\d{2}',header)]
        assert r.report_end in dates_found,'PDF report period mismatch'
        assert 'INTERNATIONAL BUSINESS MACHINES' in header,'issuer mismatch'
        assert 'FORM10'+r.form[-1] in re.sub(r'[^A-Z0-9]','',header),'form mismatch'
        # 保留完整原文；清洗只规范空白，不混入仅部分摘要。段落去重仅用于QA统计。
        paragraphs=[re.sub(r'\s+',' ',p).strip() for p in re.split(r'\n\s*\n',full) if len(p.strip())>40]
        features=text_features(full)
        assert features['effective_tokens']>1000,'extraction too short'
        row.update(features); row.update(status='usable',pages=len(pages),chars=len(full),paragraphs=len(paragraphs),
            repeated_paragraphs=len(paragraphs)-len(set(paragraphs)),sha256=hashlib.sha256(blob).hexdigest(),
            text_sha256=hashlib.sha256(full.encode()).hexdigest(),event_group='IBM_'+str(r.report_end.date())+'_'+r.form)
        TEXTS[r.accessionNumber]=full
    except Exception as e: row.update(status='rejected',reason=str(e))
    DOC_ROWS.append(row)
DOCS=pd.DataFrame(DOC_ROWS); EVENTS=DOCS[DOCS.status.eq('usable')].copy().sort_values('ready_at')
print(DOCS[['report_end','form','status']].to_string(index=False)); print(DOCS.get('reason',pd.Series(dtype=str)).value_counts().to_dict())
assert len(EVENTS)>=20,'Insufficient real documents: stop and inspect manifest'
assert EVENTS.accessionNumber.is_unique and EVENTS.event_group.is_unique
EVENTS['text']=EVENTS.accessionNumber.map(TEXTS)
print('真实全文质量报告'); display(DOCS[['form','report_end','status','pages','chars','effective_tokens','sha256','url']])
print('有效事件',len(EVENTS),'form counts',EVENTS.form.value_counts().to_dict(),'字节与文本均仅内存缓存')
print('规则：3词否定窗口；否定正面词转负面，否定负面词忽略；遇标点重置；这只是透明基线。')
pos=sorted(LEX['Positive'])[0]; neg=sorted(LEX['Negative'])[0]
assert text_features(pos)['sentiment']>text_features('NOT '+pos)['sentiment']
assert text_features(neg)['negative']>text_features('NOT '+neg)['negative']
print('04.02 PASS: actual PDF text and negation unit checks')


In [ ]:
# 04.03 公告事件时间轴、五日标签和事前市场模型
OPEN_TIMES=(OPEN.index+pd.Timedelta(hours=9,minutes=30)).tz_localize(NY)
CLOSE_TIMES=(CLOSE.index+pd.Timedelta(hours=16)).tz_localize(NY)
OO=OPEN[['IBM','SPY']].shift(-1)/OPEN[['IBM','SPY']]-1
END_TIMES=pd.Series(OPEN_TIMES,index=OPEN.index).shift(-1)
EVENT_ROWS=[]; AR_ROWS=[]
for _,r in EVENTS.iterrows():
    j=OPEN_TIMES.searchsorted(r.ready_at,side='left')
    if j<120 or j+5>=len(OPEN): continue
    entry=OPEN.index[j]; exit_session=OPEN.index[j+5]
    known=CLOSE_TIMES.searchsorted(r.ready_at,side='right')-1
    hist=CLOSE.iloc[:known+1]
    fit_cutoff=r.filingDate.tz_localize(NY)
    if pd.notna(r.acceptance_at): fit_cutoff=min(fit_cutoff,r.acceptance_at)
    eligible=OO.loc[(END_TIMES<=fit_cutoff).fillna(False)].dropna().tail(120)
    if len(eligible)<120: continue
    model=sm.OLS(eligible.IBM,sm.add_constant(eligible.SPY)).fit()
    future=OO.iloc[j:j+5]
    abnormal=future.IBM-(model.params['const']+model.params['SPY']*future.SPY)
    for k,ar in enumerate(abnormal): AR_ROWS.append({'accessionNumber':r.accessionNumber,'event_day':k+1,'AR':ar})
    rx=OPEN.IBM.iloc[j+5]/OPEN.IBM.iloc[j]-1; rm=OPEN.SPY.iloc[j+5]/OPEN.SPY.iloc[j]-1
    row=r.to_dict(); row.update(entry_session=entry,entry_at=OPEN_TIMES[j],label_end=OPEN_TIMES[j+5],
      future_excess=rx-rm,y=int(rx-rm>0),CAR5=abnormal.sum(),alpha_pre=model.params['const'],beta_pre=model.params['SPY'],
      fit_cutoff=fit_cutoff,fit_last_end=END_TIMES.loc[eligible.index[-1]],feature_last_close=CLOSE_TIMES[known],
      price_r5=hist.IBM.iloc[-1]/hist.IBM.iloc[-6]-1,price_r20=hist.IBM.iloc[-1]/hist.IBM.iloc[-21]-1,
      price_vol20=hist.IBM.pct_change(fill_method=None).tail(20).std(),market_r20=hist.SPY.iloc[-1]/hist.SPY.iloc[-21]-1)
    EVENT_ROWS.append(row)
E4=pd.DataFrame(EVENT_ROWS).sort_values('entry_at').reset_index(drop=True)
E4['fold']=np.select([E4.entry_session<TRAIN_END,E4.entry_session<VALID_END],['train','validation'],default='development_eval')
train_boundary=pd.Timestamp(TRAIN_END).tz_localize(NY); valid_boundary=pd.Timestamp(VALID_END).tz_localize(NY)
E4['purged']=((E4.fold=='train')&(E4.label_end>=train_boundary))|((E4.fold=='validation')&(E4.label_end>=valid_boundary))
E4=E4[~E4.purged].copy().reset_index(drop=True)
assert (E4.entry_at>=E4.ready_at).all() and (E4.feature_last_close<E4.entry_at).all()
assert (E4.fit_last_end<=E4.fit_cutoff).all()
assert E4.groupby('event_group').fold.nunique().max()==1
print('事件时间轴：精确申报时间保留；执行使用filed日末保守下界加一小时。')
display(E4[['accessionNumber','form','report_end','acceptance_raw','ready_at','entry_at','label_end','fit_cutoff','fit_last_end','fold','y','future_excess','CAR5']])
display(E4.groupby('fold').agg(n=('y','size'),positive=('y','sum'),mean_excess=('future_excess','mean')))
AR=pd.DataFrame(AR_ROWS).merge(E4[['accessionNumber']],on='accessionNumber',how='inner')
summary=AR.groupby('event_day').AR.agg(['mean','std','count']); summary['CAAR']=summary['mean'].cumsum()
fig,ax=plt.subplots(1,2,figsize=(12,4)); summary['CAAR'].plot(marker='o',ax=ax[0],title='Market-model mean cumulative abnormal return')
E4.plot.scatter(x='sentiment',y='future_excess',ax=ax[1],title='Dictionary sentiment vs 5-session excess'); plt.tight_layout(); plt.show()
print('04.03 PASS: observed events, 120 past market-model observations, 5 open-to-open steps')


In [ ]:
# 04.04 同窗价格、LM与TF-IDF Logistic增量对照
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import balanced_accuracy_score,roc_auc_score,brier_score_loss,log_loss,mean_squared_error
PRICE_FEATURES=['price_r5','price_r20','price_vol20','market_r20']
DICT_FEATURES=['sentiment','negative','uncertainty','negated_rate','log_length']
L4_EXPERIMENTS=pd.DataFrame([
 ['price_only',PRICE_FEATURES,False],['price+LM',PRICE_FEATURES+DICT_FEATURES,False],
 ['price+TFIDF',PRICE_FEATURES,True],['length_only',['log_length'],False]],columns=['model','numeric_features','TFIDF'])
def classifier(nums,text=False):
    num=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())])
    steps=[('num',num,nums)]
    if text: steps.append(('text',TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=.95,max_features=800,sublinear_tf=True),'text'))
    return Pipeline([('features',ColumnTransformer(steps)),('model',LogisticRegression(C=1,solver='liblinear',random_state=SEED,max_iter=2000))])
tr=E4[E4.fold.eq('train')]; va=E4[E4.fold.eq('validation')]; te=E4[E4.fold.eq('development_eval')]
assert len(tr)>=10 and tr.y.nunique()==2 and len(va)>=3 and len(te)>=3
L4_MODELS={}; L4_PRED=E4[['accessionNumber','event_group','entry_session','entry_at','label_end','fold','y','future_excess']].copy(); result=[]
for _,r in L4_EXPERIMENTS.iterrows():
    model=classifier(r.numeric_features,r.TFIDF); model.fit(tr,tr.y); L4_MODELS[r.model]=model
    L4_PRED[r.model]=model.predict_proba(E4)[:,1]
    for fold,data in [('validation',va),('development_eval',te)]:
        p=model.predict_proba(data)[:,1]
        result.append({'model':r.model,'fold':fold,'n':len(data),'balanced_accuracy':balanced_accuracy_score(data.y,p>=.5),
         'AUC':roc_auc_score(data.y,p) if data.y.nunique()==2 else np.nan,'Brier':brier_score_loss(data.y,p),'log_loss':log_loss(data.y,p,labels=[0,1])})
L4_METRICS=pd.DataFrame(result).set_index(['fold','model']); display(L4_EXPERIMENTS); display(L4_METRICS)
# 同一事件的配对误差差异；按年展示，避免把小样本差异写成统计显著结论。
paired=L4_PRED[L4_PRED.fold.ne('train')].copy(); paired['year']=paired.entry_session.dt.year
for model in ['price+LM','price+TFIDF']:
    paired['delta_Brier_'+model]=(paired[model]-paired.y)**2-(paired.price_only-paired.y)**2
DELTA=paired.groupby('year')[[c for c in paired if c.startswith('delta_Brier')]].mean(); display(DELTA)
print('delta<0代表该年Brier改善；小样本、镜像版本、词典重建及多次实验限制均保留。')
for model_name in ['price+LM','price+TFIDF']:
    base_metric=L4_METRICS.xs('price_only',level='model')['Brier']
    delta=L4_METRICS.xs(model_name,level='model')['Brier']-base_metric
    verdict='两个开发窗口Brier均降低，但小样本与版本局限下仍不足以证明稳定可交易增量' if (delta<0).all() else 'Brier改善没有在两个开发窗口一致出现，当前实验未支持稳定增量'
    print(model_name,'相对仅价格的Brier差',delta.to_dict(),'；结论：'+verdict)

# 字典密度只做有效词分母；TF-IDF不用全样本词频。运行验证集不会改变已fit词表。
v=L4_MODELS['price+TFIDF'].named_steps['features'].named_transformers_['text']; before_vocab=dict(v.vocabulary_)
v.transform(va.text); assert v.vocabulary_==before_vocab
fig,ax=plt.subplots(1,2,figsize=(12,4))
for model in ['price_only','price+LM','price+TFIDF']:
    ax[0].plot(paired.entry_session,paired[model],marker='o',label=model)
    bins=pd.cut(paired[model],[0,.25,.5,.75,1],include_lowest=True)
    cal=paired.groupby(bins,observed=True).agg(prob=(model,'mean'),frequency=('y','mean'),n=('y','size'))
    ax[1].plot(cal.prob,cal.frequency,marker='o',label=model)
ax[0].set_title('Chronological event predictions'); ax[0].legend(); ax[1].plot([0,1],[0,1],'k--'); ax[1].set_title('Small-sample calibration diagnostic'); ax[1].legend(); plt.tight_layout(); plt.show()
print('04.04 PASS: numeric and vocab fit only train; no event/random-row split')


In [ ]:
# 04.05 相同事件窗口的成本回测、模型卡与完整验收
L4_TARGETS={}; L4_LEDGERS={}; L4_TRADES={}
l4_dates=OPEN.loc['2019-01-01':'2022-12-29'].index
for name in L4_MODELS:
    w=pd.DataFrame(0.,index=l4_dates,columns=['IBM'])
    for _,r in L4_PRED[L4_PRED.fold.ne('train')].iterrows():
        j=OPEN.index.get_loc(r.entry_session); window=OPEN.index[j:j+5].intersection(l4_dates)
        w.loc[window,'IBM']=.8 if r[name]>=.5 else 0.
    L4_TARGETS[name]=w
    L4_LEDGERS[name],L4_TRADES[name]=l2_simulate_multi(OPEN[['IBM']],w,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
L4_ACCOUNT=pd.DataFrame({name:ledger_stats(x) for name,x in L4_LEDGERS.items()}).T; display(L4_ACCOUNT)
L4_CHECKS=[]
def check4(name,ok):
    assert bool(ok),name
    L4_CHECKS.append({'check':name,'status':'PASS'})
check4('actual docs with hash',EVENTS.sha256.str.len().eq(64).all() and len(EVENTS)>=20)
check4('unique accession + event',E4.accessionNumber.is_unique and E4.event_group.is_unique)
check4('document first-public conservative bound', (E4.ready_at>=E4.filed_end_at+pd.Timedelta(hours=1)).all())
check4('entry after readiness',(E4.entry_at>=E4.ready_at).all())
check4('market model uses past ended returns',(E4.fit_last_end<E4.ready_at).all())
check4('market model excludes announcement-day returns',(E4.fit_last_end<=E4.fit_cutoff).all())
check4('price feature availability',(E4.feature_last_close<E4.entry_at).all())
check4('5 trading-session exits',all(OPEN.index.get_loc(x.label_end.tz_localize(None).normalize())-OPEN.index.get_loc(x.entry_session)==5 for _,x in E4.iterrows()))
check4('event group folds',E4.groupby('event_group').fold.nunique().max()==1)
check4('train labels purged',tr.label_end.max()<train_boundary)
check4('validation labels purged',va.label_end.max()<valid_boundary)
check4('TFIDF train vocab frozen',v.vocabulary_==before_vocab)
for name,x in L4_LEDGERS.items():
    check4(name+' common window',x.index.equals(l4_dates))
    check4(name+' accounting',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
    check4(name+' gross long cap',x.gross_exposure.max()<=.800001 and (L4_TARGETS[name]>=0).all().all())
L4_VALIDATION=pd.DataFrame(L4_CHECKS); display(L4_VALIDATION)
L4_CARD={'course':4,'status':'verified','documents':len(EVENTS),'labeled_events':len(E4),
 'forms':EVENTS.form.value_counts().to_dict(),'dictionary_source':LM_SOURCE,'dictionary_sha256':hashlib.sha256(LM_BLOB).hexdigest(),
 'train_only_fit':['numeric median','standard scaler','TFIDF vocabulary/IDF','Logistic'],
 'checks':len(L4_CHECKS),'limitations':['single surviving issuer','small samples','official SEC archive 403; issuer main-filing mirrors','conservative filed-end timing; acceptance preserved','incorporated exhibits not collected','historical lexicon reconstructed from packaged snapshot, not exact 2011 release','no causal incremental-value claim','development data already observed']}
fig,ax=plt.subplots(figsize=(12,4)); pd.DataFrame({n:x.wealth for n,x in L4_LEDGERS.items()}).plot(ax=ax,title='Same-event 5-session signals with 5bp cost per traded leg'); plt.tight_layout(); plt.show()
print(json.dumps(L4_CARD,ensure_ascii=False,indent=2)); print('L4 FINAL PASS')


# 第5课 数据中心的监督学习与量化预测

交付：统一价格/动量/财务/文本特征表、数据质量表、版本化开盘标签、按整日分组且清除重叠的时间切分、仅fit训练集的Pipeline、零收益/Ridge/Logistic/LightGBM基线、固定模型的数据实验、固定数据的模型对照、逐日预测、Rank IC/校准诊断、成本台账、特征组消融和Data Card/Model Card。

核心标签：决策日t收盘已有特征，t+1开盘进入，t+6开盘退出，预测五交易日复权简单收益。label_end记录真实成熟开盘；十日标签是单独登记的数据实验。训练2014—2018、验证2019—2020、开发评估2021—2022；同日所有股票在同一集合，训练标签不能伸入验证，验证标签不能伸入评估。日标签重叠，清除边界重叠并用日期分组汇总，不能把股票行看成独立抽样。

本课使用动量/短期反转、质量、价值与第二课IWM/SPY残差市场状态四类信号；第四课IBM文本作为可用时间对齐的附加特征，其余股票明确缺失，绝不伪造全文覆盖。固定样本的中性化本身使用当天可见横截面；全时序填补、缩放、分位截尾参数都仅从训练学得。

数据实验保持Ridge与评估集不变：注入重复/异常值演示、去重、稳健截尾、完整行丢弃 vs 缺失标记、按日等权、训练内2年半衰期的时间衰减权重、五日 vs 十日标签。注入错误必须标记为人为演示，不声称发现了真实数据错误；改变标签的模型仍在同一个五日评估目标上比较，避免拿不同目标的误差排赢家。

模型实验保持真实干净数据不变：zero、Ridge、Logistic概率、LightGBM。预先登记固定参数，验证窗口用于报告、不用最后一段调参。所有交易模型在共同日期、每5个交易日决策、总多头目标0.8、每腿佣金3bp＋滑点2bp下运行；月内/持仓段每日维护目标权重的费用计入账本。分类概率只用于排序，不冒充收益预测。

数据与模型卡明确存续偏差、滞后股数、镜像版本、词典快照、文本单公司和开发期复用等限制；当前结果不构成独立留出集证明。

参考：[sklearn时间验证](https://scikit-learn.org/stable/modules/cross_validation.html)、[Pipeline防泄漏](https://scikit-learn.org/stable/common_pitfalls.html)、[LightGBM API](https://lightgbm.readthedocs.io/en/stable/Python-API.html)。


In [ ]:
# 05.01 统一特征表、标签版本与数据质量
from sklearn.base import BaseEstimator,TransformerMixin
from lightgbm import LGBMRegressor
F5=PANEL.copy()
for name,mat in {'ret5':CLOSE[ASSETS].pct_change(5,fill_method=None),'ret20':CLOSE[ASSETS].pct_change(20,fill_method=None),
                 'vol20':CLOSE[ASSETS].pct_change(fill_method=None).rolling(20).std()}.items():
    tmp=mat.stack().rename(name).rename_axis(['date','ticker']).reset_index(); F5=F5.merge(tmp,on=['date','ticker'],how='left')
# 第2课IWM/SPY协整残差z作为共同市场状态输入，不冒充每只个股的配对交易信号。
F5['market_pair_z']=F5.date.map(l2_features.z)
F5['pair_fit_last_session']=F5.date.map(l2_features.fit_last_session)
# 文本只有IBM具备来源；按实际ready_at连接，不能按report_end连接。
text_cols=['sentiment','uncertainty','negative','log_length']
F5['text_available_at']=pd.NaT
for col in text_cols: F5['text_'+col]=np.nan
ibm=F5[F5.ticker.eq('IBM')].copy(); ev=E4[['ready_at']+text_cols].sort_values('ready_at').drop_duplicates('ready_at')
joined=pd.merge_asof(ibm[['date','decision_at']].sort_values('decision_at'),ev,left_on='decision_at',right_on='ready_at',direction='backward')
for col in text_cols:
    mapping=pd.Series(joined[col].values,index=joined.date); F5.loc[F5.ticker.eq('IBM'),'text_'+col]=F5.loc[F5.ticker.eq('IBM'),'date'].map(mapping)
text_time=pd.Series(joined.ready_at.values,index=joined.date)
# 统一UTC存储时间字段，避免datetime tz dtype隐式转换。
F5['text_available_at']=pd.to_datetime(F5.date.map(text_time).where(F5.ticker.eq('IBM')),utc=True)
for h in [5,10]:
    labels=forward_open(h).stack().rename('y'+str(h)).rename_axis(['date','ticker']).reset_index()
    F5=F5.merge(labels,on=['date','ticker'],how='left')
    F5['end'+str(h)]=F5.date.map(pd.Series(CLOSE.index,index=CLOSE.index).shift(-(h+1)))
    F5['label_end'+str(h)]=(F5['end'+str(h)]+pd.Timedelta(hours=9,minutes=30)).dt.tz_localize(NY)
F5['execution_session']=F5.date.map(pd.Series(CLOSE.index,index=CLOSE.index).shift(-1))
F5['execution_at']=(F5.execution_session+pd.Timedelta(hours=9,minutes=30)).dt.tz_localize(NY)
GROUPS={'price':['ret5','ret20','vol20','momentum_z'],'stat_arb':['market_pair_z'],'fundamental':['quality_z','value_z','size'],
        'text':['text_'+c for c in text_cols]}
FEATURES=sum(GROUPS.values(),[])
F5[FEATURES]=F5[FEATURES].replace([np.inf,-np.inf],np.nan)
F5=F5.dropna(subset=['y5','y10','execution_session']).sort_values(['date','ticker']).reset_index(drop=True)
LABEL_VERSIONS=pd.DataFrame([{'id':'open_h'+str(h)+'_v1','decision':'t close','entry':'t+1 open','exit':'t+'+str(h+1)+' open','horizon':h,'basis':'adjusted simple return','maturity':'label_end'+str(h)} for h in [5,10]])
QUALITY=pd.DataFrame({'missing_rate':F5[FEATURES].isna().mean(),'nonfinite_after_cleanup':(~np.isfinite(F5[FEATURES].fillna(0))).sum(),
 'nunique':F5[FEATURES].nunique(),'source':{c:'price' if c in GROUPS['price'] else 'lesson2 causal residual' if c in GROUPS['stat_arb'] else 'SEC PIT' if c in GROUPS['fundamental'] else 'IBM real filing; others missing' for c in FEATURES}})
FEATURE_REGISTRY=pd.DataFrame([{'feature':f,'family':family,'available_rule':'decision close; past-only source',
 'definition':{'ret5':'5-session past return','ret20':'20-session past return','vol20':'20-session past daily volatility',
 'momentum_z':'12-1 momentum, current cross-sectional neutral residual','market_pair_z':'lesson2 IWM/SPY residual z; prior252-session fit; shared regime',
 'quality_z':'SEC annual ROA neutral residual; filed day-end PIT','value_z':'SEC B/M estimate neutral residual; filed day-end PIT','size':'log lagged-share marketcap estimate'}.get(f,'IBM filing LM feature; latest ready_at<=decision_at; missing other issuers')}
 for family,cols in GROUPS.items() for f in cols])
display(FEATURE_REGISTRY)
print('真实数据质量：duplicates',F5.duplicated(['date','ticker']).sum(),'rows',len(F5),'dates',F5.date.nunique()); display(QUALITY); display(LABEL_VERSIONS)
# 日切分，按10日版本最迟成熟时间清除边界，保证全部标签实验共用同一行集合。
F5['fold']=np.select([F5.date<TRAIN_END,F5.date<VALID_END],['train','validation'],default='development_eval')
F5['purged']=((F5.fold=='train')&(F5.label_end10>=train_boundary))|((F5.fold=='validation')&(F5.label_end10>=valid_boundary))
PURGE_TABLE=F5.groupby('fold').agg(rows=('date','size'),purged=('purged','sum'))
D5=F5[~F5.purged].copy(); TR5=D5[D5.fold.eq('train')]; VA5=D5[D5.fold.eq('validation')]; TE5=D5[D5.fold.eq('development_eval')]
display(PURGE_TABLE); print('05.01 PASS: date-group folds, conservative maturity purge, asof text, versioned labels')


In [ ]:
# 05.02 固定Ridge的数据实验与训练期预处理
class TrainClip(BaseEstimator,TransformerMixin):
    def __init__(self,low=.01,high=.99): self.low=low; self.high=high
    def fit(self,X,y=None):
        arr=np.asarray(X,float); self.lo_=np.nanquantile(arr,self.low,axis=0); self.hi_=np.nanquantile(arr,self.high,axis=0); return self
    def transform(self,X): return np.clip(np.asarray(X,float),self.lo_,self.hi_)
def regression_pipe(cols,model=None,clip=False,indicator=True):
    numeric=[]
    if clip: numeric.append(('clip',TrainClip()))
    numeric.extend([('impute',SimpleImputer(strategy='median',add_indicator=indicator,keep_empty_features=True)),('scale',StandardScaler())])
    return Pipeline([('features',ColumnTransformer([('numeric',Pipeline(numeric),cols)])),('model',Ridge(alpha=10) if model is None else model)])
def prediction_stats(frame,pred,target='y5'):
    z=frame[['date','ticker',target]].copy(); z['prediction']=pred
    ics=[g.prediction.corr(g[target],method='spearman') for _,g in z.groupby('date') if len(g)>=3 and g.prediction.nunique()>1]
    return {'MSE':mean_squared_error(frame[target],pred),'mean_RankIC':np.nanmean(ics) if ics else np.nan,
            'direction_accuracy':np.mean((np.asarray(pred)>0)==(frame[target].to_numpy()>0)),'n':len(frame)}
DUPLICATED=pd.concat([TR5,TR5.iloc[:500]],ignore_index=True) # labeled injected corruption
DEDUP=DUPLICATED.drop_duplicates(['date','ticker'],keep='first')
OUTLIER=TR5.copy(); inject_idx=OUTLIER.index[:100]; OUTLIER.loc[inject_idx,'ret5']=OUTLIER.loc[inject_idx,'ret5']*100
EXPERIMENTS5=[
 {'id':'clean_missing_indicator','train':TR5,'clip':False,'indicator':True,'label':'y5','injected':False},
 {'id':'injected_duplicates','train':DUPLICATED,'clip':False,'indicator':True,'label':'y5','injected':True},
 {'id':'deduplicated','train':DEDUP,'clip':False,'indicator':True,'label':'y5','injected':False},
 {'id':'injected_outliers','train':OUTLIER,'clip':False,'indicator':True,'label':'y5','injected':True},
 {'id':'outliers_train_clip','train':OUTLIER,'clip':True,'indicator':True,'label':'y5','injected':True},
 {'id':'no_missing_indicator','train':TR5,'clip':False,'indicator':False,'label':'y5','injected':False},
 {'id':'complete_case_train','train':TR5.dropna(subset=FEATURES),'clip':False,'indicator':True,'label':'y5','injected':False},
 {'id':'date_balanced_weights','train':TR5,'clip':False,'indicator':True,'label':'y5','injected':False,'weighted':True},
 {'id':'recency_weights','train':TR5,'clip':False,'indicator':True,'label':'y5','injected':False,'weighted':'recency'},
 {'id':'label_h10_to_h5','train':TR5,'clip':False,'indicator':True,'label':'y10','injected':False}]
DATA_RESULTS=[]; DATA_MODELS={}; DATA_PRED={}
for r in EXPERIMENTS5:
    data=r['train']; model=regression_pipe(FEATURES,clip=r['clip'],indicator=r['indicator'])
    kwargs={}
    if r.get('weighted'):
        sw=np.exp(-np.log(2)*(data.date.max()-data.date).dt.days/(2*365.25)) if r.get('weighted')=='recency' else 1/data.date.map(data.groupby('date').size())
        sw=sw/sw.mean(); kwargs={'model__sample_weight':sw}
    model.fit(data,data[r['label']],**kwargs); DATA_MODELS[r['id']]=model
    for fold,frame in [('validation',VA5),('development_eval',TE5)]:
        pred=model.predict(frame)
        if r['label']=='y10': pred=np.sqrt(np.maximum(1+pred,1e-6))-1 # declared constant growth hypothesis for horizon conversion
        DATA_PRED[(r['id'],fold)]=pred
        DATA_RESULTS.append({'experiment':r['id'],'fold':fold,'train_n':len(data),'injected':r['injected'],'train_label':r['label'],**prediction_stats(frame,pred)})
L5_DATA_RESULTS=pd.DataFrame(DATA_RESULTS).set_index(['fold','experiment']); display(L5_DATA_RESULTS)
print('人为异常/重复注入仅在训练副本；评估数据固定。完整行删除主要留下IBM，会改变训练群体，未把它当作推荐清洗。')
print('完整平衡面板的按日等权与逐行等权相同，保留这个无变化结果；另登记训练内2年半衰期的时间衰减权重。')
print('h10转换到h5是假设恒定增长，不保证准确；全部实验保留，无成功筛选。')
print('05.02 PASS: fixed Ridge(alpha=10), date-balanced weights, missing flags, frozen train clipping, registered label experiment')


In [ ]:
# 05.03 固定真实数据的模型对照、逐期预测与特征组消融
from sklearn.dummy import DummyRegressor
LGB_PARAMS={'n_estimators':120,'learning_rate':.05,'num_leaves':15,'max_depth':4,'min_child_samples':100,
            'reg_lambda':10,'random_state':SEED,'n_jobs':2,'verbosity':-1,'deterministic':True,'force_col_wise':True}
MODEL_REGISTRY=pd.DataFrame([
 {'id':'zero','kind':'regression','parameter':'constant=0','training_target':'y5'},
 {'id':'Ridge','kind':'regression','parameter':'alpha=10','training_target':'y5'},
 {'id':'LightGBM','kind':'regression','parameter':json.dumps(LGB_PARAMS),'training_target':'y5'},
 {'id':'Logistic','kind':'classification/ranking','parameter':'C=1; threshold=.5','training_target':'y5>0'}])
L5_MODELS={
 'zero':regression_pipe(FEATURES,DummyRegressor(strategy='constant',constant=0)),
 'Ridge':regression_pipe(FEATURES,Ridge(alpha=10)),
 'LightGBM':regression_pipe(FEATURES,LGBMRegressor(**LGB_PARAMS)),
 'Logistic':regression_pipe(FEATURES,LogisticRegression(C=1,max_iter=2000,solver='liblinear',random_state=SEED))}
L5_PRED=D5[['date','ticker','decision_at','execution_session','execution_at','label_end5','fold','y5']].copy(); MODEL_RESULTS=[]
for name,model in L5_MODELS.items():
    target=(TR5.y5>0).astype(int) if name=='Logistic' else TR5.y5
    model.fit(TR5,target)
    if name=='Logistic': L5_PRED[name]=model.predict_proba(D5)[:,1]
    else: L5_PRED[name]=model.predict(D5)
    for fold,frame in [('validation',VA5),('development_eval',TE5)]:
        if name=='Logistic':
            p=model.predict_proba(frame)[:,1]; y=(frame.y5>0).astype(int)
            ranked=prediction_stats(frame,p); ranked['MSE']=np.nan; ranked['direction_accuracy']=np.nan
            MODEL_RESULTS.append({'model':name,'fold':fold,**ranked,'Brier':brier_score_loss(y,p),'AUC':roc_auc_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,p>=.5)})
        else: MODEL_RESULTS.append({'model':name,'fold':fold,**prediction_stats(frame,model.predict(frame))})
L5_MODEL_RESULTS=pd.DataFrame(MODEL_RESULTS).set_index(['fold','model']); display(MODEL_REGISTRY); display(L5_MODEL_RESULTS)
L5_ABLATION_MODELS={}; ABLATION=[]
for name,cols in {'price_only':GROUPS['price'],'no_text':GROUPS['price']+GROUPS['stat_arb']+GROUPS['fundamental'],
                  'no_stat_arb':GROUPS['price']+GROUPS['fundamental']+GROUPS['text'],
                  'no_fundamental':GROUPS['price']+GROUPS['stat_arb']+GROUPS['text'],'fundamental_only':GROUPS['fundamental'],'all':FEATURES}.items():
    model=regression_pipe(cols,Ridge(alpha=10)); model.fit(TR5,TR5.y5); L5_ABLATION_MODELS[name]=model
    for fold,frame in [('validation',VA5),('development_eval',TE5)]: ABLATION.append({'group':name,'fold':fold,**prediction_stats(frame,model.predict(frame))})
L5_ABLATIONS=pd.DataFrame(ABLATION).set_index(['fold','group']); display(L5_ABLATIONS)
print('所有模型与消融均fit相同训练期；逐日预测保留date/ticker/available/entry/maturity，不在评估期重新选参数。')
display(L5_PRED[L5_PRED.fold.eq('development_eval')].head(24))
features_out=L5_MODELS['LightGBM'].named_steps['features'].get_feature_names_out()
IMPORTANCE=pd.Series(L5_MODELS['LightGBM'].named_steps['model'].feature_importances_,index=features_out).sort_values(ascending=False)
display(IMPORTANCE.rename('training_split_importance').to_frame())
fig,ax=plt.subplots(1,2,figsize=(13,4)); dev=L5_PRED[L5_PRED.fold.eq('development_eval')].copy()
for name in ['Ridge','LightGBM','Logistic']:
    daily=dev.groupby('date').apply(lambda g:g[name].corr(g.y5,method='spearman'),include_groups=False)
    daily.rolling(63,min_periods=20).mean().plot(ax=ax[0],label=name)
ax[0].legend(); ax[0].set_title('63-session mean cross-sectional Rank IC')
bins=pd.cut(dev.Logistic,[0,.25,.5,.75,1],include_lowest=True); dev['direction']=(dev.y5>0).astype(int)
cal=dev.groupby(bins,observed=True).agg(prob=('Logistic','mean'),frequency=('direction','mean'),n=('direction','size')); display(cal)
ax[1].plot(cal.prob,cal.frequency,'o-'); ax[1].plot([0,1],[0,1],'k--'); ax[1].set_title('Classification calibration (descriptive, correlated labels)'); plt.tight_layout(); plt.show()
print('05.03 PASS: fixed-data baselines, no random panel split, all experiments preserved')


In [ ]:
# 05.04 统一执行接口、五日决策与成本/回撤比较
DECISION5=pd.DatetimeIndex(sorted(TE5.date.unique()))[::5]
l5_dates=OPEN.loc['2021-01-05':'2022-12-29'].index
L5_SIGNALS=L5_PRED[L5_PRED.date.isin(DECISION5)&L5_PRED.fold.eq('development_eval')].copy()
L5_SIGNALS['model_id']='frozen_2018_baselines'; L5_SIGNALS['horizon']='5 sessions'
assert (L5_SIGNALS.decision_at<L5_SIGNALS.execution_at).all()
PREDICTORS={name:(lambda frame,n=name: L5_MODELS[n].predict_proba(frame)[:,1] if n=='Logistic' else L5_MODELS[n].predict(frame)) for name in L5_MODELS}
for name,m in L5_ABLATION_MODELS.items(): PREDICTORS['ablation_'+name]=lambda frame,m=m:m.predict(frame)
for r in EXPERIMENTS5:
    m=DATA_MODELS[r['id']]; h10=r['label']=='y10'
    PREDICTORS['data_'+r['id']]=lambda frame,m=m,h10=h10:np.sqrt(np.maximum(1+m.predict(frame),1e-6))-1 if h10 else m.predict(frame)
L5_TARGETS={}; L5_LEDGERS={}; L5_TRADES={}
base_rows=TE5[TE5.date.isin(DECISION5)].copy()
for name,predict in PREDICTORS.items():
    scored=base_rows[['date','ticker','execution_session']].copy(); scored['score']=predict(base_rows)
    target=pd.DataFrame(0.,index=l5_dates,columns=ASSETS)
    for date,g in scored.groupby('date'):
        threshold=.5 if name=='Logistic' else 0.
        chosen=g[g.score>threshold].sort_values(['score','ticker']).tail(3)
        if len(chosen)==0: continue
        entry=g.execution_session.iloc[0]; j=OPEN.index.get_loc(entry); window=OPEN.index[j:j+5].intersection(l5_dates)
        target.loc[window,chosen.ticker]=.8/len(chosen)
    L5_TARGETS[name]=target
    L5_LEDGERS[name],L5_TRADES[name]=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
L5_BACKTEST=pd.DataFrame({n:ledger_stats(x) for n,x in L5_LEDGERS.items()}).T
L5_BACKTEST['mean_gross']=[L5_LEDGERS[n].gross_exposure.mean() for n in L5_BACKTEST.index]
SIGNAL_INTERFACE=L5_SIGNALS.melt(id_vars=['date','ticker','decision_at','execution_at','label_end5','execution_session'],
    value_vars=list(L5_MODELS),var_name='model_id',value_name='signal').rename(columns={'ticker':'instrument','decision_at':'available_at'})
SIGNAL_INTERFACE['horizon']='5 trading sessions'
SIGNAL_INTERFACE['signal_type']=np.where(SIGNAL_INTERFACE.model_id.eq('Logistic'),'direction probability','return forecast')
display(SIGNAL_INTERFACE.head(12))
display(L5_SIGNALS.head(18)); display(L5_BACKTEST)
fig,ax=plt.subplots(2,2,figsize=(14,8))
pd.DataFrame({n:L5_LEDGERS[n].wealth for n in L5_MODELS}).plot(ax=ax[0,0],title='Frozen models, net wealth')
pd.DataFrame({n:L5_LEDGERS[n].drawdown for n in L5_MODELS}).plot(ax=ax[0,1],title='Shared-ledger drawdown')
pd.DataFrame({n:x.wealth for n,x in L5_LEDGERS.items() if n.startswith('data_')}).plot(ax=ax[1,0],title='Fixed Ridge data experiments, same 5-session evaluation')
pd.DataFrame({n:x.wealth for n,x in L5_LEDGERS.items() if n.startswith('ablation_')}).plot(ax=ax[1,1],title='Feature-group ablation'); plt.tight_layout(); plt.show()
print('05.04 PASS: all accounts use identical dates, 0.8 gross cap, 3bp commission+2bp slippage, no short borrowing')
print('zero预测不大于0，因此持现金；分类概率仅排序。总收益差异可能来自敞口/换手变化，不等于预测能力优势。')


In [ ]:
# 05.05 数据卡、模型卡、标签/防泄漏/账本验收与课程覆盖
L5_CHECKS=[]
def check5(name,ok):
    assert bool(ok),name
    L5_CHECKS.append({'check':name,'status':'PASS'})
check5('date-ticker keys unique',not D5.duplicated(['date','ticker']).any())
check5('entire date belongs to one fold',D5.groupby('date').fold.nunique().max()==1)
check5('train maximum maturity before validation',TR5.label_end10.max()<train_boundary)
check5('validation maximum maturity before eval',VA5.label_end10.max()<valid_boundary)
check5('lesson2 pair fit strictly past',(D5.pair_fit_last_session<D5.date).all())
check5('registered features equal pipeline inputs',set(FEATURE_REGISTRY.feature)==set(FEATURES))
check5('common signal interface causal',(SIGNAL_INTERFACE.available_at<SIGNAL_INTERFACE.execution_at).all())
check5('strict decision before entry',(D5.decision_at<D5.execution_at).all())
check5('text available before decision',(D5.loc[D5.text_available_at.notna(),'text_available_at']<=D5.loc[D5.text_available_at.notna(),'decision_at'].dt.tz_convert('UTC')).all())
check5('other issuers text correctly missing',D5.loc[~D5.ticker.eq('IBM'),GROUPS['text']].isna().all().all())
check5('all train labels and input prices finite',np.isfinite(TR5[['y5','y10']]).all().all())
check5('injected duplicates explicitly exceed real rows',len(DUPLICATED)>len(TR5) and not TR5.duplicated(['date','ticker']).any())
check5('dedup preserves clean training keys',len(DEDUP)==len(TR5))
# 多个时点手算验证标签，不只镜像shift表达式。
for position in [0,len(TE5)//2,len(TE5)-1]:
    r=TE5.iloc[position]; j=OPEN.index.get_loc(r.date)
    expected5=OPEN.loc[OPEN.index[j+6],r.ticker]/OPEN.loc[OPEN.index[j+1],r.ticker]-1
    expected10=OPEN.loc[OPEN.index[j+11],r.ticker]/OPEN.loc[OPEN.index[j+1],r.ticker]-1
    check5('manual y5 at '+str(position),np.isclose(r.y5,expected5))
    check5('manual y10 at '+str(position),np.isclose(r.y10,expected10))
    check5('maturity timestamp at '+str(position),r.label_end5==OPEN_TIMES[j+6])
# 核验实际已fit填补统计来自训练，未来转换不能改变它。
num=L5_MODELS['Ridge'].named_steps['features'].named_transformers_['numeric']; imp=num.named_steps['impute']
train_median=np.nanmedian(TR5[FEATURES].to_numpy(float),axis=0)
check5('imputer median equals train only',np.allclose(imp.statistics_,train_median,equal_nan=True))
stat_before=imp.statistics_.copy(); future_probe=TE5.copy(); future_probe[FEATURES]=future_probe[FEATURES]*100+100
L5_MODELS['Ridge'].predict(future_probe)
check5('future transformation does not refit',np.array_equal(stat_before,imp.statistics_,equal_nan=True))
clipper=DATA_MODELS['outliers_train_clip'].named_steps['features'].named_transformers_['numeric'].named_steps['clip']
check5('clip bounds fit actual train copy',np.allclose(clipper.lo_,np.nanquantile(OUTLIER[FEATURES],.01,axis=0),equal_nan=True))
# 历史前缀重算价格因子；修改未来价格不会影响过去动量/短期收益定义。
r=TE5.iloc[0]; history=CLOSE.loc[:r.date,r.ticker]
check5('past-only ret5 prefix',np.isclose(r.ret5,history.iloc[-1]/history.iloc[-6]-1))
for name,x in L5_LEDGERS.items():
    check5(name+' common execution window',x.index.equals(l5_dates))
    check5(name+' conserved PnL',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost-x.borrow_cost-x.funding_cost))
    check5(name+' cost nonnegative',(x.transaction_cost>=0).all())
    check5(name+' long gross constraint',(L5_TARGETS[name]>=0).all().all() and x.gross_exposure.max()<=.800001)
    check5(name+' compound wealth',np.allclose(x.wealth,(1+x.net_return).cumprod()))
check5('zero baseline remains cash',np.allclose(L5_LEDGERS['zero'].wealth,1))
L5_VALIDATION=pd.DataFrame(L5_CHECKS); display(L5_VALIDATION)
DATA_CARD={'price':PRICE_AUDIT,'universe':ASSETS,'date_rows':len(D5),'unique_dates':D5.date.nunique(),
 'target':'5-session adjusted open-to-open simple return; maturity stored','label_versions':LABEL_VERSIONS.to_dict('records'),
 'filings':{'issuer':'IBM','events':len(E4),'source':'issuer directory linked main filings','dictionary':LM_SOURCE},
 'split':{'train_end_exclusive':str(TRAIN_END.date()),'validation_end_exclusive':str(VALID_END.date()),'purge_uses':'10-session maturity'},
 'limitations':['survivorship','coarse industry','lagged marketcap shares','missing other-issuer text','revised vendor data and SEC contexts','issuer mirror vintage not independently archived','overlapping daily targets; date summaries correlated','development windows already observed']}
MODEL_CARD={'models':MODEL_REGISTRY.to_dict('records'),'frozen_fit_last_decision':str(TR5.date.max()),
 'fit_last_maturity':str(TR5.label_end10.max()),'all_preprocessing':'fit on training only',
 'trade_rule':'top3 positive regression prediction / Logistic p>.5; 0.8 long target; decision every5 sessions; daily weight maintenance',
 'cost':'3bp commission +2bp slippage each leg; adjusted synthetic units; no taxes/market impact',
 'data_experiments':[{k:v for k,v in r.items() if k!='train'} for r in EXPERIMENTS5],
 'not_proven':'independent unseen performance or causal text advantage','checks':len(L5_CHECKS)}
print(json.dumps({'DataCard':DATA_CARD,'ModelCard':MODEL_CARD},ensure_ascii=False,indent=2,default=str))
COVERAGE=pd.DataFrame([
 [3,'FF3/FF5+momentum; reference and same-cost account attribution','03.04/03.06'],[3,'actual momentum/quality/value, filing PIT and neutralization','03.02–03.03'],
 [3,'Rank IC, grouping, equal/trained score, turnover/decay/exposure','03.05–03.06'],
 [4,'real 10-K/10-Q, accession/version/timestamps, full main-PDF text QA','04.01–04.02'],
 [4,'LM sentiment/uncertainty/negation and past market-model events','04.02–04.03'],
 [4,'price/price+dictionary/price+TFIDF train-only comparisons','04.04–04.05'],
 [5,'feature table, quality, h5/h10 labels and maturity purge','05.01'],
 [5,'fixed-model data experiments and fixed-data baselines','05.02–05.03'],
 [5,'dated predictions, calibration/rank, costs/drawdown/ablations/cards','05.03–05.05']],columns=['course','requirement','verified_code'])
display(COVERAGE)
check5('previous lessons validations present',len(L3_VALIDATION)>0 and len(L4_VALIDATION)>0 and len(L2_VALIDATION)>0)
L5_CARD={'course':5,'status':'verified','checks':len(L5_CHECKS),'models':list(L5_MODELS),'accounts':len(L5_LEDGERS)}
print('L5 FINAL PASS',json.dumps(L5_CARD,ensure_ascii=False))
completion=pd.DataFrame({'课程':['第1课','第2课','第3课','第4课','第5课'],
 '验收':['原有代码完成','PASS '+str(len(L2_VALIDATION))+'项','PASS '+str(len(L3_VALIDATION))+'项','PASS '+str(len(L4_VALIDATION))+'项','PASS '+str(len(L5_CHECKS))+'项'],
 '产物':['趋势研究','均值回归/两腿账本','真实财务因子/FF归因','真实申报NLP/事件研究','数据实验/模型对照/统一成本']})
display(Markdown('### 第三、四、五课完整运行验收')); display(completion)
print('LESSONS 03–05 ALL VERIFIED |',pd.Timestamp.now(tz='UTC').isoformat())
# 所有数据、模型、清单与日志留在云端内存和本ipynb输出，无本地导出文件。


# 第6课 组合配置与市场状态风险

本课把第2—5课信号转为可执行权重，比较等权、逆波动、收缩协方差最小方差、状态过滤及风险匹配对照。固定风险资产预算0.8，余款现金；设单股、行业、总敞口、净敞口与换手限制，并输出不可行回退原因。

风险预测只用决策收盘以前的数据：历史、EWMA、滚动GARCH(1,1)，下一开盘生效。GARCH为收盘收益风险代理，不能冒充开盘交易收益的精确条件分布。VaR/ES用账本同口径历史净收益生成下一期阈值，报告突破、聚集与有限样本尾部均值。多空单独展示净/总/借券约束，不混用只做多优化条件。

所有组合共用第二课自融资账本：扣费后目标仓位、逐腿交易费、现金、借券与融资各记一次。状态过滤另与训练期冻结的等波动缩放基线比较；不会只凭更低回撤就断言更有效。

运行环境保留TPU：Pandas/SciPy/arch和SB3使用主机CPU；第8课JAX PPO梯度、第9课蒙特卡洛实际放在TPU。设备验证失败就明确停止加速实验，不把设置名称当成运行证明。所有数据窗2014—2022都是开发研究，不宣称未观察的最终留出。

参考：[arch预测](https://arch.readthedocs.io/en/stable/univariate/forecasting.html)、[JAX设备](https://docs.jax.dev/en/latest/installation.html)。


In [ ]:
# 06.01 扩展依赖、TPU实际计算与重现日志
import sys,subprocess,importlib.metadata as im,io,json,time,hashlib,warnings
subprocess.check_call([sys.executable,'-m','pip','install','-q','arch==7.2.0','gymnasium==1.2.2','stable-baselines3==2.7.0','QuantLib==1.40','nbformat==5.10.4'])
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display,Markdown
import jax,jax.numpy as jnp
DEVICES=jax.devices(); TPU_DEVICES=[d for d in DEVICES if d.platform=='tpu']
print('JAX',jax.__version__,'actual devices',DEVICES)
assert TPU_DEVICES,'User requested TPU: select TPU runtime and reconnect; do not label CPU results as TPU'
TPU=TPU_DEVICES[0]
with jax.default_device(TPU):
    a=jnp.arange(1024,dtype=jnp.float32).reshape(32,32); device_checksum=float(jax.jit(lambda x:jnp.sum(x@x.T))(a).block_until_ready())
print('TPU real matrix computation checksum',device_checksum)
L6_RUNTIME={'started_utc':pd.Timestamp.now(tz='UTC').isoformat(),'python':sys.version,'jax':jax.__version__,
 'devices':[str(d) for d in DEVICES],'TPU_checksum':device_checksum,
 'versions':{p:im.version(p) for p in ['numpy','pandas','scipy','arch','gymnasium','stable-baselines3','QuantLib']}}
print(json.dumps(L6_RUNTIME,ensure_ascii=False,indent=2))
print('Prerequisite variables present:',{k:k in globals() for k in ['OPEN','CLOSE','D5','L5_MODELS','l2_simulate_multi']})


In [ ]:
# 06.02 因果风险特征、收缩协方差与约束权重生成器
assert all(k in globals() for k in ['OPEN','CLOSE','D5','L5_MODELS','l2_simulate_multi']),'请先运行前五课'
from scipy.optimize import minimize
from scipy.stats import norm,chi2
from sklearn.covariance import LedoitWolf
from arch import arch_model
R6=CLOSE[ASSETS].pct_change(fill_method=None)
EXEC6=OPEN.loc['2014-02-01':'2022-12-29'].index
PREV6=pd.Series(OPEN.index,index=OPEN.index).shift(1).reindex(EXEC6)
GROUP6={g:[ASSETS.index(a) for a in ASSETS if UNIVERSE[a][1]==g] for g in sorted({UNIVERSE[a][1] for a in ASSETS})}
CFG6={'risk_budget':.8,'single_cap':.15,'industry_cap':.35,'gross_cap':.8,'net_cap':.8,'turnover_cap':.25,
 'cov_window':252,'cov_refit':21,'commission_bps':3,'slippage_bps':2,'state_train_end':'2019-01-01','EWMA_lambda':.94}
FALLBACK6=[]
def caps_ok(w,budget=.8,tol=1e-7):
    return abs(np.sum(w)-budget)<tol and np.min(w)>=-tol and np.max(w)<=.15+tol and all(np.sum(w[j])<=.35+tol for j in GROUP6.values())
def feasible_budget(v,budget=.8):
    if budget==0:return np.zeros(len(ASSETS))
    v=np.asarray(v,float); v=np.nan_to_num(v,nan=0,posinf=0,neginf=0)
    objective=lambda w:np.sum((w-v)**2)
    con=[{'type':'eq','fun':lambda w:w.sum()-budget}]+[{'type':'ineq','fun':lambda w,j=j:.35-w[j].sum()} for j in GROUP6.values()]
    res=minimize(objective,np.ones(len(v))*budget/len(v),bounds=[(0,.15)]*len(v),constraints=con,method='SLSQP',options={'ftol':1e-12,'maxiter':150})
    if not res.success or not caps_ok(res.x,budget):
        FALLBACK6.append({'where':'projection','reason':res.message,'budget':budget}); fallback=np.ones(len(v))*budget/len(v)
        assert caps_ok(fallback,budget); return fallback
    return res.x
SIG6=pd.DataFrame(index=CLOSE.index)
SIG6['hist_vol']=CLOSE.SPY.pct_change(fill_method=None).rolling(63).std()*np.sqrt(252)
SIG6['ewma_vol']=np.sqrt(CLOSE.SPY.pct_change(fill_method=None).pow(2).ewm(alpha=1-.94,adjust=False,min_periods=63).mean()*252)
SIG6['trend']=CLOSE.SPY/CLOSE.SPY.rolling(200).mean()-1
state_threshold=SIG6.loc['2014-01-01':'2018-12-31','ewma_vol'].quantile(.8)
SIG6['state_active']=(SIG6.ewma_vol<=state_threshold)&(SIG6.trend>0)
SIG6['GARCH_vol']=np.nan; garch_log=[]
for i in range(756,len(CLOSE),21):
    past=100*CLOSE.SPY.pct_change(fill_method=None).iloc[i-756:i+1].dropna()
    try:
        fitted=arch_model(past,mean='Zero',vol='GARCH',p=1,q=1,rescale=False).fit(disp='off',show_warning=False)
        omega,alpha,beta=[float(fitted.params[k]) for k in ['omega','alpha[1]','beta[1]']]
        variance=float(fitted.conditional_volatility.iloc[-1]**2)
        for j in range(i,min(i+21,len(CLOSE))):
            forecast=omega+alpha*(100*(CLOSE.SPY.iloc[j]/CLOSE.SPY.iloc[j-1]-1))**2+beta*variance
            SIG6.iloc[j,SIG6.columns.get_loc('GARCH_vol')]=np.sqrt(forecast*252)/100
            variance=forecast
        garch_log.append({'fit_end':CLOSE.index[i],'first_forecast_for':CLOSE.index[i+1] if i+1<len(CLOSE) else pd.NaT,'success':fitted.convergence_flag==0,'alpha_plus_beta':alpha+beta})
    except Exception as e:garch_log.append({'fit_end':CLOSE.index[i],'success':False,'error':str(e)})
# 日内决策生成，下日开盘执行。风险模型原始参数每21日重估，期间仅用已经实现的残差递推。
W6_DEC={n:pd.DataFrame(0.,index=CLOSE.index,columns=ASSETS) for n in ['equal','inverse_vol','shrink_minvar','signal_tilt']}
COV6={}; COV_ASOF6={}; last_cov=None; last_fit=None
panel6=PANEL.set_index(['date','ticker'])
for i,date in enumerate(CLOSE.index):
    if date<pd.Timestamp('2014-01-01'):continue
    hist=R6.loc[:date].tail(252).dropna()
    if len(hist)<200:continue
    if last_cov is None or i%21==0:
        last_cov=LedoitWolf().fit(hist.to_numpy()).covariance_;last_fit=date
    COV6[date]=last_cov.copy();COV_ASOF6[date]=last_fit
    eq=np.ones(len(ASSETS))*.8/len(ASSETS)
    iv=1/np.sqrt(np.maximum(np.diag(last_cov),1e-12));iv=feasible_budget(.8*iv/iv.sum())
    con=[{'type':'eq','fun':lambda w:w.sum()-.8}]+[{'type':'ineq','fun':lambda w,j=j:.35-w[j].sum()} for j in GROUP6.values()]
    if i%21==0 or 'mv' not in locals():
        result=minimize(lambda w:float(w@last_cov@w)*1e5,eq,bounds=[(0,.15)]*len(ASSETS),constraints=con,method='SLSQP',options={'ftol':1e-10,'maxiter':150})
        if result.success and caps_ok(result.x):mv=result.x
        else:mv=iv.copy();FALLBACK6.append({'where':'minvar','date':str(date),'reason':result.message})
    try:
        scores=panel6.loc[date].reindex(ASSETS)[[f+'_z' for f in FACTOR_NAMES]].mean(axis=1).fillna(0).to_numpy()
    except KeyError:scores=np.zeros(len(ASSETS))
    tilt=feasible_budget(eq*(1+.3*np.tanh(scores)))
    for name,w in [('equal',eq),('inverse_vol',iv),('shrink_minvar',mv),('signal_tilt',tilt)]:W6_DEC[name].loc[date]=w
print('Frozen training-only state threshold',state_threshold);display(pd.DataFrame(garch_log).tail());print('fallback events',len(FALLBACK6))
print('06.02 PASS: covariance fit dates and state threshold recorded')


In [ ]:
# 06.03 同一账本的风险配置、换手限制与风险匹配对照
W6_DEC['state_filtered']=W6_DEC['shrink_minvar'].mul(SIG6.state_active.astype(float)*.75+.25,axis=0)
DESIRED6={n:w.shift(1).reindex(EXEC6).fillna(0) for n,w in W6_DEC.items()}
def limited_targets(opens,desired,turnover_cap=.25,cost=.0005,initial=10000.):
    cash=initial;units=np.zeros(desired.shape[1]);rows=[];audits=[]
    def solve(w,old,E):
        post=E
        for _ in range(40):post=E-np.abs(w*post-old).sum()*cost
        return post,w*post-old
    for count,(d,target) in enumerate(desired.iterrows()):
        p=opens.loc[d].to_numpy();nxt=opens.shift(-1).loc[d].to_numpy();old=units*p;E=cash+old.sum();oldw=old/E
        # 若行情漂移使权重超过上限，先做最小比例缩减；超出换手限制的强制降险单独登记。
        repair=min(1.,.8/max(oldw.sum(),1e-12),.15/max(oldw.max(),1e-12),*[.35/max(oldw[j].sum(),1e-12) for j in GROUP6.values()])
        safe=oldw*repair;want=target.to_numpy(float);post,delta=solve(want,old,E);limited=False
        if count>0 and np.abs(delta).sum()/E>turnover_cap:
            low,high=0.,1.
            for _ in range(35):
                mid=(low+high)/2;candidate=safe+mid*(want-safe);p0,d0=solve(candidate,old,E)
                if np.abs(d0).sum()/E<=turnover_cap:low=mid
                else:high=mid
            want=safe+low*(want-safe);post,delta=solve(want,old,E);limited=True
        turn=np.abs(delta).sum()/E
        forced=bool(count>0 and turn>turnover_cap+1e-6)
        fees=np.abs(delta).sum()*cost;units+=delta/p;cash-=delta.sum()+fees
        rows.append(want.copy());audits.append({'date':d,'desired_budget':target.sum(),'executed_budget':want.sum(),'turnover':turn,'limited':limited,'forced_risk_override':forced,'startup':count==0})
        assert cash+units@nxt>0
    return pd.DataFrame(rows,index=desired.index,columns=desired.columns),pd.DataFrame(audits).set_index('date')
L6_TARGETS={};L6_LEDGERS={};L6_TRADES={};L6_LIMIT_LOG={}
def run6(name,desired):
    target,audit=limited_targets(OPEN[ASSETS],desired)
    x,t=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    assert np.allclose(x.turnover,audit.turnover,atol=1e-7)
    L6_TARGETS[name]=target;L6_LEDGERS[name]=x;L6_TRADES[name]=t;L6_LIMIT_LOG[name]=audit
for name,w in DESIRED6.items():run6(name,w)
train_period=slice('2014-02-01','2018-12-31')
matched_scale=min(1.,L6_LEDGERS['state_filtered'].loc[train_period].net_return.std()/L6_LEDGERS['shrink_minvar'].loc[train_period].net_return.std())
run6('risk_matched_unfiltered',DESIRED6['shrink_minvar']*matched_scale)
RESULT6=pd.DataFrame([{'model':n,'period':period,**ledger_stats(x.loc[a:b]),'realized_vol':x.loc[a:b].net_return.std()*np.sqrt(252),'avg_gross':x.loc[a:b].gross_exposure.mean()}
 for n,x in L6_LEDGERS.items() for period,a,b in [('fit','2014-02-01','2018-12-31'),('validation','2019-01-01','2020-12-31'),('development_eval','2021-01-01','2022-12-29')]]).set_index(['period','model'])
print('Risk matching scale frozen from 2014–2018',matched_scale);display(RESULT6)
print('实际评估波动不保证完全相等；风险匹配仅训练期冻结，不按评估结果反向缩放。')
display(pd.DataFrame({n:a[['limited','forced_risk_override']].sum() for n,a in L6_LIMIT_LOG.items()}).T)
fig,ax=plt.subplots(2,2,figsize=(14,8));pd.DataFrame({n:x.wealth for n,x in L6_LEDGERS.items()}).plot(ax=ax[0,0],title='Constrained account net wealth')
pd.DataFrame({n:x.drawdown for n,x in L6_LEDGERS.items()}).plot(ax=ax[0,1],title='Account drawdowns')
SIG6.loc['2014':'2022',['hist_vol','ewma_vol','GARCH_vol']].plot(ax=ax[1,0],title='Past-only close-return volatility forecasts')
L6_TARGETS['state_filtered'].plot.area(ax=ax[1,1],legend=False,title='Realized constrained target weights');plt.tight_layout();plt.show()
print('06.03 PASS: self-financing replay and turnover overlay agree')


In [ ]:
# 06.04 风险贡献、VaR/ES突破聚集、危机压力与多空约束
# 使用GARCH预测作为风险缩放候选：预算只能下降，不借预测提高总杠杆。
garch_multiplier=(.15/SIG6.GARCH_vol.clip(lower=.01)).clip(lower=.25,upper=1).fillna(.25)
garch_desired=W6_DEC['shrink_minvar'].mul(garch_multiplier,axis=0).shift(1).reindex(EXEC6).fillna(0)
run6('GARCH_risk_scaled',garch_desired)
TAIL6=[];TAIL_SERIES6={};CONTRIB6=[]
def independence_lr(hit):
    a=np.asarray(hit,dtype=int);counts=np.array([[np.sum((a[:-1]==i)&(a[1:]==j)) for j in [0,1]] for i in [0,1]],float)
    def ll(n,p):return sum(n[j]*np.log(np.clip([1-p,p][j],1e-12,1)) for j in [0,1])
    p=counts[:,1].sum()/max(counts.sum(),1);p0=counts[0,1]/max(counts[0].sum(),1);p1=counts[1,1]/max(counts[1].sum(),1)
    lr=2*(ll(counts[0],p0)+ll(counts[1],p1)-ll(counts.sum(axis=0),p))
    return lr,float(chi2.sf(max(lr,0),1)),int(np.sum((a[1:]==1)&(a[:-1]==1)))
for name,x in L6_LEDGERS.items():
    loss=-x.net_return;var=loss.shift(1).rolling(250,min_periods=250).quantile(.95)
    es=loss.shift(1).rolling(250,min_periods=250).apply(lambda z:np.mean(z[z>=np.quantile(z,.95)]),raw=True)
    z=pd.DataFrame({'loss':loss,'VaR95':var,'ES95':es}).dropna();z=z.loc['2019-01-01':];z['breach']=z.loss>z.VaR95
    lr,p,consecutive=independence_lr(z.breach)
    TAIL6.append({'model':name,'n':len(z),'breaches':int(z.breach.sum()),'breach_rate':z.breach.mean(),'expected_rate':.05,
      'mean_VaR':z.VaR95.mean(),'mean_ES':z.ES95.mean(),'mean_exceedance_loss':z.loc[z.breach,'loss'].mean(),
      'LR_independence':lr,'asymptotic_p':p,'consecutive_breaches':consecutive,'tail_n_per_window':'about 13; uncertain'})
    TAIL_SERIES6[name]=z
    for d in x.index[::21]:
        decision=PREV6.loc[d];cov=COV6[decision];w=L6_TARGETS[name].loc[d].to_numpy();variance=w@cov@w
        rc=w*(cov@w)/max(variance,1e-15)
        for a,v in zip(ASSETS,rc):CONTRIB6.append({'date':d,'model':name,'instrument':a,'variance_fraction':v})
TAIL_REPORT6=pd.DataFrame(TAIL6).set_index('model');display(TAIL_REPORT6)
RC6=pd.DataFrame(CONTRIB6);display(RC6.groupby(['model','instrument']).variance_fraction.mean().unstack())
STRESS6=pd.DataFrame([{'model':n,'window':w,**ledger_stats(x.loc[a:b])} for n,x in L6_LEDGERS.items()
 for w,a,b in [('2018Q4','2018-10-01','2018-12-31'),('COVID','2020-02-19','2020-04-30'),('2022','2022-01-01','2022-12-29')]]).set_index(['window','model']);display(STRESS6)
# 同波动、相关性提升到0.9的情景：这是假设压力，不能冒充历史观测。
date=PREV6.iloc[-1];cov=COV6[date];sd=np.sqrt(np.diag(cov));corr_stress=.9*np.ones_like(cov)+.1*np.eye(len(ASSETS));shock=np.outer(sd,sd)*corr_stress
w=L6_TARGETS['shrink_minvar'].iloc[-1].to_numpy()
print('Correlation-stress annual vol',np.sqrt(w@shock@w*252),'vs estimated',np.sqrt(w@cov@w*252))
# 多空独立演示：gross .8/net0/单股.1；借券不可得时关闭账户，假设必须标记。
LS=pd.DataFrame(0.,index=EXEC6,columns=ASSETS)
for d in EXEC6:
    dec=PREV6.loc[d]
    if dec not in panel6.index.get_level_values('date'):continue
    g=panel6.loc[dec].reindex(ASSETS);score=g[[f+'_z' for f in FACTOR_NAMES]].mean(axis=1).fillna(0).sort_values(kind='stable')
    LS.loc[d,score.index[:4]]=-.1;LS.loc[d,score.index[-4:]]=.1
LS_LEDGER,LS_TRADES=l2_simulate_multi(OPEN[ASSETS],LS,commission=3,slippage=2,borrow_rate=.03,funding=.05,initial=10000)
available=pd.DataFrame(True,index=LS.index,columns=ASSETS);test_date=LS.index[len(LS)//2];short_name=LS.loc[test_date].idxmin();available.loc[test_date,short_name]=False
LS_BLOCKED,_=l2_simulate_multi(OPEN[ASSETS],LS,commission=3,slippage=2,borrow_rate=.03,funding=.05,available=available,initial=10000)
assert LS_BLOCKED.loc[test_date,'blocked_by_borrow'] and abs(LS_BLOCKED.loc[test_date,'gross_exposure'])<1e-8
print('Long/short scenario, not observed historical borrow');display(pd.DataFrame({'long_short':ledger_stats(LS_LEDGER)}).T)
# 同一个成本账户的完整月收盘权益归因；丢弃不完整最后月。
x=L6_LEDGERS['shrink_minvar'];eqclose=x.cash+sum(x['units_'+a]*CLOSE[a].reindex(x.index) for a in ASSETS)
monthly=eqclose.groupby(eqclose.index.to_period('M')).last();ret=monthly.pct_change(fill_method=None)
ff6=pd.concat([ret.rename('strategy'),FF],axis=1).loc['2014-03':'2022-11'].dropna()
fit6=sm.OLS(ff6.strategy-ff6.RF,sm.add_constant(ff6[['Mkt-RF','SMB','HML','RMW','CMA','Mom']])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
display(pd.DataFrame({'loading':fit6.params,'HAC_t':fit6.tvalues}))
fig,ax=plt.subplots(1,2,figsize=(13,4));TAIL_SERIES6['shrink_minvar'][['loss','VaR95','ES95']].plot(ax=ax[0],title='Lagged same-account historical VaR and ES')
RC6[RC6.model.eq('shrink_minvar')].pivot(index='date',columns='instrument',values='variance_fraction').plot.area(ax=ax[1],legend=False,title='Shrink covariance variance contributions');plt.tight_layout();plt.show()
L6_CHECKS=[]
def check6(name,ok):
    assert bool(ok),name;L6_CHECKS.append({'check':name,'status':'PASS'})
check6('TPU executed',len(TPU_DEVICES)>0 and np.isfinite(device_checksum))
check6('covariance asof before open',all(COV_ASOF6[d]<e for e,d in PREV6.items()))
check6('frozen state threshold uses only fit dates',SIG6.loc['2014':'2018'].index.max()<TRAIN_END)
check6('long-short net and gross',(LS.sum(axis=1).abs()<1e-8).all() and (LS.abs().sum(axis=1)<=.800001).all())
check6('unavailable borrow closes portfolio',LS_BLOCKED.loc[test_date,'blocked_by_borrow'] and abs(LS_BLOCKED.loc[test_date,'gross_exposure'])<1e-8)
check6('ES is tail loss at or above VaR',all((z.ES95>=z.VaR95-1e-10).all() for z in TAIL_SERIES6.values()))
for name,x in L6_LEDGERS.items():
    w=L6_TARGETS[name];audit=L6_LIMIT_LOG[name]
    check6(name+' same window',x.index.equals(EXEC6))
    check6(name+' self finance',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
    check6(name+' positive costs',(x.transaction_cost>=0).all())
    check6(name+' position constraints',(w>=-1e-8).all().all() and w.max().max()<=.150001 and w.sum(axis=1).max()<=.800001 and all(w.iloc[:,j].sum(axis=1).max()<=.350001 for j in GROUP6.values()))
    check6(name+' turnover cap or recorded forced risk',((audit.turnover<=.250001)|audit.startup|audit.forced_risk_override).all())
    check6(name+' compound wealth',np.allclose(x.wealth,(1+x.net_return).cumprod()))
L6_VALIDATION=pd.DataFrame(L6_CHECKS);display(L6_VALIDATION)
L6_CARD={'lesson':6,'status':'verified','checks':len(L6_CHECKS),'constraints':CFG6,'risk_match_scale':matched_scale,
 'limitations':['fixed survivors','daily weight maintenance','risk vol proxy uses closes','ES small tail sample','historical borrow assumed','forced risk overrides explicitly recorded','all historical data already development']}
print('L6 FINAL PASS',json.dumps(L6_CARD,ensure_ascii=False));


# 第7课 市场微观结构与执行实验

目标：从LOBSTER message/orderbook理解价差、深度、不平衡、microprice、订单流、库存与逆向选择。microprice=(ask×bid_size+bid×ask_size)/(bid_size+ask_size)是报价代理，不是保证可成交价格。

优先读取官方2012-06-21 AAPL小样例；若官方端点不可用，严格标记为合成事件，只验证机制。不能把合成/单日结果写成真实高频盈利证据，也不套用252年化逐笔Sharpe。原始受限数据不打包进GitHub。

实验A：TWAP vs 使用更早时段成交量形状估计的VWAP调度；不得用执行窗实际成交量设计“事前VWAP”。买单成交使用延迟后可见卖一档，限制参与和可见深度，未完成部分按期末可见深度强制完成并记入剩余/费用；实现差额使用决策到达mid作为基准，报告执行金额、未完成量和冲击情景。

实验B：双边报价加入库存偏移和限额；新单及撤改单均受延迟，排队前量按报价时可见同价深度估计。仅显示的反向主动成交在相同价位消耗队列后才可能成交；取消不自动改善队列；触价本身不算成交。不处理隐藏单，不以未来成交挑报价。

这是小订单市场回放，自己的订单不改变其他人的事件。延迟、队列倍数和费用做情景敏感性，模拟成本不能直接校准日频ETF/个股账户，除非市场、年代和粒度可比。

参考：[LOBSTER样例入口](https://lobsterdata.com/info/DataSamples.php)、[HftBacktest成交与队列假设](https://hftbacktest.readthedocs.io/en/latest/order_fill.html)。


In [ ]:
# 07.01 官方LOBSTER样例、消息解析及明确标记的合成回退
import requests,zipfile,re
LOB_URL='https://data.lobsterdata.com/info/sample/LOBSTER_SampleFile_AAPL_2012-06-21_1.zip'
LOB_MANIFEST={'requested_url':LOB_URL,'retrieved_utc':pd.Timestamp.now(tz='UTC').isoformat(),'redistribution':'raw data excluded from portfolio package'}
try:
    content=fetch(LOB_URL);archive=zipfile.ZipFile(io.BytesIO(content));names=archive.namelist()
    message_file=next(n for n in names if '_message_' in n and n.endswith('.csv'))
    book_file=next(n for n in names if '_orderbook_' in n and n.endswith('.csv'))
    MSG=pd.read_csv(io.BytesIO(archive.read(message_file)),header=None,names=['time','type','order_id','size','price','direction'])
    BOOK=pd.read_csv(io.BytesIO(archive.read(book_file)),header=None).iloc[:,:4];BOOK.columns=['ask','ask_size','bid','bid_size']
    assert len(MSG)==len(BOOK)
    BOOK[['ask','bid']]=BOOK[['ask','bid']]/10000.;MSG['price']=MSG.price/10000.
    LOB_MANIFEST.update(source_kind='official_single_day_sample',sample_date='2012-06-21',sha256=hashlib.sha256(content).hexdigest(),rows=len(MSG))
except Exception as e:
    rng=np.random.default_rng(7001);n=50000;clock=np.linspace(34200,57600,n)
    ticks=10000+np.cumsum(rng.choice([-1,0,1],size=n,p=[.12,.76,.12]));mid=ticks*.01
    bid=mid-.01;ask=mid+.01;direction=rng.choice([-1,1],n);types=rng.choice([1,2,3,4],n,p=[.45,.12,.12,.31]);sizes=rng.integers(1,101,n)
    MSG=pd.DataFrame({'time':clock,'type':types,'order_id':np.arange(n),'size':sizes,'price':np.where(direction==1,bid,ask),'direction':direction})
    BOOK=pd.DataFrame({'ask':ask,'ask_size':rng.integers(50,501,n),'bid':bid,'bid_size':rng.integers(50,501,n)})
    LOB_MANIFEST.update(source_kind='SYNTHETIC_MECHANISM_ONLY',reason=str(e),seed=7001,rows=n,
      limitations='synthetic L1 quotes and event stream, not a full exchange order-ID reconstruction')
LOB=pd.concat([MSG,BOOK],axis=1)
LOB=LOB[(LOB.time>=34200)&(LOB.time<=57600)&(LOB.ask>LOB.bid)&(LOB.bid>0)&(LOB.ask_size>0)&(LOB.bid_size>0)].reset_index(drop=True)
LOB['mid']=(LOB.ask+LOB.bid)/2;LOB['spread']=LOB.ask-LOB.bid
LOB['imbalance']=(LOB.bid_size-LOB.ask_size)/(LOB.bid_size+LOB.ask_size)
LOB['microprice']=(LOB.ask*LOB.bid_size+LOB.bid*LOB.ask_size)/(LOB.bid_size+LOB.ask_size)
LOB['signed_trade_volume']=np.where(LOB.type.eq(4),-LOB.direction*LOB['size'],0)
LOB['past_order_flow']=LOB.signed_trade_volume.rolling(100,min_periods=1).sum()
LOB['past_mid_change']=LOB.mid.diff().fillna(0)
assert LOB.time.is_monotonic_increasing and np.isfinite(LOB[['mid','spread','imbalance','microprice']]).all().all()
print(json.dumps(LOB_MANIFEST,ensure_ascii=False,indent=2));display(LOB.head());display(LOB[['spread','imbalance','ask_size','bid_size']].describe())
fig,ax=plt.subplots(1,2,figsize=(12,4));LOB.iloc[::100].plot(x='time',y=['mid','microprice'],ax=ax[0],title=LOB_MANIFEST['source_kind'])
LOB.imbalance.hist(bins=40,ax=ax[1]);ax[1].set_title('L1 queue imbalance');plt.tight_layout();plt.show()
print('07.01 PASS: provenance and message/orderbook row mapping preserved')


In [ ]:
# 07.02 TWAP/事前VWAP调度与实现差额账本
CAL_START,CAL_END,EX_START,EX_END=34200.,39600.,43200.,48600.
CAL=LOB[(LOB.time>=CAL_START)&(LOB.time<CAL_END)&LOB.type.eq(4)].copy()
N_CHILD=9;Q_TARGET=1000;cal_bins=np.linspace(CAL_START,CAL_END,N_CHILD+1)
cal_volume=np.histogram(CAL.time,bins=cal_bins,weights=CAL['size'])[0].astype(float)
VWAP_PROFILE=(cal_volume+1)/(cal_volume.sum()+N_CHILD)
TWAP_PROFILE=np.ones(N_CHILD)/N_CHILD
assert CAL.time.max()<EX_START
EXEC7={};EXEC_REPORT7=[]
def execute_schedule(profile,latency=.25,participation=.1,impact_bps=0.):
    child_times=np.linspace(EX_START,EX_END,N_CHILD,endpoint=False)
    allocation=np.floor(Q_TARGET*np.asarray(profile)).astype(int);allocation[-1]+=Q_TARGET-allocation.sum()
    arrival_idx=max(0,LOB.time.searchsorted(EX_START,side='right')-1);arrival_mid=float(LOB.mid.iloc[arrival_idx]);used={};rows=[];remaining=Q_TARGET
    for t,q in zip(np.r_[child_times,EX_END],np.r_[allocation,Q_TARGET]):
        if remaining<=0:break
        i=int(LOB.time.searchsorted(t+latency,side='left'))
        if i>=len(LOB):break
        quote=LOB.iloc[i];available=max(0,int(quote.ask_size)-used.get(i,0));request=min(remaining,int(q))
        # 小订单显示深度限制；自身成交不改变回放市场，单事件重复使用深度被扣除。
        fill=min(request,int(available*participation));used[i]=used.get(i,0)+fill
        price=float(quote.ask)*(1+impact_bps/10000);fee=fill*price*.0001
        rows.append({'request_at':t,'ready_at':t+latency,'fill_at':quote.time,'event_index':i,'requested':request,'filled':fill,'price':price,'fee':fee,'remaining_after':remaining-fill,'forced_end':t==EX_END})
        remaining-=fill
    trades=pd.DataFrame(rows);filled=trades.filled.sum();notional=(trades.filled*trades.price).sum();fees=trades.fee.sum()
    end_idx=min(len(LOB)-1,int(LOB.time.searchsorted(EX_END+latency,side='left')));end_mid=float(LOB.mid.iloc[end_idx])
    report={'target':Q_TARGET,'filled':int(filled),'unfilled':int(remaining),'fill_rate':filled/Q_TARGET,'arrival_mid':arrival_mid,
      'VWAP_execution':notional/max(filled,1),'fees':fees,'IS_filled_bps':((notional+fees)/max(filled,1)/arrival_mid-1)*10000 if filled else np.nan,
      'opportunity_adjusted_IS_bps':((notional+fees+remaining*end_mid)/(Q_TARGET*arrival_mid)-1)*10000,
      'remaining_mark_is_not_a_fill':True,'latency_seconds':latency,'participation_of_displayed_depth':participation,'impact_assumption_bps':impact_bps}
    return trades,report
for name,profile in [('TWAP',TWAP_PROFILE),('past_volume_schedule',VWAP_PROFILE)]:
    for latency in [0.,.25,1.]:
        trades,report=execute_schedule(profile,latency=latency);key=name+'_latency_'+str(latency);EXEC7[key]=trades;EXEC_REPORT7.append({'algorithm':name,**report})
EXEC_RESULT7=pd.DataFrame(EXEC_REPORT7);display(pd.DataFrame({'TWAP':TWAP_PROFILE,'past_volume':VWAP_PROFILE,'calibration_volume':cal_volume}));display(EXEC_RESULT7)
print('事后市场VWAP只作描述，不回填事前调度。剩余订单按期末mid计机会成本，不能冒充真实成交。')
executed_window=LOB[(LOB.time>=EX_START)&(LOB.time<=EX_END)&LOB.type.eq(4)]
print('Ex-post market trade VWAP',np.average(executed_window.price,weights=executed_window['size']))
fig,ax=plt.subplots(1,2,figsize=(12,4));pd.DataFrame({'TWAP':TWAP_PROFILE,'past_volume':VWAP_PROFILE}).plot.bar(ax=ax[0],title='Schedules fixed before execution window')
for name,t in EXEC7.items():ax[1].step(t.fill_at,t.filled.cumsum(),where='post',label=name)
ax[1].axhline(Q_TARGET,ls='--',color='k');ax[1].legend(fontsize=7);ax[1].set_title('Cumulative actual fills');plt.tight_layout();plt.show()
print('07.02 PASS: no future volume used, displayed depth and incomplete fills retained')


In [ ]:
# 07.03 队列、撤改单延迟与库存控制做市原型
MM_WINDOW=LOB[(LOB.time>=EX_START)&(LOB.time<=EX_END)].copy().reset_index(drop=True)
TICK7=.01
# 消息type4为显示限价单执行；direction是被执行的挂单方向，不能当主动方方向。
def market_make(latency=.25,queue_multiplier=1.,inventory_control=True,fee_bps=1.):
    cash=0.;inv=0;orders={};pending=None;records=[];fills=[];limit=20;lot=5;last_refresh=-np.inf;requests=0
    for i in range(1,len(MM_WINDOW)):
        event=MM_WINDOW.iloc[i];prev=MM_WINDOW.iloc[i-1];now=float(event.time)
        if now-last_refresh>=10 and pending is None:
            shift=int(np.round(inv/5)) if inventory_control else 0
            desired={1:float(prev.bid)-shift*TICK7,-1:float(prev.ask)-shift*TICK7}
            pending={'ready':float(prev.time)+latency,'quotes':desired,'request_at':float(prev.time)};last_refresh=now;requests+=2
        if pending is not None and pending['ready']<=now:
            orders={} # cancellation only now effective; old quotes remained live through delay
            for side,price in pending['quotes'].items():
                best=float(prev.bid if side==1 else prev.ask);depth=float(prev.bid_size if side==1 else prev.ask_size)
                passive=(price<float(prev.ask) if side==1 else price>float(prev.bid))
                allowed=(inv+lot<=limit if side==1 else inv-lot>=-limit)
                if passive and allowed:
                    ahead=depth*queue_multiplier if abs(price-best)<TICK7/4 else np.inf
                    orders[side]={'price':price,'remaining':lot,'ahead':ahead,'activated_at':now,'request_at':pending['request_at']}
            pending=None
        if int(event['type'])==4:
            side=int(event.direction);order=orders.get(side)
            if order is not None and abs(float(event.price)-order['price'])<TICK7/4:
                before=order['ahead'];quantity=float(event['size']);order['ahead']=max(0,before-quantity)
                possible=max(0,quantity-before);q=int(min(order['remaining'],possible))
                q=min(q,limit-inv if side==1 else limit+inv)
                if q>0:
                    fee=q*order['price']*fee_bps/10000;cash-=side*q*order['price']+fee;inv+=side*q;order['remaining']-=q
                    fills.append({'time':now,'side':side,'quantity':q,'price':order['price'],'fee':fee,'inventory':inv,'ahead_before':before,
                       'request_at':order['request_at'],'activated_at':order['activated_at'],'source_type':4,'trade_quantity':quantity})
                if order['remaining']==0:orders.pop(side,None)
        assert abs(inv)<=limit
        records.append({'time':now,'cash':cash,'inventory':inv,'mid':float(event.mid),'equity':cash+inv*float(event.mid),'live_orders':len(orders)})
    path=pd.DataFrame(records);trades=pd.DataFrame(fills)
    # 在可见最优档容量内平仓；不足时只mark剩余，不捏造成交。
    last=MM_WINDOW.iloc[-1];closing=min(abs(inv),int(last.bid_size if inv>0 else last.ask_size))
    liquidation_price=float(last.bid if inv>0 else last.ask);side=-np.sign(inv);liq_fee=closing*liquidation_price*fee_bps/10000
    cash-=side*closing*liquidation_price+liq_fee;inv+=int(side*closing)
    final=cash+inv*float(last.mid)
    report={'latency_seconds':latency,'queue_multiplier':queue_multiplier,'inventory_control':inventory_control,'fee_bps':fee_bps,
      'source':LOB_MANIFEST['source_kind'],'posted_requests':requests,'filled_units':int(trades.quantity.sum()) if len(trades) else 0,
      'fill_events':len(trades),'net_marked_PnL':final,'max_abs_inventory':path.inventory.abs().max(),'remaining_inventory':inv,
      'closing_units':closing,'closing_fee':liq_fee,'min_marked_PnL':path.equity.min(),'source_window_events':len(path)}
    if len(trades):
        future_mid=[]
        for _,r in trades.iterrows():
            j=min(len(MM_WINDOW)-1,int(MM_WINDOW.time.searchsorted(r.time+5)))
            future_mid.append(r.side*(float(MM_WINDOW.mid.iloc[j])-r.price))
        trades['signed_5sec_markout']=future_mid;report['mean_markout_per_share']=np.average(trades.signed_5sec_markout,weights=trades.quantity)
    return path,trades,report
MM_PATHS7={};MM_FILLS7={};MM_REPORT=[]
for controlled in [False,True]:
    for latency in [0.,.25,1.]:
        for queue in [1.,2.]:
            key=(controlled,latency,queue);path,trades,report=market_make(latency,queue,controlled)
            MM_PATHS7[key]=path;MM_FILLS7[key]=trades;MM_REPORT.append(report)
MM_RESULT7=pd.DataFrame(MM_REPORT);display(MM_RESULT7)
fig,ax=plt.subplots(1,2,figsize=(13,4))
for key,p in MM_PATHS7.items():
    if key[2]==1:ax[0].plot(p.time,p.equity,label=str(key));ax[1].plot(p.time,p.inventory,label=str(key))
ax[0].legend(fontsize=7);ax[0].set_title('Marked PnL, replay queue/latency assumptions');ax[1].set_title('Inventory, hard cap ±20 shares');plt.tight_layout();plt.show()
print('取消不推进队列；触价不成交。改单就重新排队。未建模隐藏量、跨档优先级、冲击反馈与交易所微秒协议。')
L7_CHECKS=[]
def check7(name,ok):
    assert bool(ok),name;L7_CHECKS.append({'check':name,'status':'PASS'})
check7('source explicitly actual or synthetic',LOB_MANIFEST['source_kind'] in ['official_single_day_sample','SYNTHETIC_MECHANISM_ONLY'])
check7('volume schedule ends before execution',CAL.time.max()<EX_START)
check7('buy/sell quotes ordered',(LOB.ask>LOB.bid).all())
check7('microprice inside spread',((LOB.microprice>=LOB.bid-1e-9)&(LOB.microprice<=LOB.ask+1e-9)).all())
check7('imbalance in [-1,1]',LOB.imbalance.abs().max()<=1)
# 手算队列：前方100，第一笔60不能成交，第二笔50只能成交10。
ahead=100;fill1=max(0,60-ahead);ahead=max(0,ahead-60);fill2=min(20,max(0,50-ahead))
check7('manual queue conservation',fill1==0 and fill2==10)
for name,t in EXEC7.items():
    check7(name+' temporal latency',(t.fill_at>=t.ready_at).all())
    check7(name+' fill conservation',int(t.filled.sum()+t.remaining_after.iloc[-1])==Q_TARGET)
    check7(name+' partial not fabricated',(t.filled<=t.requested).all() and (t.fee>=0).all())
for key,p in MM_PATHS7.items():
    check7(str(key)+' cash-inventory mark',np.allclose(p.equity,p.cash+p.inventory*p.mid))
    check7(str(key)+' inventory cap',p.inventory.abs().max()<=20)
    f=MM_FILLS7[key]
    check7(str(key)+' displayed trade only',f.empty or f.source_type.eq(4).all())
    check7(str(key)+' latency and queue',f.empty or ((f.time>=f.activated_at)&(f.activated_at>=f.request_at+key[1])&(f.quantity<=np.maximum(0,f.trade_quantity-f.ahead_before)+1e-8)).all())
L7_VALIDATION=pd.DataFrame(L7_CHECKS);display(L7_VALIDATION)
L7_CARD={'lesson':7,'checks':len(L7_CHECKS),'source':LOB_MANIFEST,'status':'verified_mechanisms','limits':['single-day or synthetic explicitly marked','no market impact response','cancel/hidden queues uncertain','not evidence of live HFT profitability','not used to calibrate daily stock costs']}
print('L7 FINAL PASS',json.dumps(L7_CARD,ensure_ascii=False))


# 第8课 强化学习与序贯配置（TPU训练＋SB3基线）

目标：定义观察、动作、奖励和episode，让PPO在第六课相同资产、总多头0.8、单股0.15、行业0.35与换手上限0.25下选择配置。动作从现金、等权、逆波动、最小方差、状态过滤5个已知可行组合中选择，执行端进一步处理真实漂移和换手；所有策略最终输出target_weight并由第二课账户重放。

状态包含过去动量、波动、趋势、统计套利残差、因子摘要、当前仓位与回撤；不含未来收益或未成熟标签。数值归一化仅fit2014—2018。未来价格仅在step转移后产生净收益和奖励。

奖励=100×log(Vnext/V)−λ×max(Dnext−0.10,0)，V已扣一次真实交易费用。回撤惩罚是学习目标，不再次从现金扣钱。现金可能提高奖励但降低资本参与率，必须对照平均敞口、换手和净收益。历史回放不响应智能体，不能当真实在线学习。

先check_env、随机动作和固定权重手工核账。Stable-Baselines3 PPO作为主机参考；另实现离散动作PPO的JAX梯度更新，在实际TPU上训练，保留裁剪目标、GAE、价值损失和熵项。两种实现各5随机种子、固定训练步数；λ=0作为TPU奖励消融。验证2019—2020选择模型家族；2021—2022为本项目已观察开发评估，冻结选择后一次报告，不声称全新测试。

episode到数据边界为terminated，到人为长度为truncated；GAE对truncated保留末态bootstrap，对terminated归零。模型检查点保存为内存字节，最终打入云端项目包，不写本地文件。多种子报告中位数、四分位差、失败和不及规则基线次数。

参考：[Gymnasium自定义环境](https://gymnasium.farama.org/introduction/create_custom_env/)、[SB3评估和多种子](https://stable-baselines3.readthedocs.io/en/v2.7.0/guide/rl_tips.html)。


In [ ]:
# 08.01 因果状态、可行候选动作与同账本一步转移
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
import torch
SEEDS8=[11,22,33,44,55];BUDGET8=8192
DATES8=EXEC6;PREV8=PREV6
FEATURE8=pd.DataFrame(index=DATES8)
for c in ['hist_vol','ewma_vol','trend']:
    FEATURE8[c]=PREV8.map(SIG6[c])
FEATURE8['pair_z']=PREV8.map(l2_features.z)
FEATURE8['market_ret20']=PREV8.map(CLOSE.SPY.pct_change(20,fill_method=None))
for f in FACTOR_NAMES:
    daily=PANEL.groupby('date')[f].mean();FEATURE8['cross_'+f]=PREV8.map(daily)
FEATURE8=FEATURE8.replace([np.inf,-np.inf],np.nan)
TRAIN8=np.flatnonzero(DATES8<TRAIN_END);VALID8=np.flatnonzero((DATES8>=TRAIN_END)&(DATES8<VALID_END));TEST8=np.flatnonzero(DATES8>=VALID_END)
# 训练最后一个转移的估值开盘也必须在验证前，保守剔除边界。
END8=pd.Series(OPEN.index,index=OPEN.index).shift(-1).reindex(DATES8)
TRAIN8=np.flatnonzero((DATES8<TRAIN_END)&(END8.to_numpy()<TRAIN_END.to_datetime64()))
VALID8=np.flatnonzero((DATES8>=TRAIN_END)&(DATES8<VALID_END)&(END8.to_numpy()<VALID_END.to_datetime64()))
MED8=FEATURE8.iloc[TRAIN8].median();MEAN8=FEATURE8.iloc[TRAIN8].fillna(MED8).mean();STD8=FEATURE8.iloc[TRAIN8].fillna(MED8).std().clip(lower=1e-6)
X8=((FEATURE8.fillna(MED8)-MEAN8)/STD8).clip(-10,10).to_numpy(np.float32)
SLEEVES8=['cash','equal','inverse_vol','shrink_minvar','state_filtered']
A8=np.stack([np.zeros((len(DATES8),len(ASSETS)))]+[DESIRED6[n].reindex(DATES8).to_numpy() for n in SLEEVES8[1:]],axis=1).astype(np.float32)
RETURN8=(OPEN[ASSETS].shift(-1)/OPEN[ASSETS]-1).reindex(DATES8).to_numpy(np.float64)
assert np.isfinite(X8).all() and np.isfinite(RETURN8).all()
def step_account8(weight,wealth,peak,desired,ret,cost=.0005,penalty=.05):
    old=np.asarray(weight,float);desired=np.asarray(desired,float)
    repair=min(1.,.8/max(old.sum(),1e-12),.15/max(old.max(),1e-12),*[.35/max(old[j].sum(),1e-12) for j in GROUP6.values()])
    safe=old*repair
    def trade(w):
        post=1.
        for _ in range(20):post=1-cost*np.abs(w*post-old).sum()
        delta=w*post-old;return post,delta
    post,delta=trade(desired);turn=np.abs(delta).sum()
    if turn>.25:
        lo,hi=0.,1.
        for _ in range(28):
            mid=(lo+hi)/2;p,d=trade(safe+mid*(desired-safe))
            if np.abs(d).sum()<=.25:lo=mid
            else:hi=mid
        desired=safe+lo*(desired-safe);post,delta=trade(desired)
    fee=cost*np.abs(delta).sum();cash=1-(desired*post).sum()-fee
    end_values=desired*post*(1+ret);growth=cash+end_values.sum();new_wealth=wealth*growth;new_peak=max(peak,new_wealth)
    dd=1-new_wealth/new_peak;reward=100*np.log(max(growth,1e-12))-penalty*max(dd-.10,0)
    return end_values/growth,new_wealth,new_peak,reward,desired,fee,np.abs(delta).sum()
class PortfolioEnv8(gym.Env):
    metadata={'render_modes':[]}
    def __init__(self,indices,training=False,penalty=.05,horizon=128):
        self.indices=np.asarray(indices);self.training=training;self.penalty=penalty;self.horizon=horizon
        self.action_space=spaces.Discrete(len(SLEEVES8));self.observation_space=spaces.Box(-np.inf,np.inf,shape=(X8.shape[1]+len(ASSETS)+1,),dtype=np.float32)
    def observation(self):return np.r_[X8[self.i],self.weight,1-self.wealth/self.peak].astype(np.float32)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed);start=int(self.np_random.integers(0,max(1,len(self.indices)-self.horizon))) if self.training else 0
        self.local=start;self.i=int(self.indices[start]);self.steps=0;self.weight=np.zeros(len(ASSETS));self.wealth=1.;self.peak=1.;self.history=[]
        return self.observation(),{}
    def step(self,action):
        d=DATES8[self.i];oldwealth=self.wealth
        self.weight,self.wealth,self.peak,reward,target,fee,turn=step_account8(self.weight,self.wealth,self.peak,A8[self.i,int(action)],RETURN8[self.i],penalty=self.penalty)
        info={'date':d,'target_weight':target.copy(),'fee_fraction':fee,'turnover':turn,'wealth':self.wealth,'net_return':self.wealth/oldwealth-1}
        self.history.append(info);self.steps+=1;self.local+=1
        terminated=self.local>=len(self.indices) or self.wealth<=0
        truncated=bool(self.training and self.steps>=self.horizon and not terminated)
        if not terminated:self.i=int(self.indices[self.local])
        obs=self.observation()
        return obs,float(reward),bool(terminated),truncated,info
ENV8=PortfolioEnv8(TRAIN8,True);check_env(ENV8,warn=True)
obs,_=ENV8.reset(seed=11)
for _ in range(20):
    obs,reward,terminated,truncated,info=ENV8.step(ENV8.action_space.sample())
    assert np.isfinite(obs).all() and np.isfinite(reward)
    if terminated or truncated:break
print('08.01 PASS',X8.shape,A8.shape,'normalizer fit cutoff',str(DATES8[TRAIN8[-1]]),'actions',SLEEVES8)


In [ ]:
# 08.02 SB3五种子基线、冻结评估与内存检查点
SB3_MODELS8={};CHECKPOINTS8={};TRAIN_LOG8=[]
def evaluate_policy8(policy,indices,penalty=.05):
    env=PortfolioEnv8(indices,False,penalty=penalty);obs,_=env.reset();actions=[];done=False
    while not done:
        action=int(policy(obs));actions.append(action);obs,r,term,trunc,info=env.step(action);done=term or trunc
    target=pd.DataFrame([x['target_weight'] for x in env.history],index=[x['date'] for x in env.history],columns=ASSETS)
    ledger,trades=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    assert np.allclose(ledger.wealth,[h['wealth'] for h in env.history],rtol=2e-5,atol=2e-6)
    return ledger,target,np.asarray(actions)
SB3_EVAL8={};BASELINE_EVAL8={};SEED_RESULTS8=[]
for name,a in [('cash',0),('equal',1),('inverse_vol',2),('rule_risk',4)]:
    for fold,indices in [('validation',VALID8),('development_eval',TEST8)]:
        x,t,act=evaluate_policy8(lambda obs,a=a:a,indices);BASELINE_EVAL8[(name,fold)]=(x,t,act)
        SEED_RESULTS8.append({'implementation':'rule','penalty':.05,'seed':-1,'model':name,'fold':fold,**ledger_stats(x),'avg_gross':x.gross_exposure.mean()})
torch.set_num_threads(2)
for seed in SEEDS8:
    started=time.perf_counter()
    try:
        model=PPO('MlpPolicy',PortfolioEnv8(TRAIN8,True,penalty=.05,horizon=128),seed=seed,device='cpu',n_steps=128,batch_size=64,
          n_epochs=4,learning_rate=3e-4,gamma=.99,gae_lambda=.95,clip_range=.2,ent_coef=.01,vf_coef=.5,
          policy_kwargs={'net_arch':dict(pi=[32,32],vf=[32,32])},verbose=0)
        model.learn(total_timesteps=BUDGET8);SB3_MODELS8[seed]=model
        buffer=io.BytesIO();model.save(buffer);CHECKPOINTS8['SB3_seed_'+str(seed)+'.zip']=buffer.getvalue()
        for fold,indices in [('validation',VALID8),('development_eval',TEST8)]:
            x,t,act=evaluate_policy8(lambda obs,m=model:int(m.predict(obs,deterministic=True)[0]),indices)
            SB3_EVAL8[(seed,fold)]=(x,t,act)
            SEED_RESULTS8.append({'implementation':'SB3_CPU','penalty':.05,'seed':seed,'model':'PPO','fold':fold,**ledger_stats(x),'avg_gross':x.gross_exposure.mean()})
        TRAIN_LOG8.append({'implementation':'SB3_CPU','seed':seed,'timesteps':int(model.num_timesteps),'status':'completed','seconds':time.perf_counter()-started})
    except Exception as e:TRAIN_LOG8.append({'implementation':'SB3_CPU','seed':seed,'status':'failed','error':str(e)})
    print('SB3 seed',seed,TRAIN_LOG8[-1]['status'])
assert len(SB3_MODELS8)==5,'Some seed failed: retain failure log; do not silently replace seeds'
display(pd.DataFrame(SEED_RESULTS8));display(pd.DataFrame(TRAIN_LOG8));print('08.02 PASS: five reference seeds, exact replay, in-memory checkpoints')


In [ ]:
# 08.03 JAX离散PPO：实际TPU梯度、GAE与五种子奖励消融
from functools import partial
OBS_DIM8=X8.shape[1]+len(ASSETS)+1
with jax.default_device(TPU):
    def init_ppo8(seed):
        keys=jax.random.split(jax.random.PRNGKey(seed),4)
        return {'W1':jax.random.normal(keys[0],(OBS_DIM8,32))*np.sqrt(2/OBS_DIM8),'b1':jnp.zeros(32),
         'W2':jax.random.normal(keys[1],(32,32))*np.sqrt(2/32),'b2':jnp.zeros(32),
         'Wp':jax.random.normal(keys[2],(32,5))*.01,'bp':jnp.zeros(5),
         'Wv':jax.random.normal(keys[3],(32,1))*.1,'bv':jnp.zeros(1)}
@jax.jit
def actor8(params,obs):
    h=jnp.tanh(obs@params['W1']+params['b1']);h=jnp.tanh(h@params['W2']+params['b2'])
    return h@params['Wp']+params['bp'],(h@params['Wv']+params['bv']).squeeze(-1)
def ppo_loss8(params,batch):
    obs,act,old_log,adv,target=batch;logits,value=actor8(params,obs);logp=jax.nn.log_softmax(logits);new_log=jnp.take_along_axis(logp,act[:,None],axis=1).squeeze(1)
    ratio=jnp.exp(new_log-old_log);a=(adv-adv.mean())/(adv.std()+1e-8)
    policy=-jnp.mean(jnp.minimum(ratio*a,jnp.clip(ratio,.8,1.2)*a));v_loss=.5*jnp.mean((value-target)**2)
    entropy=-jnp.mean(jnp.sum(jnp.exp(logp)*logp,axis=1))
    return policy+.5*v_loss-.01*entropy,(policy,v_loss,entropy,jnp.mean((ratio-1)-jnp.log(ratio)),jnp.mean(jnp.abs(ratio-1)>.2))
@jax.jit
def adam_update8(params,m,v,step,batch):
    (loss,diagnostics),grad=jax.value_and_grad(ppo_loss8,has_aux=True)(params,batch)
    magnitude=jnp.sqrt(sum(jnp.sum(g*g) for g in jax.tree_util.tree_leaves(grad)))
    grad=jax.tree_util.tree_map(lambda g:g*jnp.minimum(1.,.5/(magnitude+1e-8)),grad)
    m=jax.tree_util.tree_map(lambda x,g:.9*x+.1*g,m,grad);v=jax.tree_util.tree_map(lambda x,g:.999*x+.001*g*g,v,grad)
    step=step+1
    params=jax.tree_util.tree_map(lambda p,a,b:p-3e-4*(a/(1-.9**step))/(jnp.sqrt(b/(1-.999**step))+1e-8),params,m,v)
    return params,m,v,step,loss,diagnostics
JAX_MODELS8={};JAX_EVAL8={};PPO_DIAGNOSTICS8=[]
for penalty in [.05,0.]:
    for seed in SEEDS8:
        started=time.perf_counter();rng=np.random.default_rng(seed);env=PortfolioEnv8(TRAIN8,True,penalty=penalty,horizon=128)
        try:
            with jax.default_device(TPU):
                params=init_ppo8(seed);m=jax.tree_util.tree_map(jnp.zeros_like,params);v=jax.tree_util.tree_map(jnp.zeros_like,params);step=jnp.array(0)
            for update in range(BUDGET8//128):
                obs,_=env.reset(seed=seed*1000+update);O=[];ACT=[];LP=[];V=[];NEXT=[];REW=[];TERM=[];CUT=[]
                for k in range(128):
                    logits,value=actor8(params,jax.device_put(np.asarray(obs,np.float32),TPU));logit=np.asarray(logits);prob=np.exp(logit-logit.max());prob=prob/prob.sum()
                    action=int(rng.choice(5,p=prob));nextobs,reward,term,trunc,info=env.step(action)
                    _,nextvalue=actor8(params,jax.device_put(nextobs,TPU))
                    O.append(obs);ACT.append(action);LP.append(np.log(max(prob[action],1e-12)));V.append(float(value));NEXT.append(float(nextvalue));REW.append(reward);TERM.append(term);CUT.append(trunc)
                    obs=nextobs
                    if term or trunc:break
                advantages=np.zeros(len(REW),np.float32);carry=0.
                for k in reversed(range(len(REW))):
                    delta=REW[k]+.99*NEXT[k]*(1-int(TERM[k]))-V[k]
                    carry=delta+.99*.95*(1-int(TERM[k] or CUT[k]))*carry;advantages[k]=carry
                batch=tuple(jax.device_put(np.asarray(z,dtype=d),TPU) for z,d in zip([O,ACT,LP,advantages,advantages+np.asarray(V)], [np.float32,np.int32,np.float32,np.float32,np.float32]))
                assert len(REW)==128,'fixed rollout budget changed unexpectedly'
                for epoch in range(4):params,m,v,step,loss,diag=adam_update8(params,m,v,step,batch)
                if update%16==0 or update==BUDGET8//128-1:
                    loss.block_until_ready();PPO_DIAGNOSTICS8.append({'penalty':penalty,'seed':seed,'update':update,'loss':float(loss),'policy_loss':float(diag[0]),'value_loss':float(diag[1]),'entropy':float(diag[2]),'approx_KL':float(diag[3]),'clip_fraction':float(diag[4])})
            assert all(np.isfinite(np.asarray(p)).all() for p in jax.tree_util.tree_leaves(params))
            JAX_MODELS8[(penalty,seed)]=params
            buffer=io.BytesIO();np.savez_compressed(buffer,**{k:np.asarray(v) for k,v in params.items()});CHECKPOINTS8['JAX_penalty_'+str(penalty)+'_seed_'+str(seed)+'.npz']=buffer.getvalue()
            for fold,indices in [('validation',VALID8),('development_eval',TEST8)]:
                x,t,act=evaluate_policy8(lambda obs,p=params:int(np.asarray(actor8(p,jax.device_put(obs,TPU))[0]).argmax()),indices,penalty=penalty)
                JAX_EVAL8[(penalty,seed,fold)]=(x,t,act)
                SEED_RESULTS8.append({'implementation':'JAX_TPU','penalty':penalty,'seed':seed,'model':'PPO','fold':fold,**ledger_stats(x),'avg_gross':x.gross_exposure.mean()})
            TRAIN_LOG8.append({'implementation':'JAX_TPU','penalty':penalty,'seed':seed,'timesteps':BUDGET8,'updates':int(step),'device':str(TPU),'status':'completed','seconds':time.perf_counter()-started})
        except Exception as e:
            TRAIN_LOG8.append({'implementation':'JAX_TPU','penalty':penalty,'seed':seed,'status':'failed','error':str(e)});print('JAX failure',penalty,seed,str(e));raise
        print('JAX TPU',penalty,seed,'completed',round(time.perf_counter()-started,1),'seconds',flush=True)
assert len(JAX_MODELS8)==10
print('Gradient arrays actually on',next(iter(JAX_MODELS8.values()))['W1'].devices());display(pd.DataFrame(PPO_DIAGNOSTICS8).tail(10));display(pd.DataFrame(TRAIN_LOG8))
print('08.03 PASS: 10 TPU checkpoints; fixed seeds and budgets; truncation bootstrap explicit')


In [ ]:
# 08.04 多种子冻结选择、奖励消融及环境漏洞审计
RESULT8=pd.DataFrame(SEED_RESULTS8)
PPO_RESULTS8=RESULT8[RESULT8.model.eq('PPO')].copy();PPO_RESULTS8['family']=PPO_RESULTS8.implementation+'_lambda_'+PPO_RESULTS8.penalty.astype(str)
SUMMARY8=PPO_RESULTS8.groupby(['fold','family']).agg(median_return=('net_return','median'),q25_return=('net_return',lambda x:x.quantile(.25)),
 q75_return=('net_return',lambda x:x.quantile(.75)),median_Sharpe=('Sharpe_0rf','median'),median_drawdown=('max_drawdown','median'),avg_gross=('avg_gross','median'),runs=('seed','size'))
display(SUMMARY8);display(RESULT8[RESULT8.implementation.eq('rule')])
family_validation=SUMMARY8.xs('validation').median_Sharpe
SELECTED_FAMILY8=family_validation.sort_values(ascending=False,kind='stable').index[0]
SELECTED_IMPL8='JAX_TPU' if SELECTED_FAMILY8.startswith('JAX') else 'SB3_CPU'
SELECTED_PENALTY8=float(SELECTED_FAMILY8.rsplit('_',1)[1])
print('Family selected on validation median Sharpe only:',SELECTED_FAMILY8)
FAILURES8=[]
for fold in ['validation','development_eval']:
    baseline=float(RESULT8[(RESULT8.model=='rule_risk')&(RESULT8.fold==fold)].net_return.iloc[0])
    for family,g in PPO_RESULTS8[PPO_RESULTS8.fold.eq(fold)].groupby('family'):
        FAILURES8.append({'fold':fold,'family':family,'n':len(g),'negative_returns':int((g.net_return<0).sum()),'below_rule_risk':int((g.net_return<=baseline).sum())})
display(pd.DataFrame(FAILURES8));print('训练运行失败',sum(r['status']=='failed' for r in TRAIN_LOG8),'；性能不及规则与程序失败分开报告。')
# 共资本合成：平均的是同资产目标权重，再用一个账户记账，不平均独立账户净值。
ENSEMBLE8={}
for fold in ['validation','development_eval']:
    targets=[SB3_EVAL8[(s,fold)][1] if SELECTED_IMPL8=='SB3_CPU' else JAX_EVAL8[(SELECTED_PENALTY8,s,fold)][1] for s in SEEDS8]
    proposed=sum(targets)/len(targets);actual,audit=limited_targets(OPEN[ASSETS],proposed)
    ledger,trades=l2_simulate_multi(OPEN[ASSETS],actual,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    ENSEMBLE8[fold]=(ledger,actual,audit);print('Selected family ensemble',fold,ledger_stats(ledger))
L8_CHECKS=[]
def check8(name,ok):
    assert bool(ok),name;L8_CHECKS.append({'check':name,'status':'PASS'})
check8('SB3 five completed seeds',len(SB3_MODELS8)==5)
check8('TPU five seeds per reward variant',len(JAX_MODELS8)==10 and all(v['W1'].devices()=={TPU} for v in JAX_MODELS8.values()))
check8('fixed actual training budget',all(r.get('timesteps')==BUDGET8 for r in TRAIN_LOG8 if r['status']=='completed'))
check8('normalizer train median',np.allclose(MED8,FEATURE8.iloc[TRAIN8].median(),equal_nan=True))
check8('train end before validation',(END8.iloc[TRAIN8]<TRAIN_END).all())
check8('validation end before evaluation',(END8.iloc[VALID8]<VALID_END).all())
# 人工价格突变只能影响转移，不应改变当前观察。
env=PortfolioEnv8(VALID8,False);o1,_=env.reset();i=env.i;saved=RETURN8[i].copy();RETURN8[i]=saved+.05;o2=env.observation();RETURN8[i]=saved
check8('future return absent from current state',np.array_equal(o1,o2))
# 手算无波动买入：费用只扣一次，扣费后权益=1/(1+c*targetgross)。
_,wealth,_,reward,target,fee,turn=step_account8(np.zeros(12),1.,1.,np.ones(12)*.2/12,np.zeros(12),penalty=0)
check8('manual self financing fee',np.isclose(wealth,1/(1+.0005*.2),atol=1e-9) and np.isclose(fee,1-wealth,atol=1e-9))
check8('reward is net log return',np.isclose(reward,100*np.log(wealth)))
short_env=PortfolioEnv8(TRAIN8,True,horizon=2);short_env.reset(seed=3);short_env.step(1);_,_,term,trunc,_=short_env.step(1)
check8('artificial horizon is truncation',trunc and not term)
last_env=PortfolioEnv8(TRAIN8[-2:],False);last_env.reset();last_env.step(1);_,_,term,trunc,_=last_env.step(1)
check8('data boundary is termination',term and not trunc)
for family,collection in [('SB3',SB3_EVAL8),('JAX',JAX_EVAL8)]:
    for key,(x,t,a) in collection.items():
        check8(family+str(key)+' accounting',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
        check8(family+str(key)+' constraints',(t>=-1e-7).all().all() and t.sum(axis=1).max()<=.800001 and t.max().max()<=.150001 and all(t.iloc[:,j].sum(axis=1).max()<=.350001 for j in GROUP6.values()))
        check8(family+str(key)+' no invalid action',np.isin(a,np.arange(5)).all())
        check8(family+str(key)+' turnover bound',x.turnover.max()<=.25001)
L8_VALIDATION=pd.DataFrame(L8_CHECKS);display(L8_VALIDATION)
fig,ax=plt.subplots(1,2,figsize=(13,4));dev=PPO_RESULTS8[PPO_RESULTS8.fold.eq('development_eval')]
for family,g in dev.groupby('family'):ax[0].scatter([family]*len(g),g.net_return);ax[0].tick_params(axis='x',rotation=20)
ax[0].set_title('All seeds: already observed development evaluation')
for n in ['equal','rule_risk']:
    x=BASELINE_EVAL8[(n,'development_eval')][0];ax[1].plot(x.index,x.wealth,label=n)
x=ENSEMBLE8['development_eval'][0];ax[1].plot(x.index,x.wealth,label='validation selected family target ensemble');ax[1].legend();plt.tight_layout();plt.show()
L8_CARD={'lesson':8,'status':'verified','checks':len(L8_CHECKS),'seeds':SEEDS8,'steps_per_seed':BUDGET8,
 'checkpoints':len(CHECKPOINTS8),'selected_on_validation':SELECTED_FAMILY8,'normalizer_fit_max':str(DATES8[TRAIN8[-1]]),
 'limits':['small training budget','discrete feasible sleeve space','fixed survivors','historical replay, not market interaction','already observed development windows','no performance guarantee']}
print('L8 FINAL PASS',json.dumps(L8_CARD,ensure_ascii=False))


# 第9课 衍生品定价、Greeks与离散对冲

目标：实现欧式Black–Scholes–Merton，交叉核对QuantLib、二叉树与TPU蒙特卡洛；从自融资账本解释Delta对冲频率、费用与跳跃尾损。模拟对冲是独立专题，不与股票历史策略净值混合。

d₁ = [ln(S/K) + (r − q + σ²/2)T] / (σ√T)，d₂ = d₁ − σ√T。

$C=S e^{-qT}N(d_1)-K e^{-rT}N(d_2)$；$P=K e^{-rT}N(-d_2)-S e^{-qT}N(-d_1)$。风险中性期望按r−q漂移并以r折现。Delta/Gamma/Vega/Theta分别表示价格对S、一阶Delta对S、波动率和时间的局部敏感度；Vega按波动率变化1.00、Theta按一年计，输出时说明单位。

实验：1）手算ATM无息价格、put-call parity、有限差分Greeks与QuantLib；2）TPU反向样本蒙特卡洛和CRR收敛；3）同一组GBM与跳跃路径比较1/5/21日再平衡、自融资现金利息、交易费用、末端交割；4）Cboe VIX的30日年化隐含方差，与SPX完全相同30个日历日窗、365日口径的前瞻实现方差作诊断。实现方差在未来成熟后才可用于评价，不能当当天输入。

VIX本身不可直接交易。没有历史期权bid/ask、合约、行权/到期与公司行动数据，本课不宣称完成真实期权策略回测。Delta对冲不能消除Gamma、Vega、跳跃和离散交易误差。对比VIX与SPX忽略微观结构、非交易时段分布和指数方法差异，只能作为方差溢价代理。

参考：[QuantLib](https://www.quantlib.org/docs.shtml)、[Cboe官方历史指数](https://www.cboe.com/tradable-products/vix/vix-historical-data)。


In [ ]:
# 09.01 BSM定价、Greeks、隐含波动率与QuantLib交叉核验
from scipy.stats import norm
from scipy.optimize import brentq
import QuantLib as ql

def bsm9(S,K,r,q,sigma,T,kind='call'):
    S=np.asarray(S,float);T=float(T)
    if T<=0:
        call=np.maximum(S-K,0);price=call if kind=='call' else np.maximum(K-S,0)
        delta=np.where(S>K,1.,np.where(S<K,0.,.5))-(kind=='put')
        return {'price':price,'delta':delta,'gamma':np.zeros_like(S),'vega':np.zeros_like(S),'theta':np.zeros_like(S)}
    if sigma<=0 or np.any(S<=0) or K<=0:raise ValueError('positive S,K,sigma required before expiry')
    root=np.sqrt(T);d1=(np.log(S/K)+(r-q+.5*sigma*sigma)*T)/(sigma*root);d2=d1-sigma*root;eq=np.exp(-q*T);er=np.exp(-r*T);phi=norm.pdf(d1)
    if kind=='call':
        price=S*eq*norm.cdf(d1)-K*er*norm.cdf(d2);delta=eq*norm.cdf(d1)
        theta=-S*eq*phi*sigma/(2*root)+q*S*eq*norm.cdf(d1)-r*K*er*norm.cdf(d2)
    elif kind=='put':
        price=K*er*norm.cdf(-d2)-S*eq*norm.cdf(-d1);delta=eq*(norm.cdf(d1)-1)
        theta=-S*eq*phi*sigma/(2*root)-q*S*eq*norm.cdf(-d1)+r*K*er*norm.cdf(-d2)
    else:raise ValueError('call or put')
    return {'price':price,'delta':delta,'gamma':eq*phi/(S*sigma*root),'vega':S*eq*phi*root,'theta':theta}
PARAM9={'S':100.,'K':100.,'r':.05,'q':.02,'sigma':.2,'T':1.}
BS9={k:bsm9(**PARAM9,kind=k) for k in ['call','put']};display(pd.DataFrame(BS9).T)
today9=ql.Date(2,ql.January,2020);ql.Settings.instance().evaluationDate=today9;maturity9=today9+365;dc9=ql.Actual365Fixed();calendar9=ql.NullCalendar()
spot9=ql.QuoteHandle(ql.SimpleQuote(100));rf9=ql.YieldTermStructureHandle(ql.FlatForward(today9,.05,dc9));div9=ql.YieldTermStructureHandle(ql.FlatForward(today9,.02,dc9));vol9=ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today9,calendar9,.2,dc9))
process9=ql.BlackScholesMertonProcess(spot9,div9,rf9,vol9);QL9={}
for kind,opt_type in [('call',ql.Option.Call),('put',ql.Option.Put)]:
    option=ql.VanillaOption(ql.PlainVanillaPayoff(opt_type,100),ql.EuropeanExercise(maturity9));option.setPricingEngine(ql.AnalyticEuropeanEngine(process9))
    QL9[kind]={'price':option.NPV(),'delta':option.delta(),'gamma':option.gamma(),'vega':option.vega(),'theta':option.theta()}
display(pd.DataFrame(QL9).T);print('Vega per +1.00 volatility; Theta per calendar year')
FD9=[]
for kind in ['call','put']:
    p=PARAM9.copy();base=float(BS9[kind]['price']);h=.01;hs=1e-4;ht=1e-5
    price=lambda **kwargs:float(bsm9(**(p|kwargs),kind=kind)['price'])
    numerical={'delta':(price(S=100+h)-price(S=100-h))/(2*h),'gamma':(price(S=100+h)-2*base+price(S=100-h))/(h*h),
      'vega':(price(sigma=.2+hs)-price(sigma=.2-hs))/(2*hs),'theta':-(price(T=1+ht)-price(T=1-ht))/(2*ht)}
    for greek,val in numerical.items():FD9.append({'kind':kind,'Greek':greek,'analytic':float(BS9[kind][greek]),'finite_difference':val,'abs_error':abs(float(BS9[kind][greek])-val)})
FD9=pd.DataFrame(FD9);display(FD9)
IV9=brentq(lambda sig:float(bsm9(100,100,.05,.02,sig,1)['price'])-float(BS9['call']['price']),.001,3.)
print('recovered IV',IV9,'hand ATM r=q=0 call',100*(2*norm.cdf(.1)-1))
print('09.01 PASS: independent QuantLib and finite differences available')


In [ ]:
# 09.02 CRR收敛与TPU蒙特卡洛独立核验
TREE9=[]
for steps in [50,100,500,1000]:
    option=ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call,100),ql.EuropeanExercise(maturity9));option.setPricingEngine(ql.BinomialVanillaEngine(process9,'crr',steps))
    price=option.NPV();TREE9.append({'steps':steps,'CRR_price':price,'BSM_price':float(BS9['call']['price']),'error':price-float(BS9['call']['price'])})
@jax.jit
def monte_carlo9(key):
    z=jax.random.normal(key,(65536,),dtype=jnp.float32)
    def payoff(z):return jnp.exp(-.05)*jnp.maximum(100*jnp.exp(.05-.02-.5*.2**2+.2*z)-100,0)
    paired=(payoff(z)+payoff(-z))/2
    return paired.mean(),paired.std(ddof=1)/jnp.sqrt(paired.size)
with jax.default_device(TPU):mc_price9,mc_se9=monte_carlo9(jax.random.PRNGKey(9001))
mc_price9.block_until_ready();MC9={'estimate':float(mc_price9),'standard_error_of_independent_pairs':float(mc_se9),'antithetic_pairs':65536,'terminal_paths':131072,'device':str(mc_price9.devices()),'analytic':float(BS9['call']['price'])}
print(json.dumps(MC9,indent=2));display(pd.DataFrame(TREE9))
assert abs(MC9['estimate']-MC9['analytic'])<5*MC9['standard_error_of_independent_pairs']+.0001
print('09.02 PASS: actual TPU MC; error uses independent antithetic pairs')


In [ ]:
# 09.03 离散Delta对冲的自融资现金账本与跳跃压力
N_PATH9=512;N_STEP9=252;RATE9=.02;SIGMA9=.2;DT9=1/252;K9=100.
@jax.jit
def simulate_gbm9(key):
    z=jax.random.normal(key,(N_PATH9,N_STEP9));lr=(RATE9-.5*SIGMA9**2)*DT9+SIGMA9*jnp.sqrt(DT9)*z
    return jnp.concatenate([jnp.full((N_PATH9,1),100.),100*jnp.exp(jnp.cumsum(lr,axis=1))],axis=1)
with jax.default_device(TPU):PATH_DEVICE9=simulate_gbm9(jax.random.PRNGKey(9002))
PATH_DEVICE9.block_until_ready();PATH9=np.asarray(PATH_DEVICE9,dtype=float);JUMP9=PATH9.copy();JUMP9[:,126:]*=.8
PREMIUM9=float(bsm9(100,100,RATE9,0,SIGMA9,1)['price'])
def delta_hedge9(paths,every=1,cost_bps=0.):
    shares=np.asarray(bsm9(paths[:,0],K9,RATE9,0,SIGMA9,1)['delta']);fee=np.abs(shares)*paths[:,0]*cost_bps/10000
    cash=np.full(len(paths),PREMIUM9)-shares*paths[:,0]-fee;fees=fee.copy();record=[];residual=0.
    for i in range(1,N_STEP9+1):
        cash*=np.exp(RATE9*DT9);S=paths[:,i];before=cash+shares*S
        if i<N_STEP9 and i%every==0:
            desired=np.asarray(bsm9(S,K9,RATE9,0,SIGMA9,(N_STEP9-i)/252)['delta']);trade=desired-shares
            fee=np.abs(trade)*S*cost_bps/10000;cash-=trade*S+fee;shares=desired;fees+=fee
            residual=max(residual,float(np.max(np.abs(cash+shares*S-(before-fee)))))
        option=np.asarray(bsm9(S,K9,RATE9,0,SIGMA9,max(0,(N_STEP9-i)/252))['price'])
        record.append({'day':i,'cash_first':cash[0],'delta_first':shares[0],'spot_first':S[0],'option_first':option[0],'marked_hedge_PnL_first':cash[0]+shares[0]*S[0]-option[0]})
    fee=np.abs(shares)*paths[:,-1]*cost_bps/10000;cash+=shares*paths[:,-1]-fee;fees+=fee;shares=np.zeros(len(paths));terminal=cash-np.maximum(paths[:,-1]-K9,0)
    return {'terminal_PnL':terminal,'fees':fees,'cash_after_payoff':terminal.copy(),'shares_final':shares,'max_accounting_residual':residual,'path':pd.DataFrame(record)}
HEDGE9={};rows=[]
for scenario,paths in [('risk_neutral_GBM',PATH9),('minus20pct_jump_stress',JUMP9)]:
    for every in [1,5,21]:
        for cost in [0.,1.]:
            z=delta_hedge9(paths,every,cost);HEDGE9[(scenario,every,cost)]=z;p=z['terminal_PnL']
            rows.append({'scenario':scenario,'rebalance_every_sessions':every,'cost_bps':cost,'mean_PnL':p.mean(),'hedge_error_std':p.std(ddof=1),'q01_PnL':np.quantile(p,.01),'q05_PnL':np.quantile(p,.05),'mean_fees':z['fees'].mean(),'paths':len(p),'option_premium':PREMIUM9})
HEDGE_TABLE9=pd.DataFrame(rows);display(HEDGE_TABLE9)
fig,ax=plt.subplots(1,2,figsize=(13,4))
for every in [1,5,21]:
    for j,scenario in enumerate(['risk_neutral_GBM','minus20pct_jump_stress']):ax[j].hist(HEDGE9[(scenario,every,1.)]['terminal_PnL'],bins=35,alpha=.4,label=str(every)+' sessions')
for j,title in enumerate(['Same GBM paths: hedge error after costs','Named -20% jump: short-call hedge tail']):ax[j].legend();ax[j].set_title(title)
plt.tight_layout();plt.show();display(HEDGE9[('risk_neutral_GBM',5,1.)]['path'].head(10))
print('09.03 PASS: zero initial wealth, per-option dollar PnL, fees once, cash interest and terminal settlement')


In [ ]:
# 09.04 Cboe VIX与SPX同30日历窗方差诊断、定价对冲验收
VIX_URL9='https://cdn-api.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv'
VIX_BLOB9=fetch(VIX_URL9);VIX9=pd.read_csv(io.BytesIO(VIX_BLOB9));VIX9.columns=VIX9.columns.str.strip().str.upper();VIX9['DATE']=pd.to_datetime(VIX9.DATE);VIX9=VIX9.set_index('DATE').sort_index()
spx_raw9=yf.download('^GSPC',start='2012-01-01',end='2023-01-01',auto_adjust=False,progress=False,threads=False,multi_level_index=False)
if isinstance(spx_raw9.columns,pd.MultiIndex):spx_raw9.columns=spx_raw9.columns.get_level_values(0)
SPX9=spx_raw9['Close'].dropna();SPX9.index=pd.DatetimeIndex(SPX9.index).tz_localize(None).normalize();LOGSPX9=np.log(SPX9).diff()
VAR_ROWS9=[]
for d in SPX9.loc['2014-01-01':'2022-11-30'].index:
    maturity=d+pd.Timedelta(days=30)
    if d not in VIX9.index or maturity>SPX9.index.max():continue
    future=LOGSPX9[(LOGSPX9.index>d)&(LOGSPX9.index<=maturity)].dropna()
    if len(future)<15:continue
    iv=float(VIX9.loc[d,'CLOSE'])/100
    VAR_ROWS9.append({'decision_date':d,'label_calendar_end':maturity,'last_realized_close':future.index[-1],'trading_observations':len(future),
                     'IV_annual_variance':iv*iv,'RV_annual_variance':float((future**2).sum()*365/30)})
VAR9=pd.DataFrame(VAR_ROWS9).set_index('decision_date');VAR9['variance_premium_proxy']=VAR9.IV_annual_variance-VAR9.RV_annual_variance
VIX_MANIFEST9={'url':VIX_URL9,'sha256':hashlib.sha256(VIX_BLOB9).hexdigest(),'source':'Cboe VIX official daily closes','SPX_source':'Yahoo ^GSPC index close',
 'window':'(t,t+30 calendar days], close-to-close squared log returns; 365/30 annualization','not_a_trade':'VIX is an index, no option-chain execution evidence'}
display(VAR9.head());display(VAR9.groupby(VAR9.index.year)[['IV_annual_variance','RV_annual_variance','variance_premium_proxy']].mean());print(json.dumps(VIX_MANIFEST9,indent=2))
# 重叠30日窗相关，另报非重叠近似子样本，每月首个可比日期。
NONOVERLAP9=[];next_allowed=pd.Timestamp.min
for d,r in VAR9.iterrows():
    if d>=next_allowed:NONOVERLAP9.append(d);next_allowed=r.label_calendar_end
print('non-overlap diagnostic n',len(NONOVERLAP9));display(VAR9.loc[NONOVERLAP9][['variance_premium_proxy']].describe())
fig,ax=plt.subplots(1,2,figsize=(13,4));VAR9[['IV_annual_variance','RV_annual_variance']].plot(ax=ax[0],title='Same 30-calendar-day annual variance (365 basis)')
VAR9.variance_premium_proxy.plot(ax=ax[1],title='Ex-post variance premium proxy, not strategy PnL');ax[1].axhline(0,color='k',ls='--');plt.tight_layout();plt.show()
L9_CHECKS=[]
def check9(name,ok):
    assert bool(ok),name;L9_CHECKS.append({'check':name,'status':'PASS'})
check9('put-call parity',np.isclose(float(BS9['call']['price']-BS9['put']['price']),100*np.exp(-.02)-100*np.exp(-.05),atol=1e-10))
check9('ATM hand calculation',np.isclose(float(bsm9(100,100,0,0,.2,1)['price']),100*(2*norm.cdf(.1)-1),atol=1e-10))
check9('recovered implied vol',abs(IV9-.2)<1e-9)
check9('finite-difference Greeks',FD9.abs_error.max()<.001)
for kind in ['call','put']:
    for greek in ['price','delta','gamma','vega','theta']:check9(kind+' QuantLib '+greek,abs(float(BS9[kind][greek])-QL9[kind][greek])<1e-9)
check9('expiry intrinsic',float(bsm9(120,100,.05,0,.2,0)['price'])==20)
check9('tree refinement',abs(TREE9[-1]['error'])<.01 and abs(TREE9[-1]['error'])<abs(TREE9[0]['error']))
check9('TPU Monte Carlo location',mc_price9.devices()=={TPU})
check9('Monte Carlo sampling agreement',abs(MC9['estimate']-MC9['analytic'])<5*MC9['standard_error_of_independent_pairs']+.0001)
for key,result in HEDGE9.items():
    check9(str(key)+' conserved rebalance',result['max_accounting_residual']<1e-9)
    check9(str(key)+' final zero inventory',np.allclose(result['shares_final'],0))
    check9(str(key)+' nonnegative fee',(result['fees']>=0).all())
for scenario in ['risk_neutral_GBM','minus20pct_jump_stress']:
    for every in [1,5,21]:
        check9(scenario+str(every)+' fee reduces same-path PnL',np.all(HEDGE9[(scenario,every,1.)]['terminal_PnL']<=HEDGE9[(scenario,every,0.)]['terminal_PnL']+1e-10))
check9('real VIX manifest',len(VAR9)>1000 and len(VIX_MANIFEST9['sha256'])==64)
check9('RV labels end exactly 30 calendar days',((VAR9.label_calendar_end-VAR9.index).dt.days==30).all())
check9('future maturity not treated as current feature',(VAR9.last_realized_close>VAR9.index).all() and (VAR9.last_realized_close<=VAR9.label_calendar_end).all())
L9_VALIDATION=pd.DataFrame(L9_CHECKS);display(L9_VALIDATION)
L9_CARD={'lesson':9,'status':'verified','checks':len(L9_CHECKS),'pricing':'BSM vs QuantLib, CRR and TPU MC','simulation_paths':N_PATH9,
 'observational_variance_rows':len(VAR9),'limits':['European constant-vol assumptions','hypothetical hedging paths','named deterministic jump stress','no historical option-chain backtest','VIX not tradable','overlapping ex-post variance labels']}
print('L9 FINAL PASS',json.dumps(L9_CARD,ensure_ascii=False))


# 第10课 严格验证、完整项目与GitHub交付

冻结配置后做2019—2022逐年扩展窗口：内层仅用过去一年选择Ridge alpha，标签成熟时间清除到外层起点前5个交易日；同日所有资产在一折。候选、失败、否决全部登记，不按外层结果重新改参。

**最终留出真实状态：** 2023—2026历史区间曾在第一课旧版本展示，2021—2022也已反复用于开发评估。当前没有可恢复为未观察的历史最终测试；本课如实报告嵌套时间评估和冻结开发诊断，不能伪造独立样本外证据。未来新增数据只能在冻结后另行观察。

主线把趋势、财务因子、文本、ML、RL同资产目标先合成，再按第六课约束在一个账户只扣一次费用。第2课残差作为ML/RL市场状态；原双腿多空账户另列。去文本/去ML/去RL/去状态过滤逐一消融，并提供过去验证期冻结的风险匹配对照，解释敞口差异。

日收益区块bootstrap保留局部序列依赖，配对比较共享日期。DSR用日Sharpe、偏度、非超额峰度和试验数敏感性；历史试验总数不全、候选相关且样本有限，DSR仅作诊断。逐笔执行和零初始资本期权对冲各用适当指标，不能混算股票Sharpe。

云端项目包包括00_run_all、10课入口、src、固定依赖与配置、来源哈希、离线测试、完整实验台账、英文README与研究报告、静态图表、检查点、1—10课回顾。原始价格、申报全文和受限词典不再分发；本地课程目录不增加文件。

简历只引用已实现并验证的指标。模拟微观结构、历史回测、模拟期权对冲、TPU计算与真实资金交易分开；教学工程不能冒充个人原创研究或未见数据证明。

参考：[DSR原文](https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf)、[时间验证](https://scikit-learn.org/stable/modules/cross_validation.html)。


In [ ]:
# 10.01 冻结配置、嵌套扩展窗口与成熟标签隔离
CAPSTONE_CONFIG={'frozen_utc':pd.Timestamp.now(tz='UTC').isoformat(),'outer_years':[2019,2020,2021,2022],
 'inner_alpha_grid':[1.,10.,100.],'label':'5-session adjusted open return','embargo_sessions':5,
 'main_weights':{'trend':.20,'factor':.25,'text':.05,'ML':.25,'RL':.25},'cost_bps_per_leg':5,
 'gross_cap':.8,'single_cap':.15,'industry_cap':.35,'turnover_cap':.25,
 'holdout_status':'not pristine: 2023–2026 previously displayed; 2021–2022 reused in development','selection':'inner past-year MSE only'}
NESTED_TRIALS10=[];PRED10=[];OUTER_MODELS10={};SPLIT_AUDIT10=[]
def cutoff10(start):
    k=OPEN.index.searchsorted(pd.Timestamp(start));d=OPEN.index[max(0,k-5)]
    return (d+pd.Timedelta(hours=9,minutes=30)).tz_localize(NY)
for year in CAPSTONE_CONFIG['outer_years']:
    outer_start=pd.Timestamp(year,1,1);outer_end=pd.Timestamp(year+1,1,1);inner_start=pd.Timestamp(year-1,1,1)
    inner_cut=cutoff10(inner_start);outer_cut=cutoff10(outer_start)
    inner_train=D5[(D5.date<inner_start)&(D5.label_end5<inner_cut)].copy()
    inner_valid=D5[(D5.date>=inner_start)&(D5.date<outer_start)&(D5.label_end5<outer_cut)].copy()
    outer_train=D5[(D5.date<outer_start)&(D5.label_end5<outer_cut)].copy()
    outer_test=D5[(D5.date>=outer_start)&(D5.date<outer_end)].copy();scores=[]
    for alpha in CAPSTONE_CONFIG['inner_alpha_grid']:
        try:
            model=regression_pipe(FEATURES,Ridge(alpha=alpha));model.fit(inner_train,inner_train.y5);p=model.predict(inner_valid);mse=mean_squared_error(inner_valid.y5,p)
            scores.append((mse,alpha));NESTED_TRIALS10.append({'outer_year':year,'alpha':alpha,'inner_MSE':mse,'status':'completed','inner_train_n':len(inner_train),'inner_valid_n':len(inner_valid)})
        except Exception as e:NESTED_TRIALS10.append({'outer_year':year,'alpha':alpha,'status':'failed','error':str(e)})
    assert len(scores)==3,'Candidate failure must be investigated'
    _,alpha=min(scores,key=lambda z:(z[0],z[1]));model=regression_pipe(FEATURES,Ridge(alpha=alpha));model.fit(outer_train,outer_train.y5);OUTER_MODELS10[year]=model
    p=model.predict(outer_test);frame=outer_test[['date','ticker','decision_at','execution_session','execution_at','label_end5','y5']].copy();frame['prediction']=p;frame['outer_year']=year;frame['chosen_alpha']=alpha;PRED10.append(frame)
    SPLIT_AUDIT10.append({'year':year,'inner_fit_last_label':inner_train.label_end5.max(),'inner_valid_start':inner_valid.decision_at.min(),
      'outer_fit_last_label':outer_train.label_end5.max(),'outer_decision_start':outer_test.decision_at.min(),'embargo_cutoff':outer_cut,
      'alpha':alpha,'outer_n':len(outer_test),'unique_dates':outer_test.date.nunique(),**prediction_stats(outer_test,p)})
PRED10=pd.concat(PRED10,ignore_index=True).sort_values(['date','ticker']);TRIAL_TABLE10=pd.DataFrame(NESTED_TRIALS10);SPLIT_TABLE10=pd.DataFrame(SPLIT_AUDIT10)
display(pd.DataFrame([CAPSTONE_CONFIG]));display(TRIAL_TABLE10);display(SPLIT_TABLE10)
print('10.01 PASS: all alpha candidates retained, mature labels, five-session embargo, whole-date outer folds')


In [ ]:
# 10.02 同资产先合成仓位再记账，主线消融与风险匹配
DATES10=OPEN.loc['2019-01-02':'2022-12-29'].index
SLEEVE10={n:pd.DataFrame(0.,index=DATES10,columns=ASSETS) for n in ['trend','factor','text','ML','RL']}
trend10=(CLOSE[ASSETS]>CLOSE[ASSETS].rolling(200).mean()).astype(float).shift(1).reindex(DATES10).fillna(0)
SLEEVE10['trend']=trend10.div(trend10.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*.8
SLEEVE10['factor']=L3_TARGETS['equal'].reindex(DATES10).fillna(0)
SLEEVE10['text'].loc[:,'IBM']=L4_TARGETS['price+LM'].IBM.reindex(DATES10).fillna(0)
# 嵌套模型的逐期预测，每5个合格决策日生成下一开盘起五日仓位。
decisions10=pd.DatetimeIndex(sorted(PRED10.date.unique()))[::5]
for d,g in PRED10[PRED10.date.isin(decisions10)].groupby('date'):
    chosen=g[g.prediction>0].sort_values(['prediction','ticker']).tail(3)
    if len(chosen)==0:continue
    entry=g.execution_session.iloc[0];j=OPEN.index.get_loc(entry);window=OPEN.index[j:j+5].intersection(DATES10)
    SLEEVE10['ML'].loc[window,chosen.ticker]=.8/len(chosen)
SLEEVE10['RL']=pd.concat([ENSEMBLE8['validation'][1],ENSEMBLE8['development_eval'][1]]).reindex(DATES10).fillna(0)
BASE10=sum(SLEEVE10[n]*weight for n,weight in CAPSTONE_CONFIG['main_weights'].items())
STATE_MULT10=PREV6.reindex(DATES10).map(SIG6.state_active).fillna(False).astype(float)*.75+.25
RAW10={'full':BASE10.mul(STATE_MULT10,axis=0),'no_state_filter':BASE10.copy()}
for dropped in ['text','ML','RL']:
    RAW10['no_'+dropped]=(BASE10-SLEEVE10[dropped]*CAPSTONE_CONFIG['main_weights'][dropped]).mul(STATE_MULT10,axis=0)
RAW10['equal_reference']=DESIRED6['equal'].reindex(DATES10).fillna(0)
def repair10(target):
    w=target.clip(lower=0).copy();scale=np.minimum(1.,.8/w.sum(axis=1).clip(lower=1e-12))
    scale=np.minimum(scale,.15/w.max(axis=1).clip(lower=1e-12))
    for j in GROUP6.values():scale=np.minimum(scale,.35/w.iloc[:,j].sum(axis=1).clip(lower=1e-12))
    return w.mul(scale,axis=0)
TARGETS10={};LEDGERS10={};TRADES10={};LIMIT10={}
def run10(name,raw,commission=3,slippage=2):
    w,audit=limited_targets(OPEN[ASSETS],repair10(raw),cost=(commission+slippage)/10000)
    x,t=l2_simulate_multi(OPEN[ASSETS],w,commission=commission,slippage=slippage,borrow_rate=0,funding=0,initial=10000)
    TARGETS10[name]=w;LEDGERS10[name]=x;TRADES10[name]=t;LIMIT10[name]=audit
for n,w in RAW10.items():run10(n,w)
RISK_MATCH10=[]
for name in ['no_text','no_ML','no_RL','no_state_filter']:
    reference=LEDGERS10['full'].loc['2019':'2020'].net_return.std();den=LEDGERS10[name].loc['2019':'2020'].net_return.std();scale=min(1.,reference/max(den,1e-12))
    RISK_MATCH10.append({'ablation':name,'validation_frozen_scale':scale})
    run10(name+'_riskmatched',RAW10[name]*scale)
for name,cost in [('full_zero_cost',0),('full_10bp',10)]:run10(name,RAW10['full'],commission=cost,slippage=0)
RESULT10=pd.DataFrame([{'model':n,'period':period,**ledger_stats(x.loc[a:b]),'volatility':x.loc[a:b].net_return.std()*np.sqrt(252),'avg_gross':x.loc[a:b].gross_exposure.mean()}
 for n,x in LEDGERS10.items() for period,a,b in [('validation_reused','2019-01-02','2020-12-31'),('frozen_development_diagnostic','2021-01-01','2022-12-29')]]).set_index(['period','model'])
display(pd.DataFrame(RISK_MATCH10));display(RESULT10)
print('Removed sleeves are not renormalized; exposure reduction shown. Risk matching freezes validation scale and cannot guarantee same future vol.')
fig,ax=plt.subplots(1,2,figsize=(14,4))
for n in ['full','no_text','no_ML','no_RL','no_state_filter','equal_reference']:
    x=LEDGERS10[n].loc['2021':];wealth=(1+x.net_return).cumprod();ax[0].plot(x.index,wealth,label=n);ax[1].plot(x.index,wealth/wealth.cummax().clip(lower=1)-1,label=n)
ax[0].legend(fontsize=7);ax[0].set_title('Frozen development diagnostic: common-account net wealth');ax[1].set_title('Same-capital drawdown');plt.tight_layout();plt.show()
print('10.02 PASS: raw asset targets combined before one fee calculation; intraday/option reports remain separate')


In [ ]:
# 10.03 配对区块bootstrap、DSR敏感性与负结果台账
from scipy.stats import skew,kurtosis
DEV_RET10=pd.DataFrame({n:x.net_return.loc['2021-01-01':] for n,x in LEDGERS10.items()}).dropna()
BOOT10=[];rng10=np.random.default_rng(10001)
for comparator in ['equal_reference','no_text','no_ML','no_RL','no_state_filter']:
    difference=(DEV_RET10['full']-DEV_RET10[comparator]).to_numpy();n=len(difference)
    for block in [5,21,63]:
        starts=rng10.integers(0,n,size=(1000,int(np.ceil(n/block))))
        indices=((starts[:,:,None]+np.arange(block)[None,None,:])%n).reshape(1000,-1)[:,:n]
        samples=difference[indices].mean(axis=1)
        BOOT10.append({'comparison':'full minus '+comparator,'block_sessions':block,'replications':1000,'mean_daily_difference':difference.mean(),
          'bootstrap_q025':np.quantile(samples,.025),'bootstrap_q975':np.quantile(samples,.975),'fraction_bootstrap_le_zero':np.mean(samples<=0),
          'interpretation':'descriptive paired circular block bootstrap, not independent final test'})
BOOT_TABLE10=pd.DataFrame(BOOT10);display(BOOT_TABLE10)
# 已登记的试验下界：保留种子、数据处理、模型、成本与组合实验，不选择性删除失败。
TRIAL_CATALOG10=[]
for lesson,objects in [(1,experiment_registry),(2,L2_EXPERIMENTS)]:
    for i,obj in enumerate(objects):TRIAL_CATALOG10.append({'lesson':lesson,'trial_id':str(i),'detail':str(obj)})
for name in L3_LEDGERS:TRIAL_CATALOG10.append({'lesson':3,'trial_id':name,'detail':'factor score or tercile'})
for name in L4_MODELS:TRIAL_CATALOG10.append({'lesson':4,'trial_id':name,'detail':'event classifier'})
for name in L5_LEDGERS:TRIAL_CATALOG10.append({'lesson':5,'trial_id':name,'detail':'data, model or ablation account'})
for name in L6_LEDGERS:TRIAL_CATALOG10.append({'lesson':6,'trial_id':name,'detail':'risk allocation'})
for row in TRAIN_LOG8:TRIAL_CATALOG10.append({'lesson':8,'trial_id':str(row),'detail':'training seed or reward variant, includes failures'})
for row in NESTED_TRIALS10:TRIAL_CATALOG10.append({'lesson':10,'trial_id':str(row),'detail':'inner candidate'})
for name in LEDGERS10:TRIAL_CATALOG10.append({'lesson':10,'trial_id':name,'detail':'main ablation or cost account'})
TRIAL_CATALOG10=pd.DataFrame(TRIAL_CATALOG10);KNOWN_TRIALS10=len(TRIAL_CATALOG10)
# PSR/DSR使用日Sharpe和非超额峰度；基于iid公式，序列相关限制必须保留。
def dsr10(returns,trial_count,dispersion):
    x=np.asarray(returns,float);n=len(x);s=x.mean()/max(x.std(ddof=1),1e-12);g=skew(x,bias=False);k=kurtosis(x,fisher=False,bias=False)
    euler=.5772156649015329
    benchmark=dispersion*((1-euler)*norm.ppf(1-1/trial_count)+euler*norm.ppf(1-1/(trial_count*np.e))) if trial_count>1 else 0.
    denominator=np.sqrt(max(1-g*s+(k-1)*s*s/4,1e-12))
    return {'daily_SR':s,'annual_SR_252':s*np.sqrt(252),'daily_SR_benchmark':benchmark,'skewness':g,'nonexcess_kurtosis':k,
      'iid_DSR_diagnostic':norm.cdf((s-benchmark)*np.sqrt(n-1)/denominator),'n_days':n,'trial_count_assumed_independent':trial_count}
sharpes=DEV_RET10.mean()/DEV_RET10.std(ddof=1);dispersion=float(sharpes.std(ddof=1));DSR_ROWS10=[]
for trials in [KNOWN_TRIALS10,2*KNOWN_TRIALS10,5*KNOWN_TRIALS10]:
    DSR_ROWS10.append({'model':'full','known_trial_lower_bound':KNOWN_TRIALS10,'SR_dispersion_among_shown_accounts':dispersion,**dsr10(DEV_RET10.full,trials,dispersion)})
DSR_TABLE10=pd.DataFrame(DSR_ROWS10);display(DSR_TABLE10)
print('DSR assumes independent trial counts and iid-return asymptotics; shown strategies correlate, unknown older trials and serial dependence prevent a definitive significance claim.')
NEGATIVE_RESULTS10=[]
full=RESULT10.loc[('frozen_development_diagnostic','full')]
for name in ['no_text','no_ML','no_RL','no_state_filter','equal_reference']:
    row=RESULT10.loc[('frozen_development_diagnostic',name)]
    NEGATIVE_RESULTS10.append({'hypothesis':'full improves net return vs '+name,'delta_net_return':full.net_return-row.net_return,
      'observed_result':'not_supported_in_this_window' if full.net_return<=row.net_return else 'positive_descriptive_difference_only',
      'reject_reason_or_limit':'development data already observed; no independent final holdout; exposure and costs differ'})
NEGATIVE_TABLE10=pd.DataFrame(NEGATIVE_RESULTS10);display(NEGATIVE_TABLE10)
print('10.03 PASS: paired dependence-aware diagnostic, explicit multiple-trial sensitivity, unsuccessful hypotheses retained')


In [ ]:
# 10.04 全链路防泄漏、费用、约束与数值验收
L10_CHECKS=[]
def check10(name,ok):
    assert bool(ok),name;L10_CHECKS.append({'check':name,'status':'PASS'})
for lesson in [2,3,4,5,6,7,8,9]:
    checks=globals()['L'+str(lesson)+'_VALIDATION'];check10('lesson '+str(lesson)+' completed checks',len(checks)>0 and checks.status.eq('PASS').all())
check10('outer day groups unique',PRED10.groupby('date').outer_year.nunique().max()==1)
check10('outer asset-date unique',not PRED10.duplicated(['date','ticker']).any())
check10('all outer labels mature before embargo',(SPLIT_TABLE10.outer_fit_last_label<SPLIT_TABLE10.embargo_cutoff).all())
check10('inner training before validation',(SPLIT_TABLE10.inner_fit_last_label<SPLIT_TABLE10.inner_valid_start).all())
check10('full outer predictions strictly prior to execution',(PRED10.decision_at<PRED10.execution_at).all())
check10('all inner trials kept',len(TRIAL_TABLE10)==12 and TRIAL_TABLE10.status.eq('completed').all())
check10('trial lower bound disclosed',KNOWN_TRIALS10>=len(LEDGERS10)+len(TRAIN_LOG8)+12)
check10('bootstrap interval ordered',(BOOT_TABLE10.bootstrap_q025<=BOOT_TABLE10.bootstrap_q975).all())
check10('DSR probability range',DSR_TABLE10.iid_DSR_diagnostic.between(0,1).all())
check10('DSR more assumed trials never helps',DSR_TABLE10.iid_DSR_diagnostic.is_monotonic_decreasing)
check10('no pristine holdout claim','not pristine' in CAPSTONE_CONFIG['holdout_status'])
check10('no real HFT claim',LOB_MANIFEST['source_kind']=='SYNTHETIC_MECHANISM_ONLY' or LOB_MANIFEST['source_kind']=='official_single_day_sample')
check10('actual TPU arrays and checkpoints',mc_price9.devices()=={TPU} and len(JAX_MODELS8)==10 and len(CHECKPOINTS8)==15)
for name,x in LEDGERS10.items():
    w=TARGETS10[name];audit=LIMIT10[name]
    check10(name+' one shared window',x.index.equals(DATES10))
    check10(name+' equity accounting',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost-x.borrow_cost-x.funding_cost,atol=1e-6))
    check10(name+' compounded return',np.allclose(x.wealth,(1+x.net_return).cumprod()))
    check10(name+' weight constraints',w.min().min()>=-1e-8 and w.sum(axis=1).max()<=.800001 and w.max().max()<=.150001 and all(w.iloc[:,j].sum(axis=1).max()<=.350001 for j in GROUP6.values()))
    check10(name+' nonnegative costs',(x[['transaction_cost','borrow_cost','funding_cost']]>=0).all().all())
    check10(name+' turnover and forced-risk audit',((x.turnover<=.250001)|audit.startup|audit.forced_risk_override).all())
check10('zero-cost account pays no fees',np.allclose(LEDGERS10['full_zero_cost'].transaction_cost,0))
# 冻结模型只做transform不能悄悄fit；极端未来输入不改变训练填补统计。
model=OUTER_MODELS10[2022];imputer=model.named_steps['features'].named_transformers_['numeric'].named_steps['impute'];before=imputer.statistics_.copy()
probe=D5[D5.date.dt.year.eq(2022)].head(24).copy();probe[FEATURES]=probe[FEATURES]*10+100;model.predict(probe)
check10('outer transform never refits',np.array_equal(before,imputer.statistics_,equal_nan=True))
# 未来价格扰动不改变决策时的prefix return；来源审计和版本保留。
d=PRED10.date.iloc[0];a=PRED10.ticker.iloc[0];past=CLOSE.loc[:d,a];prefix=past.iloc[-1]/past.iloc[-6]-1
perturbed=CLOSE[a].copy();perturbed.loc[perturbed.index>d]*=10
check10('future perturbation does not alter historical return',np.isclose(prefix,perturbed.loc[:d].iloc[-1]/perturbed.loc[:d].iloc[-6]-1))
L10_VALIDATION=pd.DataFrame(L10_CHECKS);display(L10_VALIDATION)
ALL_VALIDATION10=pd.concat([globals()['L'+str(i)+'_VALIDATION'].assign(lesson=i) for i in range(2,11)],ignore_index=True)
print('Verified total recorded checks (lessons2–10)',len(ALL_VALIDATION10));display(ALL_VALIDATION10.groupby('lesson').size().rename('passed_checks'))
L10_CARD={'lesson':10,'status':'verified','checks':len(L10_CHECKS),'outer_years':CAPSTONE_CONFIG['outer_years'],
 'main_account_count':len(LEDGERS10),'known_trial_lower_bound':KNOWN_TRIALS10,'historical_holdout':CAPSTONE_CONFIG['holdout_status'],'completed_utc':pd.Timestamp.now(tz='UTC').isoformat()}
print('L10 RESEARCH FINAL PASS',json.dumps(L10_CARD,ensure_ascii=False))
